# 02. Train / Validation / Test Split & Preprocessing

## 목적
- Survival modeling cohort 529명을 Train / Validation / Test로 고정 분할
- PFI event 비율을 유지하도록 stratified split 수행
- Split 정보를 저장하여 이후 모든 모델에서 동일한 환자 분할 사용
- RNA 전처리는 Train 데이터 기준으로만 수행하여 data leakage 방지

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

SEED = 42

data_dir = Path("../data")

cohort_path = data_dir / "processed" / "kirc_modeling_cohort.parquet"
rna_path = data_dir / "processed" / "rna_tpm_modeling_raw.parquet"

cohort = pd.read_parquet(cohort_path)
rna_tpm = pd.read_parquet(rna_path)

print("Clinical cohort shape:", cohort.shape)
print("RNA TPM shape:", rna_tpm.shape)

Clinical cohort shape: (529, 10)
RNA TPM shape: (529, 19962)


In [5]:
print("환자 수:", len(cohort))
print("고유 patient_id:", cohort["patient_id"].nunique())

print("\nPFI event 분포")
print(cohort["pfi_event"].value_counts().sort_index())

print("\nPFI time")
print(cohort["pfi_time"].describe())

print("\nRNA 환자 수:", len(rna_tpm))
print("RNA index 고유 수:", rna_tpm.index.nunique())

# QC
assert len(cohort) == 529
assert cohort["patient_id"].nunique() == 529

assert cohort["pfi_event"].isin([0, 1]).all()
assert (cohort["pfi_time"] > 0).all()

assert rna_tpm.shape[0] == 529
assert rna_tpm.index.nunique() == 529

assert set(cohort["patient_id"]) == set(rna_tpm.index)

print("\n===== PRE-SPLIT QC PASSED =====")

환자 수: 529
고유 patient_id: 529

PFI event 분포
pfi_event
0.0    370
1.0    159
Name: count, dtype: int64

PFI time
count     529.000000
mean     1164.875236
std       945.494688
min         2.000000
25%       362.000000
50%      1014.000000
75%      1757.000000
max      4074.000000
Name: pfi_time, dtype: float64

RNA 환자 수: 529
RNA index 고유 수: 529

===== PRE-SPLIT QC PASSED =====


In [6]:
from sklearn.model_selection import train_test_split

SEED = 42

# 1차: Train 70% / 임시셋 30%
train_df, temp_df = train_test_split(
    cohort,
    test_size=0.30,
    random_state=SEED,
    stratify=cohort["pfi_event"]
)

# 2차: 임시셋을 Validation / Test로 절반씩 분할
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["pfi_event"]
)

print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))
print("Total      :", len(train_df) + len(val_df) + len(test_df))

Train      : 370
Validation : 79
Test       : 80
Total      : 529


In [7]:
def check_event_ratio(df, name):
    event_count = int(df["pfi_event"].sum())
    event_rate = df["pfi_event"].mean()

    print(
        f"{name:10s} | "
        f"N={len(df):3d} | "
        f"Event={event_count:3d} | "
        f"Event rate={event_rate:.3f}"
    )

check_event_ratio(train_df, "Train")
check_event_ratio(val_df, "Validation")
check_event_ratio(test_df, "Test")

Train      | N=370 | Event=111 | Event rate=0.300
Validation | N= 79 | Event= 24 | Event rate=0.304
Test       | N= 80 | Event= 24 | Event rate=0.300


In [8]:
splits = pd.concat([
    train_df[["patient_id"]].assign(split="train"),
    val_df[["patient_id"]].assign(split="validation"),
    test_df[["patient_id"]].assign(split="test")
], ignore_index=True)

splits.to_csv(
    "../data/processed/splits.csv",
    index=False
)

print(splits["split"].value_counts())
print("\n총 환자 수:", len(splits))
print("고유 환자 수:", splits["patient_id"].nunique())

split
train         370
test           80
validation     79
Name: count, dtype: int64

총 환자 수: 529
고유 환자 수: 529


In [9]:
splits_check = pd.read_csv("../data/processed/splits.csv")

print(splits_check["split"].value_counts())
print("총 환자 수:", len(splits_check))
print("고유 환자 수:", splits_check["patient_id"].nunique())

split
train         370
test           80
validation     79
Name: count, dtype: int64
총 환자 수: 529
고유 환자 수: 529


In [10]:
train_ids = splits_check.loc[
    splits_check["split"] == "train", "patient_id"
]

val_ids = splits_check.loc[
    splits_check["split"] == "validation", "patient_id"
]

test_ids = splits_check.loc[
    splits_check["split"] == "test", "patient_id"
]

rna_train = rna_tpm.loc[train_ids].copy()
rna_val = rna_tpm.loc[val_ids].copy()
rna_test = rna_tpm.loc[test_ids].copy()

print("RNA Train     :", rna_train.shape)
print("RNA Validation:", rna_val.shape)
print("RNA Test      :", rna_test.shape)

RNA Train     : (370, 19962)
RNA Validation: (79, 19962)
RNA Test      : (80, 19962)


In [11]:
min_tpm = 1.0
min_fraction = 0.20

min_samples = int(np.ceil(len(rna_train) * min_fraction))

keep_mask = (rna_train >= min_tpm).sum(axis=0) >= min_samples
keep_genes = rna_train.columns[keep_mask]

print("Train 환자 수:", len(rna_train))
print("최소 필요 환자 수:", min_samples)
print("필터 전 gene 수:", rna_train.shape[1])
print("필터 후 gene 수:", len(keep_genes))

Train 환자 수: 370
최소 필요 환자 수: 74
필터 전 gene 수: 19962
필터 후 gene 수: 14220


In [12]:
rna_train_filtered = rna_train.loc[:, keep_genes].copy()
rna_val_filtered = rna_val.loc[:, keep_genes].copy()
rna_test_filtered = rna_test.loc[:, keep_genes].copy()

print("Train     :", rna_train_filtered.shape)
print("Validation:", rna_val_filtered.shape)
print("Test      :", rna_test_filtered.shape)

Train     : (370, 14220)
Validation: (79, 14220)
Test      : (80, 14220)


In [13]:
rna_train_log = np.log2(rna_train_filtered + 1)
rna_val_log = np.log2(rna_val_filtered + 1)
rna_test_log = np.log2(rna_test_filtered + 1)

print("Train     :", rna_train_log.shape)
print("Validation:", rna_val_log.shape)
print("Test      :", rna_test_log.shape)

print("\nTrain 최소값:", rna_train_log.min().min())
print("Train 최대값:", rna_train_log.max().max())

Train     : (370, 14220)
Validation: (79, 14220)
Test      : (80, 14220)

Train 최소값: 0.0
Train 최대값: 16.989737


In [14]:
import numpy as np

for name, df in [
    ("Train", rna_train_log),
    ("Validation", rna_val_log),
    ("Test", rna_test_log)
]:
    print(
        name,
        "| NaN:", df.isna().sum().sum(),
        "| Inf:", np.isinf(df.to_numpy()).sum()
    )

Train | NaN: 0 | Inf: 0
Validation | NaN: 0 | Inf: 0
Test | NaN: 0 | Inf: 0


In [15]:
gene_variance = rna_train_log.var(axis=0)

zero_var_genes = gene_variance[gene_variance == 0].index

print("전체 gene 수:", len(gene_variance))
print("분산 0 gene 수:", len(zero_var_genes))

전체 gene 수: 14220
분산 0 gene 수: 0


In [16]:
gene_variance.describe()

count    14220.000000
mean         0.695362
std          0.735984
min          0.069184
25%          0.320109
50%          0.464267
75%          0.780313
max         12.933584
dtype: float64

In [17]:
gene_median = rna_train_log.median(axis=0)

gene_mad = (
    rna_train_log
    .sub(gene_median, axis=1)
    .abs()
    .median(axis=0)
)

print(gene_mad.describe())

count    14220.000000
mean         0.462363
std          0.219562
min          0.070252
25%          0.325158
50%          0.401043
75%          0.530979
max          2.983450
dtype: float64


In [18]:
top_mad_genes = gene_mad.sort_values(ascending=False)

print(top_mad_genes.head(20))

ENSG00000164434.12    2.983450
ENSG00000187048.13    2.617446
ENSG00000129824.16    2.495842
ENSG00000197901.12    2.425745
ENSG00000143839.15    2.381181
ENSG00000115386.6     2.350639
ENSG00000138207.14    2.330923
ENSG00000157765.13    2.302546
ENSG00000124107.5     2.260641
ENSG00000197891.12    2.205963
ENSG00000136872.20    2.178046
ENSG00000174358.16    2.163082
ENSG00000162896.6     2.149508
ENSG00000262655.4     2.125751
ENSG00000100170.10    2.102740
ENSG00000007062.12    2.083239
ENSG00000198692.10    2.066936
ENSG00000145423.5     2.045257
ENSG00000171345.13    2.042628
ENSG00000175785.13    2.022525
dtype: float32


In [19]:
top_mad_genes = gene_mad.sort_values(ascending=False)

print(top_mad_genes.head(20))

ENSG00000164434.12    2.983450
ENSG00000187048.13    2.617446
ENSG00000129824.16    2.495842
ENSG00000197901.12    2.425745
ENSG00000143839.15    2.381181
ENSG00000115386.6     2.350639
ENSG00000138207.14    2.330923
ENSG00000157765.13    2.302546
ENSG00000124107.5     2.260641
ENSG00000197891.12    2.205963
ENSG00000136872.20    2.178046
ENSG00000174358.16    2.163082
ENSG00000162896.6     2.149508
ENSG00000262655.4     2.125751
ENSG00000100170.10    2.102740
ENSG00000007062.12    2.083239
ENSG00000198692.10    2.066936
ENSG00000145423.5     2.045257
ENSG00000171345.13    2.042628
ENSG00000175785.13    2.022525
dtype: float32


In [20]:
gene_annotation = pd.read_parquet(
    data_dir / "interim" / "rna_gene_annotation.parquet"
)

print(gene_annotation.columns.tolist())
print(gene_annotation.head())

['gene_id', 'gene_name', 'gene_type']
              gene_id gene_name       gene_type
0  ENSG00000000003.15    TSPAN6  protein_coding
1   ENSG00000000005.6      TNMD  protein_coding
2  ENSG00000000419.13      DPM1  protein_coding
3  ENSG00000000457.14     SCYL3  protein_coding
4  ENSG00000000460.17  C1orf112  protein_coding


In [21]:
top_mad_df = (
    top_mad_genes.head(20)
    .rename("MAD")
    .reset_index()
    .rename(columns={"index": "gene_id"})
)

top_mad_annotated = top_mad_df.merge(
    gene_annotation[["gene_id", "gene_name"]],
    on="gene_id",
    how="left"
)

print(top_mad_annotated)

               gene_id       MAD gene_name
0   ENSG00000164434.12  2.983450     FABP7
1   ENSG00000187048.13  2.617446   CYP4A11
2   ENSG00000129824.16  2.495842    RPS4Y1
3   ENSG00000197901.12  2.425745   SLC22A6
4   ENSG00000143839.15  2.381181       REN
5    ENSG00000115386.6  2.350639     REG1A
6   ENSG00000138207.14  2.330923      RBP4
7   ENSG00000157765.13  2.302546   SLC34A2
8    ENSG00000124107.5  2.260641      SLPI
9   ENSG00000197891.12  2.205963  SLC22A12
10  ENSG00000136872.20  2.178046     ALDOB
11  ENSG00000174358.16  2.163082   SLC6A19
12   ENSG00000162896.6  2.149508      PIGR
13   ENSG00000262655.4  2.125751     SPON1
14  ENSG00000100170.10  2.102740    SLC5A1
15  ENSG00000007062.12  2.083239     PROM1
16  ENSG00000198692.10  2.066936    EIF1AY
17   ENSG00000145423.5  2.045257     SFRP2
18  ENSG00000171345.13  2.042628     KRT19
19  ENSG00000175785.13  2.022525    PRIMA1


In [22]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(train_df, train_df["pfi_event"]),
    start=1
):
    fit_events = train_df.iloc[fit_idx]["pfi_event"]
    cv_events = train_df.iloc[cv_idx]["pfi_event"]

    print(
        f"Fold {fold} | "
        f"Fit N={len(fit_idx)}, Event={int(fit_events.sum())} | "
        f"CV N={len(cv_idx)}, Event={int(cv_events.sum())}"
    )

Fold 1 | Fit N=296, Event=88 | CV N=74, Event=23
Fold 2 | Fit N=296, Event=89 | CV N=74, Event=22
Fold 3 | Fit N=296, Event=89 | CV N=74, Event=22
Fold 4 | Fit N=296, Event=89 | CV N=74, Event=22
Fold 5 | Fit N=296, Event=89 | CV N=74, Event=22


In [23]:
# Fold 1 가져오기
fit_idx, cv_idx = next(
    cv.split(train_df, train_df["pfi_event"])
)

fit_ids = train_df.iloc[fit_idx]["patient_id"]
cv_ids_fold = train_df.iloc[cv_idx]["patient_id"]

# 원본 TPM 사용
rna_fit_raw = rna_tpm.loc[fit_ids].copy()
rna_cv_raw = rna_tpm.loc[cv_ids_fold].copy()

# 1. Fold의 Fit 데이터에서만 저발현 필터 결정
min_samples_fold = int(np.ceil(len(rna_fit_raw) * 0.20))

keep_mask_fold = (
    (rna_fit_raw >= 1.0).sum(axis=0)
    >= min_samples_fold
)

fold_genes = rna_fit_raw.columns[keep_mask_fold]

# 2. 같은 gene을 Fit / CV에 적용
rna_fit_filtered = rna_fit_raw.loc[:, fold_genes]
rna_cv_filtered = rna_cv_raw.loc[:, fold_genes]

# 3. log2(TPM + 1)
rna_fit_log_fold = np.log2(rna_fit_filtered + 1)
rna_cv_log_fold = np.log2(rna_cv_filtered + 1)

# 4. Fit 데이터에서만 MAD 계산
fold_median = rna_fit_log_fold.median(axis=0)

fold_mad = (
    rna_fit_log_fold
    .sub(fold_median, axis=1)
    .abs()
    .median(axis=0)
    .sort_values(ascending=False)
)

print("Fold 1 Fit 환자:", len(fit_ids))
print("Fold 1 CV 환자 :", len(cv_ids_fold))
print("저발현 필터 최소 환자 수:", min_samples_fold)
print("필터 전 gene:", rna_fit_raw.shape[1])
print("필터 후 gene:", len(fold_genes))
print("MAD 계산 gene:", len(fold_mad))

Fold 1 Fit 환자: 296
Fold 1 CV 환자 : 74
저발현 필터 최소 환자 수: 60
필터 전 gene: 19962
필터 후 gene: 14219
MAD 계산 gene: 14219


In [24]:
fold_summary = []

for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(train_df, train_df["pfi_event"]),
    start=1
):
    fit_ids = train_df.iloc[fit_idx]["patient_id"]
    cv_ids_fold = train_df.iloc[cv_idx]["patient_id"]

    # 원본 TPM
    rna_fit_raw = rna_tpm.loc[fit_ids].copy()
    rna_cv_raw = rna_tpm.loc[cv_ids_fold].copy()

    # Fit 데이터에서만 저발현 필터
    min_samples_fold = int(np.ceil(len(rna_fit_raw) * 0.20))

    keep_mask_fold = (
        (rna_fit_raw >= 1.0).sum(axis=0)
        >= min_samples_fold
    )

    fold_genes = rna_fit_raw.columns[keep_mask_fold]

    # 동일 gene을 Fit / CV에 적용
    rna_fit_filtered = rna_fit_raw.loc[:, fold_genes]
    rna_cv_filtered = rna_cv_raw.loc[:, fold_genes]

    # log2(TPM + 1)
    rna_fit_log_fold = np.log2(rna_fit_filtered + 1)
    rna_cv_log_fold = np.log2(rna_cv_filtered + 1)

    # Fit에서만 MAD 계산
    fold_median = rna_fit_log_fold.median(axis=0)

    fold_mad = (
        rna_fit_log_fold
        .sub(fold_median, axis=1)
        .abs()
        .median(axis=0)
        .sort_values(ascending=False)
    )

    fold_summary.append({
        "fold": fold,
        "fit_n": len(fit_idx),
        "cv_n": len(cv_idx),
        "filtered_genes": len(fold_genes),
        "mad_genes": len(fold_mad)
    })

fold_summary = pd.DataFrame(fold_summary)
print(fold_summary)

   fold  fit_n  cv_n  filtered_genes  mad_genes
0     1    296    74           14219      14219
1     2    296    74           14206      14206
2     3    296    74           14209      14209
3     4    296    74           14225      14225
4     5    296    74           14222      14222


In [25]:
K_VALUES = [500, 1000, 2000, 5000]

fold_topk = {}

for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(train_df, train_df["pfi_event"]),
    start=1
):
    fit_ids = train_df.iloc[fit_idx]["patient_id"]

    # Fold Fit의 원본 TPM
    rna_fit_raw = rna_tpm.loc[fit_ids].copy()

    # Fit에서만 저발현 필터
    min_samples_fold = int(np.ceil(len(rna_fit_raw) * 0.20))

    keep_mask_fold = (
        (rna_fit_raw >= 1.0).sum(axis=0)
        >= min_samples_fold
    )

    fold_genes = rna_fit_raw.columns[keep_mask_fold]

    # log2(TPM + 1)
    rna_fit_log_fold = np.log2(
        rna_fit_raw.loc[:, fold_genes] + 1
    )

    # Fit에서만 MAD
    fold_median = rna_fit_log_fold.median(axis=0)

    fold_mad = (
        rna_fit_log_fold
        .sub(fold_median, axis=1)
        .abs()
        .median(axis=0)
        .sort_values(ascending=False)
    )

    # Top-K gene 저장
    fold_topk[fold] = {
        k: fold_mad.head(k).index.tolist()
        for k in K_VALUES
    }

    print(
        f"Fold {fold} | "
        + " | ".join(
            f"Top-{k}: {len(fold_topk[fold][k])}"
            for k in K_VALUES
        )
    )

Fold 1 | Top-500: 500 | Top-1000: 1000 | Top-2000: 2000 | Top-5000: 5000
Fold 2 | Top-500: 500 | Top-1000: 1000 | Top-2000: 2000 | Top-5000: 5000
Fold 3 | Top-500: 500 | Top-1000: 1000 | Top-2000: 2000 | Top-5000: 5000
Fold 4 | Top-500: 500 | Top-1000: 1000 | Top-2000: 2000 | Top-5000: 5000
Fold 5 | Top-500: 500 | Top-1000: 1000 | Top-2000: 2000 | Top-5000: 5000


In [26]:
import sksurv
from sksurv.util import Surv

print("scikit-survival version:", sksurv.__version__)

scikit-survival version: 0.28.0


In [27]:
y_train = Surv.from_arrays(
    event=train_df["pfi_event"].astype(bool).to_numpy(),
    time=train_df["pfi_time"].astype(float).to_numpy()
)

print("y_train 길이:", len(y_train))
print("event 수:", y_train["event"].sum())
print("처음 5개:")
print(y_train[:5])

y_train 길이: 370
event 수: 111
처음 5개:
[( True, 1714.) (False,   41.) ( True,  958.) ( True,  240.)
 ( True, 1238.)]


In [28]:
from sklearn.preprocessing import StandardScaler

# Fold 1 다시 가져오기
fit_idx, cv_idx = next(
    cv.split(train_df, train_df["pfi_event"])
)

fit_df = train_df.iloc[fit_idx].copy()
cv_df_fold = train_df.iloc[cv_idx].copy()

fit_ids = fit_df["patient_id"]
cv_ids_fold = cv_df_fold["patient_id"]

# Fold 1에서 선정된 Top-500 genes
top500_genes = fold_topk[1][500]

# log2(TPM + 1)
X_fit = np.log2(
    rna_tpm.loc[fit_ids, top500_genes] + 1
)

X_cv = np.log2(
    rna_tpm.loc[cv_ids_fold, top500_genes] + 1
)

# Fit에서만 scaling 학습
scaler = StandardScaler()

X_fit_scaled = scaler.fit_transform(X_fit)
X_cv_scaled = scaler.transform(X_cv)

# survival target
y_fit = Surv.from_arrays(
    event=fit_df["pfi_event"].astype(bool).to_numpy(),
    time=fit_df["pfi_time"].astype(float).to_numpy()
)

y_cv = Surv.from_arrays(
    event=cv_df_fold["pfi_event"].astype(bool).to_numpy(),
    time=cv_df_fold["pfi_time"].astype(float).to_numpy()
)

print("X Fit:", X_fit_scaled.shape)
print("X CV :", X_cv_scaled.shape)
print("Fit events:", y_fit["event"].sum())
print("CV events :", y_cv["event"].sum())

X Fit: (296, 500)
X CV : (74, 500)
Fit events: 88
CV events : 23


In [29]:
import numpy
import scipy
import sklearn
import sksurv

from sksurv.linear_model import CoxnetSurvivalAnalysis

print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("scikit-survival:", sksurv.__version__)
print("CoxNet import OK")

numpy: 2.4.6
scipy: 1.17.1
scikit-learn: 1.9.1
scikit-survival: 0.28.0
CoxNet import OK


In [30]:
from sksurv.linear_model import CoxnetSurvivalAnalysis

coxnet_test = CoxnetSurvivalAnalysis(
    l1_ratio=0.5,
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

coxnet_test.fit(X_fit_scaled, y_fit)

print("alpha 개수:", len(coxnet_test.alphas_))
print("가장 큰 alpha:", coxnet_test.alphas_[0])
print("가장 작은 alpha:", coxnet_test.alphas_[-1])
print("coefficient matrix shape:", coxnet_test.coef_.shape)

alpha 개수: 50
가장 큰 alpha: 0.3736196738723995
가장 작은 alpha: 0.0037361967387240028
coefficient matrix shape: (500, 50)


In [31]:
from sksurv.metrics import concordance_index_censored
import pandas as pd
import numpy as np

alpha_results = []

for i, alpha in enumerate(coxnet_test.alphas_):

    # CV 환자의 risk score
    risk_score = coxnet_test.predict(
        X_cv_scaled,
        alpha=alpha
    )

    # Harrell's C-index
    c_index = concordance_index_censored(
        y_cv["event"],
        y_cv["time"],
        risk_score
    )[0]

    # 해당 alpha에서 0이 아닌 coefficient 수
    n_nonzero = np.count_nonzero(coxnet_test.coef_[:, i])

    alpha_results.append({
        "alpha": alpha,
        "c_index": c_index,
        "n_nonzero_genes": n_nonzero
    })

alpha_results = pd.DataFrame(alpha_results)

print(
    alpha_results
    .sort_values("c_index", ascending=False)
    .head(10)
)

       alpha   c_index  n_nonzero_genes
47  0.004509  0.665377              252
46  0.004953  0.664410              242
43  0.006566  0.664410              231
45  0.005441  0.664410              237
48  0.004104  0.664410              253
44  0.005977  0.663443              236
42  0.007213  0.662476              228
49  0.003736  0.660542              256
38  0.010505  0.658607              202
41  0.007924  0.657640              219


In [32]:
from sklearn.preprocessing import StandardScaler
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

cv_results = []

for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(train_df, train_df["pfi_event"]),
    start=1
):
    fit_df = train_df.iloc[fit_idx].copy()
    cv_df_fold = train_df.iloc[cv_idx].copy()

    fit_ids = fit_df["patient_id"]
    cv_ids_fold = cv_df_fold["patient_id"]

    y_fit = Surv.from_arrays(
        event=fit_df["pfi_event"].astype(bool).to_numpy(),
        time=fit_df["pfi_time"].astype(float).to_numpy()
    )

    y_cv = Surv.from_arrays(
        event=cv_df_fold["pfi_event"].astype(bool).to_numpy(),
        time=cv_df_fold["pfi_time"].astype(float).to_numpy()
    )

    for k in K_VALUES:

        genes = fold_topk[fold][k]

        # log2(TPM + 1)
        X_fit = np.log2(rna_tpm.loc[fit_ids, genes] + 1)
        X_cv = np.log2(rna_tpm.loc[cv_ids_fold, genes] + 1)

        # Fit fold에서만 scaling
        scaler = StandardScaler()
        X_fit_scaled = scaler.fit_transform(X_fit)
        X_cv_scaled = scaler.transform(X_cv)

        # CoxNet
        model = CoxnetSurvivalAnalysis(
            l1_ratio=0.5,
            n_alphas=50,
            alpha_min_ratio=0.01,
            max_iter=100000
        )

        model.fit(X_fit_scaled, y_fit)

        alpha_max = model.alphas_[0]

        for alpha_idx, alpha in enumerate(model.alphas_):

            risk = model.predict(
                X_cv_scaled,
                alpha=alpha
            )

            c_index = concordance_index_censored(
                y_cv["event"],
                y_cv["time"],
                risk
            )[0]

            n_nonzero = np.count_nonzero(
                model.coef_[:, alpha_idx]
            )

            cv_results.append({
                "fold": fold,
                "k": k,
                "alpha_idx": alpha_idx,
                "alpha": alpha,
                "alpha_ratio": alpha / alpha_max,
                "c_index": c_index,
                "n_nonzero": n_nonzero
            })

    print(f"Fold {fold} 완료")

cv_results = pd.DataFrame(cv_results)

print("총 결과 행:", len(cv_results))
print(cv_results.head())

Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료
총 결과 행: 1000
   fold    k  alpha_idx     alpha  alpha_ratio   c_index  n_nonzero
0     1  500          0  0.373620     1.000000  0.500000          0
1     1  500          1  0.340105     0.910298  0.560928          3
2     1  500          2  0.309597     0.828643  0.584139          6
3     1  500          3  0.281826     0.754312  0.589942          8
4     1  500          4  0.256546     0.686649  0.585106          8


In [33]:
cv_summary = (
    cv_results
    .groupby(["k", "alpha_idx"], as_index=False)
    .agg(
        alpha_ratio=("alpha_ratio", "mean"),
        mean_c_index=("c_index", "mean"),
        std_c_index=("c_index", "std"),
        mean_nonzero=("n_nonzero", "mean")
    )
)

best_by_k = (
    cv_summary
    .loc[
        cv_summary.groupby("k")["mean_c_index"].idxmax()
    ]
    .sort_values("mean_c_index", ascending=False)
    .reset_index(drop=True)
)

print(best_by_k[
    [
        "k",
        "alpha_idx",
        "alpha_ratio",
        "mean_c_index",
        "std_c_index",
        "mean_nonzero"
    ]
])

      k  alpha_idx  alpha_ratio  mean_c_index  std_c_index  mean_nonzero
0  1000         26     0.086851      0.734150     0.054912         149.4
1   500         23     0.115140      0.718533     0.060576          87.2
2  5000         18     0.184207      0.713083     0.064382         106.2
3  2000         16     0.222300      0.712514     0.051486          71.6


In [34]:
L1_RATIOS = [0.1, 0.25, 0.5, 0.75, 0.9]

joint_results = []

for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(train_df, train_df["pfi_event"]),
    start=1
):
    fit_df = train_df.iloc[fit_idx].copy()
    cv_df_fold = train_df.iloc[cv_idx].copy()

    fit_ids = fit_df["patient_id"]
    cv_ids_fold = cv_df_fold["patient_id"]

    y_fit = Surv.from_arrays(
        event=fit_df["pfi_event"].astype(bool).to_numpy(),
        time=fit_df["pfi_time"].astype(float).to_numpy()
    )

    y_cv = Surv.from_arrays(
        event=cv_df_fold["pfi_event"].astype(bool).to_numpy(),
        time=cv_df_fold["pfi_time"].astype(float).to_numpy()
    )

    for k in K_VALUES:

        genes = fold_topk[fold][k]

        X_fit = np.log2(rna_tpm.loc[fit_ids, genes] + 1)
        X_cv = np.log2(rna_tpm.loc[cv_ids_fold, genes] + 1)

        scaler = StandardScaler()
        X_fit_scaled = scaler.fit_transform(X_fit)
        X_cv_scaled = scaler.transform(X_cv)

        for l1_ratio in L1_RATIOS:

            model = CoxnetSurvivalAnalysis(
                l1_ratio=l1_ratio,
                n_alphas=50,
                alpha_min_ratio=0.01,
                max_iter=100000
            )

            model.fit(X_fit_scaled, y_fit)

            alpha_max = model.alphas_[0]

            for alpha_idx, alpha in enumerate(model.alphas_):

                risk = model.predict(
                    X_cv_scaled,
                    alpha=alpha
                )

                c_index = concordance_index_censored(
                    y_cv["event"],
                    y_cv["time"],
                    risk
                )[0]

                n_nonzero = np.count_nonzero(
                    model.coef_[:, alpha_idx]
                )

                joint_results.append({
                    "fold": fold,
                    "k": k,
                    "l1_ratio": l1_ratio,
                    "alpha_idx": alpha_idx,
                    "alpha": alpha,
                    "alpha_ratio": alpha / alpha_max,
                    "c_index": c_index,
                    "n_nonzero": n_nonzero
                })

    print(f"Fold {fold} 완료")

joint_results = pd.DataFrame(joint_results)

print("총 결과 행:", len(joint_results))
print(joint_results.head())

Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료
총 결과 행: 5000
   fold    k  l1_ratio  alpha_idx     alpha  alpha_ratio   c_index  n_nonzero
0     1  500       0.1          0  1.868098     1.000000  0.500000          0
1     1  500       0.1          1  1.700527     0.910298  0.568665          4
2     1  500       0.1          2  1.547986     0.828643  0.583172          9
3     1  500       0.1          3  1.409129     0.754312  0.585106         12
4     1  500       0.1          4  1.282728     0.686649  0.589942         15


In [35]:
joint_summary = (
    joint_results
    .groupby(
        ["k", "l1_ratio", "alpha_idx"],
        as_index=False
    )
    .agg(
        alpha_ratio=("alpha_ratio", "mean"),
        mean_c_index=("c_index", "mean"),
        std_c_index=("c_index", "std"),
        mean_nonzero=("n_nonzero", "mean")
    )
)

best_joint = (
    joint_summary
    .sort_values(
        ["mean_c_index", "std_c_index"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print(best_joint.head(20))

       k  l1_ratio  alpha_idx  alpha_ratio  mean_c_index  std_c_index  \
0   1000      0.75         27     0.079060      0.738022     0.053463   
1   1000      0.75         26     0.086851      0.737827     0.055089   
2   1000      0.75         28     0.071969      0.735539     0.046331   
3   1000      0.90         26     0.086851      0.735375     0.051396   
4   1000      0.90         27     0.079060      0.734223     0.049495   
5   1000      0.50         26     0.086851      0.734150     0.054912   
6   1000      0.90         25     0.095410      0.733914     0.050107   
7   1000      0.50         27     0.079060      0.733580     0.053711   
8   1000      0.90         23     0.115140      0.732968     0.052456   
9   1000      0.75         29     0.065513      0.732895     0.044124   
10  1000      0.90         22     0.126486      0.732636     0.053694   
11  1000      0.90         28     0.071969      0.732502     0.044836   
12  1000      0.75         25     0.095410      0.7

In [36]:
best_fold_scores = joint_results[
    (joint_results["k"] == 1000) &
    (joint_results["l1_ratio"] == 0.75) &
    (joint_results["alpha_idx"] == 27)
][
    ["fold", "alpha", "alpha_ratio", "c_index", "n_nonzero"]
]

print(best_fold_scores)
print()
print("Mean C-index:", best_fold_scores["c_index"].mean())
print("Std C-index :", best_fold_scores["c_index"].std())

      fold     alpha  alpha_ratio   c_index  n_nonzero
427      1  0.019692      0.07906  0.685687        124
1427     2  0.020340      0.07906  0.761435        146
2427     3  0.019587      0.07906  0.676337        138
3427     4  0.019327      0.07906  0.794195        141
4427     5  0.018333      0.07906  0.772455        143

Mean C-index: 0.7380218467685836
Std C-index : 0.05346315237762338


In [37]:
FINAL_K = 1000
FINAL_L1_RATIO = 0.75
FINAL_ALPHA_IDX = 27

# -------------------------
# 1. Train / Validation 환자 ID
# -------------------------
train_ids_final = train_df["patient_id"]
val_ids_final = val_df["patient_id"]

# -------------------------
# 2. 전체 Train 370명에서 저발현 필터
# TPM >= 1인 환자가 20% 이상
# -------------------------
min_samples_final = int(np.ceil(len(train_ids_final) * 0.20))

rna_train_raw_final = rna_tpm.loc[train_ids_final].copy()

keep_mask_final = (
    (rna_train_raw_final >= 1.0).sum(axis=0)
    >= min_samples_final
)

filtered_genes_final = rna_train_raw_final.columns[keep_mask_final]

# -------------------------
# 3. log2(TPM + 1)
# -------------------------
rna_train_log_final = np.log2(
    rna_train_raw_final.loc[:, filtered_genes_final] + 1
)

# -------------------------
# 4. 전체 Train에서 MAD 계산
# -------------------------
train_median_final = rna_train_log_final.median(axis=0)

train_mad_final = (
    rna_train_log_final
    .sub(train_median_final, axis=1)
    .abs()
    .median(axis=0)
    .sort_values(ascending=False)
)

# -------------------------
# 5. 최종 Top-1000 gene 고정
# -------------------------
final_genes = train_mad_final.head(FINAL_K).index.tolist()

# -------------------------
# 6. Train / Validation에 동일 gene 적용
# -------------------------
X_train_final = np.log2(
    rna_tpm.loc[train_ids_final, final_genes] + 1
)

X_val_final = np.log2(
    rna_tpm.loc[val_ids_final, final_genes] + 1
)

# -------------------------
# 7. Train 기준 scaling
# -------------------------
final_scaler = StandardScaler()

X_train_final_scaled = final_scaler.fit_transform(X_train_final)
X_val_final_scaled = final_scaler.transform(X_val_final)

print("저발현 필터 기준 환자 수:", min_samples_final)
print("필터 후 gene 수:", len(filtered_genes_final))
print("최종 gene 수:", len(final_genes))
print("Train:", X_train_final_scaled.shape)
print("Validation:", X_val_final_scaled.shape)

저발현 필터 기준 환자 수: 74
필터 후 gene 수: 14220
최종 gene 수: 1000
Train: (370, 1000)
Validation: (79, 1000)


In [38]:
# survival target
y_train_final = Surv.from_arrays(
    event=train_df["pfi_event"].astype(bool).to_numpy(),
    time=train_df["pfi_time"].astype(float).to_numpy()
)

y_val_final = Surv.from_arrays(
    event=val_df["pfi_event"].astype(bool).to_numpy(),
    time=val_df["pfi_time"].astype(float).to_numpy()
)

# 전체 Train 370명으로 최종 RNA CoxNet 학습
rna_coxnet_final = CoxnetSurvivalAnalysis(
    l1_ratio=FINAL_L1_RATIO,   # 0.75
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

rna_coxnet_final.fit(
    X_train_final_scaled,
    y_train_final
)

# CV에서 선택된 같은 regularization-path 위치
final_alpha = rna_coxnet_final.alphas_[FINAL_ALPHA_IDX]

# Validation risk score
val_risk = rna_coxnet_final.predict(
    X_val_final_scaled,
    alpha=final_alpha
)

# Validation Harrell C-index
val_c_index = concordance_index_censored(
    y_val_final["event"],
    y_val_final["time"],
    val_risk
)[0]

# 최종 alpha에서 0이 아닌 gene 수
final_nonzero = np.count_nonzero(
    rna_coxnet_final.coef_[:, FINAL_ALPHA_IDX]
)

print("Final K:", FINAL_K)
print("Final l1_ratio:", FINAL_L1_RATIO)
print("Final alpha index:", FINAL_ALPHA_IDX)
print("Final alpha:", final_alpha)
print("Non-zero genes:", final_nonzero)
print("Validation C-index:", val_c_index)

Final K: 1000
Final l1_ratio: 0.75
Final alpha index: 27
Final alpha: 0.019114577707188597
Non-zero genes: 144
Validation C-index: 0.7262617621899059


In [39]:
# 최종 alpha에서 coefficient 추출
final_coef = rna_coxnet_final.coef_[:, FINAL_ALPHA_IDX]

rna_coef_df = pd.DataFrame({
    "gene_id": final_genes,
    "coefficient": final_coef
})

# coefficient가 0이 아닌 gene만
rna_nonzero_df = (
    rna_coef_df[rna_coef_df["coefficient"] != 0]
    .copy()
)

# gene symbol 연결
rna_nonzero_df = (
    rna_nonzero_df
    .merge(
        gene_annotation[["gene_id", "gene_name"]],
        on="gene_id",
        how="left"
    )
)

# 절댓값이 큰 순서
rna_nonzero_df["abs_coefficient"] = (
    rna_nonzero_df["coefficient"].abs()
)

rna_nonzero_df = (
    rna_nonzero_df
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

print("Non-zero genes:", len(rna_nonzero_df))
print(rna_nonzero_df.head(20))

Non-zero genes: 144
               gene_id  coefficient gene_name  abs_coefficient
0   ENSG00000182013.18     0.521748    PNMA8A         0.521748
1   ENSG00000174564.13     0.510696    IL20RB         0.510696
2   ENSG00000153902.14     0.441033      LGI4         0.441033
3   ENSG00000075223.14    -0.429454    SEMA3C         0.429454
4    ENSG00000189058.9     0.427601      APOD         0.427601
5    ENSG00000147003.7    -0.408891     CLTRN         0.408891
6   ENSG00000124564.17    -0.394313   SLC17A3         0.394313
7   ENSG00000162897.16     0.349543     FCAMR         0.349543
8    ENSG00000186462.9    -0.301748    NAP1L2         0.301748
9    ENSG00000064300.9     0.298684      NGFR         0.298684
10   ENSG00000133169.6     0.287175      BEX1         0.287175
11  ENSG00000215915.10    -0.275589    ATAD3C         0.275589
12  ENSG00000052344.16    -0.275003     PRSS8         0.275003
13  ENSG00000163017.14    -0.263841     ACTG2         0.263841
14  ENSG00000196878.15    -0.261273

In [40]:
# 144개 non-zero gene 저장
output_path = data_dir / "processed" / "rna_coxnet_nonzero_genes.csv"

rna_nonzero_df.to_csv(
    output_path,
    index=False
)

print("저장 위치:", output_path)
print("저장 gene 수:", len(rna_nonzero_df))

저장 위치: ..\data\processed\rna_coxnet_nonzero_genes.csv
저장 gene 수: 144


In [41]:
clinical_features = ["age", "gender", "stage", "grade"]

print("=== dtype ===")
print(train_df[clinical_features].dtypes)

print("\n=== missing ===")
print(train_df[clinical_features].isna().sum())

for col in clinical_features:
    print(f"\n=== {col} ===")
    print(train_df[col].value_counts(dropna=False))

=== dtype ===
age       float64
gender        str
stage         str
grade         str
dtype: object

=== missing ===
age       0
gender    0
stage     0
grade     0
dtype: int64

=== age ===
age
63.0    15
60.0    14
57.0    13
75.0    13
69.0    12
51.0    12
67.0    12
58.0    12
62.0    12
61.0    11
72.0    11
68.0    11
79.0    10
54.0    10
76.0    10
53.0     9
52.0     9
65.0     9
46.0     8
64.0     8
74.0     8
59.0     7
73.0     7
40.0     7
56.0     7
77.0     7
49.0     7
47.0     7
41.0     6
48.0     6
70.0     6
43.0     6
71.0     5
55.0     5
50.0     5
42.0     4
66.0     4
81.0     4
78.0     4
39.0     3
44.0     3
83.0     3
45.0     3
82.0     3
34.0     2
85.0     2
86.0     2
37.0     2
38.0     2
84.0     2
90.0     2
26.0     1
80.0     1
36.0     1
33.0     1
29.0     1
88.0     1
35.0     1
32.0     1
Name: count, dtype: int64

=== gender ===
gender
MALE      239
FEMALE    131
Name: count, dtype: int64

=== stage ===
stage
Stage I      183
Stage III     9

In [42]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 사용할 임상 변수
numeric_features = ["age"]
categorical_features = ["gender", "stage", "grade"]

# Train에서 preprocessing 학습
clinical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "age",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                drop=[
                    "MALE",      # gender reference
                    "Stage I",   # stage reference
                    "G2"         # grade reference
                ],
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    remainder="drop"
)

X_clin_train = clinical_preprocessor.fit_transform(
    train_df[clinical_features]
)

X_clin_val = clinical_preprocessor.transform(
    val_df[clinical_features]
)

feature_names = clinical_preprocessor.get_feature_names_out()

print("Clinical Train:", X_clin_train.shape)
print("Clinical Validation:", X_clin_val.shape)

print("\nFeature names:")
for name in feature_names:
    print(name)

Clinical Train: (370, 10)
Clinical Validation: (79, 10)

Feature names:
age__age
cat__gender_FEMALE
cat__stage_Stage II
cat__stage_Stage III
cat__stage_Stage IV
cat__stage_Unknown
cat__grade_G1
cat__grade_G3
cat__grade_G4
cat__grade_Unknown


In [43]:
CLINICAL_L1_RATIOS = [0.1, 0.25, 0.5, 0.75, 0.9]

clinical_cv_results = []

# fold마다 열 구성이 달라지는 것을 막기 위해 category를 고정
clinical_categories = [
    ["MALE", "FEMALE"],
    ["Stage I", "Stage II", "Stage III", "Stage IV", "Unknown"],
    ["G1", "G2", "G3", "G4", "Unknown"]
]

for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(train_df, train_df["pfi_event"]),
    start=1
):
    fit_df = train_df.iloc[fit_idx].copy()
    cv_df_fold = train_df.iloc[cv_idx].copy()

    # 1. Fold Fit에서만 임상 preprocessing 학습
    fold_preprocessor = ColumnTransformer(
        transformers=[
            (
                "age",
                "passthrough",
                ["age"]
            ),
            (
                "cat",
                OneHotEncoder(
                    categories=clinical_categories,
                    drop=["MALE", "Stage I", "G2"],
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                ["gender", "stage", "grade"]
            )
        ]
    )

    X_fit_clin = fold_preprocessor.fit_transform(
        fit_df[clinical_features]
    )

    X_cv_clin = fold_preprocessor.transform(
        cv_df_fold[clinical_features]
    )

    # 2. Fold Fit 기준으로 10개 feature 전체 scaling
    fold_scaler = StandardScaler()

    X_fit_clin = fold_scaler.fit_transform(X_fit_clin)
    X_cv_clin = fold_scaler.transform(X_cv_clin)

    # 3. Survival target
    y_fit = Surv.from_arrays(
        event=fit_df["pfi_event"].astype(bool).to_numpy(),
        time=fit_df["pfi_time"].astype(float).to_numpy()
    )

    y_cv = Surv.from_arrays(
        event=cv_df_fold["pfi_event"].astype(bool).to_numpy(),
        time=cv_df_fold["pfi_time"].astype(float).to_numpy()
    )

    # 4. l1_ratio별 CoxNet
    for l1_ratio in CLINICAL_L1_RATIOS:

        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            n_alphas=50,
            alpha_min_ratio=0.01,
            max_iter=100000
        )

        model.fit(X_fit_clin, y_fit)

        alpha_max = model.alphas_[0]

        for alpha_idx, alpha in enumerate(model.alphas_):

            risk = model.predict(
                X_cv_clin,
                alpha=alpha
            )

            c_index = concordance_index_censored(
                y_cv["event"],
                y_cv["time"],
                risk
            )[0]

            clinical_cv_results.append({
                "fold": fold,
                "l1_ratio": l1_ratio,
                "alpha_idx": alpha_idx,
                "alpha": alpha,
                "alpha_ratio": alpha / alpha_max,
                "c_index": c_index,
                "n_nonzero": np.count_nonzero(
                    model.coef_[:, alpha_idx]
                )
            })

    print(f"Fold {fold} 완료")

clinical_cv_results = pd.DataFrame(clinical_cv_results)

print("총 결과 행:", len(clinical_cv_results))
print(clinical_cv_results.head())

Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료
총 결과 행: 1250
   fold  l1_ratio  alpha_idx     alpha  alpha_ratio   c_index  n_nonzero
0     1       0.1          0  3.082137     1.000000  0.500000          0
1     1       0.1          1  2.805664     0.910298  0.671663          1
2     1       0.1          2  2.553991     0.828643  0.671663          1
3     1       0.1          3  2.324893     0.754312  0.671663          1
4     1       0.1          4  2.116346     0.686649  0.671663          1


In [44]:
clinical_cv_summary = (
    clinical_cv_results
    .groupby(
        ["l1_ratio", "alpha_idx"],
        as_index=False
    )
    .agg(
        alpha_ratio=("alpha_ratio", "mean"),
        mean_c_index=("c_index", "mean"),
        std_c_index=("c_index", "std"),
        mean_nonzero=("n_nonzero", "mean")
    )
)

clinical_best = (
    clinical_cv_summary
    .sort_values(
        ["mean_c_index", "std_c_index"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print(clinical_best.head(20))

    l1_ratio  alpha_idx  alpha_ratio  mean_c_index  std_c_index  mean_nonzero
0       0.90         37     0.030888      0.822046     0.050878           8.6
1       0.90         39     0.025595      0.821883     0.051664           8.6
2       0.90         38     0.028118      0.821864     0.051101           8.6
3       0.90         42     0.019307      0.821707     0.053198           8.6
4       0.90         40     0.023300      0.821701     0.051890           8.6
5       0.75         38     0.028118      0.821683     0.051325           8.6
6       0.75         39     0.025595      0.821683     0.051325           8.6
7       0.75         36     0.033932      0.821551     0.052829           8.4
8       0.90         41     0.021210      0.821539     0.052469           8.6
9       0.75         43     0.017575      0.821526     0.053444           8.6
10      0.75         40     0.023300      0.821520     0.052118           8.6
11      0.90         36     0.033932      0.821366     0.053082 

In [45]:
best_clinical_fold_scores = clinical_cv_results[
    (clinical_cv_results["l1_ratio"] == 0.90) &
    (clinical_cv_results["alpha_idx"] == 37)
][
    ["fold", "alpha", "alpha_ratio", "c_index", "n_nonzero"]
]

print(best_clinical_fold_scores)

print()
print(
    "Mean C-index:",
    best_clinical_fold_scores["c_index"].mean()
)

print(
    "Std C-index :",
    best_clinical_fold_scores["c_index"].std()
)

      fold     alpha  alpha_ratio   c_index  n_nonzero
237      1  0.010578     0.030888  0.771760          7
487      2  0.009433     0.030888  0.830045          9
737      3  0.010447     0.030888  0.772439          9
987      4  0.009671     0.030888  0.844767          9
1237     5  0.010112     0.030888  0.891218          9

Mean C-index: 0.8220456592882399
Std C-index : 0.05087845261365547


In [46]:
FINAL_CLIN_L1_RATIO = 0.90
FINAL_CLIN_ALPHA_IDX = 37

# 1. 전체 Train에서 임상 preprocessing 학습
final_clin_preprocessor = ColumnTransformer(
    transformers=[
        (
            "age",
            "passthrough",
            ["age"]
        ),
        (
            "cat",
            OneHotEncoder(
                categories=[
                    ["MALE", "FEMALE"],
                    ["Stage I", "Stage II", "Stage III", "Stage IV", "Unknown"],
                    ["G1", "G2", "G3", "G4", "Unknown"]
                ],
                drop=["MALE", "Stage I", "G2"],
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["gender", "stage", "grade"]
        )
    ]
)

X_clin_train_raw = final_clin_preprocessor.fit_transform(
    train_df[clinical_features]
)

X_clin_val_raw = final_clin_preprocessor.transform(
    val_df[clinical_features]
)

# 2. 전체 Train 기준으로 10개 feature 표준화
final_clin_scaler = StandardScaler()

X_clin_train_final = final_clin_scaler.fit_transform(
    X_clin_train_raw
)

X_clin_val_final = final_clin_scaler.transform(
    X_clin_val_raw
)

# 3. Clinical-only CoxNet
clinical_coxnet_final = CoxnetSurvivalAnalysis(
    l1_ratio=FINAL_CLIN_L1_RATIO,
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

clinical_coxnet_final.fit(
    X_clin_train_final,
    y_train_final
)

# 4. CV에서 선택한 같은 alpha 위치 사용
clinical_final_alpha = (
    clinical_coxnet_final.alphas_[FINAL_CLIN_ALPHA_IDX]
)

# 5. Validation risk score
clinical_val_risk = clinical_coxnet_final.predict(
    X_clin_val_final,
    alpha=clinical_final_alpha
)

# 6. Validation C-index
clinical_val_cindex = concordance_index_censored(
    y_val_final["event"],
    y_val_final["time"],
    clinical_val_risk
)[0]

# 7. non-zero feature 수
clinical_nonzero = np.count_nonzero(
    clinical_coxnet_final.coef_[:, FINAL_CLIN_ALPHA_IDX]
)

print("Clinical l1_ratio:", FINAL_CLIN_L1_RATIO)
print("Clinical alpha index:", FINAL_CLIN_ALPHA_IDX)
print("Clinical alpha:", clinical_final_alpha)
print("Non-zero clinical features:", clinical_nonzero)
print("Clinical Validation C-index:", clinical_val_cindex)

print("\nRNA Validation C-index:", val_c_index)

Clinical l1_ratio: 0.9
Clinical alpha index: 37
Clinical alpha: 0.010057565713433423
Non-zero clinical features: 8
Clinical Validation C-index: 0.7840034217279727

RNA Validation C-index: 0.7262617621899059


In [47]:
# 최종 Clinical feature 이름
clinical_feature_names = (
    final_clin_preprocessor.get_feature_names_out()
)

# 선택된 alpha에서 coefficient
clinical_coef = clinical_coxnet_final.coef_[
    :, FINAL_CLIN_ALPHA_IDX
]

clinical_coef_df = pd.DataFrame({
    "feature": clinical_feature_names,
    "coefficient": clinical_coef
})

clinical_coef_df["abs_coefficient"] = (
    clinical_coef_df["coefficient"].abs()
)

clinical_nonzero_df = (
    clinical_coef_df[
        clinical_coef_df["coefficient"] != 0
    ]
    .sort_values(
        "abs_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Non-zero clinical features:", len(clinical_nonzero_df))
print(clinical_nonzero_df)

Non-zero clinical features: 8
                feature  coefficient  abs_coefficient
0   cat__stage_Stage IV     0.888577         0.888577
1  cat__stage_Stage III     0.513900         0.513900
2         cat__grade_G4     0.393142         0.393142
3   cat__stage_Stage II     0.195621         0.195621
4         cat__grade_G3     0.178655         0.178655
5    cat__gender_FEMALE    -0.146482         0.146482
6              age__age     0.035419         0.035419
7         cat__grade_G1     0.026393         0.026393


In [48]:
# Clinical CoxNet 전체 coefficient 저장
clinical_coef_path = (
    data_dir / "processed" / "clinical_coxnet_coefficients.csv"
)

clinical_coef_df.to_csv(
    clinical_coef_path,
    index=False
)

# non-zero 8개 feature도 별도 저장
clinical_nonzero_path = (
    data_dir / "processed" / "clinical_coxnet_nonzero_features.csv"
)

clinical_nonzero_df.to_csv(
    clinical_nonzero_path,
    index=False
)

print("전체 Clinical coefficient 저장:", clinical_coef_path)
print("Non-zero Clinical feature 저장:", clinical_nonzero_path)
print("전체 feature 수:", len(clinical_coef_df))
print("Non-zero feature 수:", len(clinical_nonzero_df))

전체 Clinical coefficient 저장: ..\data\processed\clinical_coxnet_coefficients.csv
Non-zero Clinical feature 저장: ..\data\processed\clinical_coxnet_nonzero_features.csv
전체 feature 수: 10
Non-zero feature 수: 8


In [49]:
# =========================================
# RNA-only + Clinical-only 결과 저장
# =========================================

# 1. RNA 전체 Top-1000 coefficient
rna_coef_df = pd.DataFrame({
    "gene_id": final_genes,
    "coefficient": rna_coxnet_final.coef_[:, FINAL_ALPHA_IDX]
})

rna_coef_df["abs_coefficient"] = rna_coef_df["coefficient"].abs()

rna_coef_df.to_csv(
    data_dir / "processed" / "rna_coxnet_coefficients.csv",
    index=False
)

# RNA non-zero 144개
rna_nonzero_df.to_csv(
    data_dir / "processed" / "rna_coxnet_nonzero_genes.csv",
    index=False
)

# 2. Clinical 전체 10개 coefficient
clinical_coef_df.to_csv(
    data_dir / "processed" / "clinical_coxnet_coefficients.csv",
    index=False
)

# Clinical non-zero 8개
clinical_nonzero_df.to_csv(
    data_dir / "processed" / "clinical_coxnet_nonzero_features.csv",
    index=False
)

print("=== 저장 완료 ===")
print("RNA 전체:", len(rna_coef_df))
print("RNA non-zero:", len(rna_nonzero_df))
print("Clinical 전체:", len(clinical_coef_df))
print("Clinical non-zero:", len(clinical_nonzero_df))

=== 저장 완료 ===
RNA 전체: 1000
RNA non-zero: 144
Clinical 전체: 10
Clinical non-zero: 8


In [50]:
# Clinical + RNA : Fold 1 결합 QC

fit_idx, cv_idx = next(
    cv.split(train_df, train_df["pfi_event"])
)

fit_df = train_df.iloc[fit_idx].copy()
cv_df_fold = train_df.iloc[cv_idx].copy()

# -------------------------
# Clinical
# -------------------------
combined_clin_preprocessor = ColumnTransformer(
    transformers=[
        ("age", "passthrough", ["age"]),
        (
            "cat",
            OneHotEncoder(
                categories=[
                    ["MALE", "FEMALE"],
                    ["Stage I", "Stage II", "Stage III", "Stage IV", "Unknown"],
                    ["G1", "G2", "G3", "G4", "Unknown"]
                ],
                drop=["MALE", "Stage I", "G2"],
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["gender", "stage", "grade"]
        )
    ]
)

X_fit_clin = combined_clin_preprocessor.fit_transform(
    fit_df[clinical_features]
)

X_cv_clin = combined_clin_preprocessor.transform(
    cv_df_fold[clinical_features]
)

# -------------------------
# RNA
# -------------------------
rna_fit_raw = rna_tpm.loc[fit_df["patient_id"]]
rna_cv_raw = rna_tpm.loc[cv_df_fold["patient_id"]]

# 저발현 필터
min_samples = int(np.ceil(len(fit_df) * 0.20))

keep_genes = (
    (rna_fit_raw >= 1).sum(axis=0)
    >= min_samples
)

rna_fit_filtered = rna_fit_raw.loc[:, keep_genes]
rna_cv_filtered = rna_cv_raw.loc[:, keep_genes]

# log2(TPM + 1)
rna_fit_log = np.log2(rna_fit_filtered + 1)
rna_cv_log = np.log2(rna_cv_filtered + 1)

# MAD 기준 Top 1000
rna_median = rna_fit_log.median(axis=0)

rna_mad = (
    rna_fit_log
    .sub(rna_median, axis=1)
    .abs()
    .median(axis=0)
)

top1000_genes = (
    rna_mad
    .sort_values(ascending=False)
    .head(1000)
    .index
)

X_fit_rna = rna_fit_log[top1000_genes].to_numpy()
X_cv_rna = rna_cv_log[top1000_genes].to_numpy()

# -------------------------
# Clinical + RNA 결합
# -------------------------
X_fit_combined_raw = np.hstack([
    X_fit_clin,
    X_fit_rna
])

X_cv_combined_raw = np.hstack([
    X_cv_clin,
    X_cv_rna
])

print("Fold Fit patients:", len(fit_df))
print("Fold CV patients :", len(cv_df_fold))
print("Clinical features:", X_fit_clin.shape[1])
print("RNA features     :", X_fit_rna.shape[1])
print("Combined Fit     :", X_fit_combined_raw.shape)
print("Combined CV      :", X_cv_combined_raw.shape)

Fold Fit patients: 296
Fold CV patients : 74
Clinical features: 10
RNA features     : 1000
Combined Fit     : (296, 1010)
Combined CV      : (74, 1010)


In [51]:
# =========================================
# Clinical + RNA CoxNet
# 5-fold CV hyperparameter tuning
# =========================================

COMBINED_K_VALUES = [500, 1000, 2000, 5000]
COMBINED_L1_RATIOS = [0.1, 0.25, 0.5, 0.75, 0.9]

combined_cv_results = []

for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(train_df, train_df["pfi_event"]),
    start=1
):
    fit_df = train_df.iloc[fit_idx].copy()
    cv_df_fold = train_df.iloc[cv_idx].copy()

    # ---------------------------------
    # 1. Clinical preprocessing
    # ---------------------------------
    fold_clin_preprocessor = ColumnTransformer(
        transformers=[
            ("age", "passthrough", ["age"]),
            (
                "cat",
                OneHotEncoder(
                    categories=[
                        ["MALE", "FEMALE"],
                        ["Stage I", "Stage II", "Stage III", "Stage IV", "Unknown"],
                        ["G1", "G2", "G3", "G4", "Unknown"]
                    ],
                    drop=["MALE", "Stage I", "G2"],
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                ["gender", "stage", "grade"]
            )
        ]
    )

    X_fit_clin = fold_clin_preprocessor.fit_transform(
        fit_df[clinical_features]
    )

    X_cv_clin = fold_clin_preprocessor.transform(
        cv_df_fold[clinical_features]
    )

    # ---------------------------------
    # 2. RNA preprocessing
    #    Fold Fit에서만 결정
    # ---------------------------------
    rna_fit_raw = rna_tpm.loc[fit_df["patient_id"]]
    rna_cv_raw = rna_tpm.loc[cv_df_fold["patient_id"]]

    # 저발현 필터
    min_samples = int(np.ceil(len(fit_df) * 0.20))

    keep_genes = (
        (rna_fit_raw >= 1).sum(axis=0)
        >= min_samples
    )

    rna_fit_filtered = rna_fit_raw.loc[:, keep_genes]
    rna_cv_filtered = rna_cv_raw.loc[:, keep_genes]

    # log2(TPM + 1)
    rna_fit_log = np.log2(rna_fit_filtered + 1)
    rna_cv_log = np.log2(rna_cv_filtered + 1)

    # MAD
    rna_median = rna_fit_log.median(axis=0)

    rna_mad = (
        rna_fit_log
        .sub(rna_median, axis=1)
        .abs()
        .median(axis=0)
        .sort_values(ascending=False)
    )

    # Survival target
    y_fit = Surv.from_arrays(
        event=fit_df["pfi_event"].astype(bool).to_numpy(),
        time=fit_df["pfi_time"].astype(float).to_numpy()
    )

    y_cv = Surv.from_arrays(
        event=cv_df_fold["pfi_event"].astype(bool).to_numpy(),
        time=cv_df_fold["pfi_time"].astype(float).to_numpy()
    )

    # ---------------------------------
    # 3. RNA Top-K별 결합 모델
    # ---------------------------------
    for k in COMBINED_K_VALUES:

        top_genes = rna_mad.head(k).index

        X_fit_rna = rna_fit_log[top_genes].to_numpy()
        X_cv_rna = rna_cv_log[top_genes].to_numpy()

        # Clinical + RNA
        X_fit_combined = np.hstack([
            X_fit_clin,
            X_fit_rna
        ])

        X_cv_combined = np.hstack([
            X_cv_clin,
            X_cv_rna
        ])

        # Fold Fit 기준 scaling
        scaler = StandardScaler()

        X_fit_combined = scaler.fit_transform(
            X_fit_combined
        )

        X_cv_combined = scaler.transform(
            X_cv_combined
        )

        # ---------------------------------
        # 4. CoxNet tuning
        # ---------------------------------
        for l1_ratio in COMBINED_L1_RATIOS:

            model = CoxnetSurvivalAnalysis(
                l1_ratio=l1_ratio,
                n_alphas=50,
                alpha_min_ratio=0.01,
                max_iter=100000
            )

            model.fit(
                X_fit_combined,
                y_fit
            )

            alpha_max = model.alphas_[0]

            for alpha_idx, alpha in enumerate(model.alphas_):

                risk = model.predict(
                    X_cv_combined,
                    alpha=alpha
                )

                c_index = concordance_index_censored(
                    y_cv["event"],
                    y_cv["time"],
                    risk
                )[0]

                combined_cv_results.append({
                    "fold": fold,
                    "k": k,
                    "l1_ratio": l1_ratio,
                    "alpha_idx": alpha_idx,
                    "alpha": alpha,
                    "alpha_ratio": alpha / alpha_max,
                    "c_index": c_index,
                    "n_nonzero": np.count_nonzero(
                        model.coef_[:, alpha_idx]
                    )
                })

        print(f"Fold {fold} | K={k} 완료")


combined_cv_results = pd.DataFrame(
    combined_cv_results
)

print("\n=== 완료 ===")
print("총 결과 행:", len(combined_cv_results))
print(combined_cv_results.head())

Fold 1 | K=500 완료
Fold 1 | K=1000 완료
Fold 1 | K=2000 완료
Fold 1 | K=5000 완료
Fold 2 | K=500 완료
Fold 2 | K=1000 완료
Fold 2 | K=2000 완료
Fold 2 | K=5000 완료
Fold 3 | K=500 완료
Fold 3 | K=1000 완료
Fold 3 | K=2000 완료
Fold 3 | K=5000 완료
Fold 4 | K=500 완료
Fold 4 | K=1000 완료
Fold 4 | K=2000 완료
Fold 4 | K=5000 완료
Fold 5 | K=500 완료
Fold 5 | K=1000 완료
Fold 5 | K=2000 완료
Fold 5 | K=5000 완료

=== 완료 ===
총 결과 행: 5000
   fold    k  l1_ratio  alpha_idx     alpha  alpha_ratio   c_index  n_nonzero
0     1  500       0.1          0  3.082137     1.000000  0.500000          0
1     1  500       0.1          1  2.805664     0.910298  0.671663          1
2     1  500       0.1          2  2.553991     0.828643  0.671663          1
3     1  500       0.1          3  2.324893     0.754312  0.671663          1
4     1  500       0.1          4  2.116346     0.686649  0.671663          1


In [52]:
combined_cv_summary = (
    combined_cv_results
    .groupby(
        ["k", "l1_ratio", "alpha_idx"],
        as_index=False
    )
    .agg(
        alpha_ratio=("alpha_ratio", "mean"),
        mean_c_index=("c_index", "mean"),
        std_c_index=("c_index", "std"),
        mean_nonzero=("n_nonzero", "mean")
    )
)

combined_best = (
    combined_cv_summary
    .sort_values(
        ["mean_c_index", "std_c_index"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print(combined_best.head(20))

      k  l1_ratio  alpha_idx  alpha_ratio  mean_c_index  std_c_index  \
0   500      0.50         18     0.184207      0.810671     0.047333   
1   500      0.50         17     0.202359      0.810645     0.047177   
2   500      0.50         16     0.222300      0.810572     0.044511   
3   500      0.75         19     0.167683      0.810308     0.048133   
4   500      0.75         18     0.184207      0.810121     0.045245   
5   500      0.90         18     0.184207      0.810095     0.046797   
6   500      0.75         15     0.244205      0.809689     0.040739   
7   500      0.90         19     0.167683      0.809435     0.048324   
8   500      0.50         19     0.167683      0.809279     0.048531   
9   500      0.50         20     0.152642      0.809235     0.048937   
10  500      0.25         20     0.152642      0.808493     0.046898   
11  500      0.90         15     0.244205      0.808488     0.041305   
12  500      0.25         17     0.202359      0.808397     0.04

In [53]:
best_combined_fold_scores = combined_cv_results[
    (combined_cv_results["k"] == 500) &
    (combined_cv_results["l1_ratio"] == 0.50) &
    (combined_cv_results["alpha_idx"] == 18)
][
    ["fold", "alpha", "alpha_ratio", "c_index", "n_nonzero"]
]

print(best_combined_fold_scores)

print()
print(
    "Mean C-index:",
    best_combined_fold_scores["c_index"].mean()
)

print(
    "Std C-index:",
    best_combined_fold_scores["c_index"].std()
)

      fold     alpha  alpha_ratio   c_index  n_nonzero
118      1  0.113550     0.184207  0.785300         28
1118     2  0.101262     0.184207  0.843946         32
2118     3  0.112139     0.184207  0.739801         24
3118     4  0.103817     0.184207  0.832014         25
4118     5  0.108553     0.184207  0.852295         27

Mean C-index: 0.8106712040378895
Std C-index: 0.047333062756462964


In [54]:
# =========================================
# Final Clinical + RNA CoxNet
# Train 370명 재학습 → Validation 평가
# =========================================

FINAL_COMBINED_K = 500
FINAL_COMBINED_L1 = 0.50
FINAL_COMBINED_ALPHA_IDX = 18

# -----------------------------------------
# 1. Clinical: 전체 Train에서 preprocessing
# -----------------------------------------
final_combined_clin_preprocessor = ColumnTransformer(
    transformers=[
        ("age", "passthrough", ["age"]),
        (
            "cat",
            OneHotEncoder(
                categories=[
                    ["MALE", "FEMALE"],
                    ["Stage I", "Stage II", "Stage III", "Stage IV", "Unknown"],
                    ["G1", "G2", "G3", "G4", "Unknown"]
                ],
                drop=["MALE", "Stage I", "G2"],
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["gender", "stage", "grade"]
        )
    ]
)

X_train_clin_combined = (
    final_combined_clin_preprocessor
    .fit_transform(train_df[clinical_features])
)

X_val_clin_combined = (
    final_combined_clin_preprocessor
    .transform(val_df[clinical_features])
)


# -----------------------------------------
# 2. RNA: 전체 Train에서 다시 feature selection
# -----------------------------------------
rna_train_raw = rna_tpm.loc[train_df["patient_id"]]
rna_val_raw = rna_tpm.loc[val_df["patient_id"]]

# 저발현 필터 - Train에서만 결정
min_samples = int(np.ceil(len(train_df) * 0.20))

keep_genes = (
    (rna_train_raw >= 1).sum(axis=0)
    >= min_samples
)

rna_train_filtered = rna_train_raw.loc[:, keep_genes]
rna_val_filtered = rna_val_raw.loc[:, keep_genes]

# log2(TPM + 1)
rna_train_log = np.log2(rna_train_filtered + 1)
rna_val_log = np.log2(rna_val_filtered + 1)

# Train MAD
rna_train_median = rna_train_log.median(axis=0)

rna_train_mad = (
    rna_train_log
    .sub(rna_train_median, axis=1)
    .abs()
    .median(axis=0)
    .sort_values(ascending=False)
)

# Top 500
final_combined_genes = (
    rna_train_mad
    .head(FINAL_COMBINED_K)
    .index
)

X_train_rna_combined = (
    rna_train_log[final_combined_genes].to_numpy()
)

X_val_rna_combined = (
    rna_val_log[final_combined_genes].to_numpy()
)


# -----------------------------------------
# 3. Clinical + RNA 결합
# -----------------------------------------
X_train_combined_raw = np.hstack([
    X_train_clin_combined,
    X_train_rna_combined
])

X_val_combined_raw = np.hstack([
    X_val_clin_combined,
    X_val_rna_combined
])

# Train 기준 scaling
final_combined_scaler = StandardScaler()

X_train_combined = final_combined_scaler.fit_transform(
    X_train_combined_raw
)

X_val_combined = final_combined_scaler.transform(
    X_val_combined_raw
)


# -----------------------------------------
# 4. 최종 CoxNet 학습
# -----------------------------------------
combined_coxnet_final = CoxnetSurvivalAnalysis(
    l1_ratio=FINAL_COMBINED_L1,
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

combined_coxnet_final.fit(
    X_train_combined,
    y_train_final
)

combined_final_alpha = (
    combined_coxnet_final.alphas_[
        FINAL_COMBINED_ALPHA_IDX
    ]
)


# -----------------------------------------
# 5. Validation 평가
# -----------------------------------------
combined_val_risk = combined_coxnet_final.predict(
    X_val_combined,
    alpha=combined_final_alpha
)

combined_val_cindex = concordance_index_censored(
    y_val_final["event"],
    y_val_final["time"],
    combined_val_risk
)[0]

combined_coef = combined_coxnet_final.coef_[
    :, FINAL_COMBINED_ALPHA_IDX
]

print("Combined RNA Top-K:", FINAL_COMBINED_K)
print("Combined l1_ratio:", FINAL_COMBINED_L1)
print("Combined alpha index:", FINAL_COMBINED_ALPHA_IDX)
print("Combined alpha:", combined_final_alpha)

print(
    "Combined total non-zero:",
    np.count_nonzero(combined_coef)
)

print("\n=== Validation C-index ===")
print("Clinical only :", clinical_val_cindex)
print("RNA only      :", val_c_index)
print("Clinical + RNA:", combined_val_cindex)

Combined RNA Top-K: 500
Combined l1_ratio: 0.5
Combined alpha index: 18
Combined alpha: 0.1079631601142658
Combined total non-zero: 24

=== Validation C-index ===
Clinical only : 0.7840034217279727
RNA only      : 0.7262617621899059
Clinical + RNA: 0.8135158254918734


In [55]:
# =========================================
# Clinical + RNA coefficient 확인 및 저장
# =========================================

# Clinical feature 이름 10개
combined_clinical_names = (
    final_combined_clin_preprocessor
    .get_feature_names_out()
)

# RNA gene 이름 500개
combined_rna_names = list(final_combined_genes)

# 전체 feature 이름
combined_feature_names = (
    list(combined_clinical_names)
    + combined_rna_names
)

# feature type 표시
combined_feature_types = (
    ["Clinical"] * len(combined_clinical_names)
    + ["RNA"] * len(combined_rna_names)
)

combined_coef_df = pd.DataFrame({
    "feature": combined_feature_names,
    "feature_type": combined_feature_types,
    "coefficient": combined_coef
})

combined_coef_df["abs_coefficient"] = (
    combined_coef_df["coefficient"].abs()
)

# non-zero만
combined_nonzero_df = (
    combined_coef_df[
        combined_coef_df["coefficient"] != 0
    ]
    .sort_values(
        "abs_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

print("전체 feature:", len(combined_coef_df))
print("전체 non-zero:", len(combined_nonzero_df))

print("\n=== Non-zero breakdown ===")
print(
    combined_nonzero_df["feature_type"]
    .value_counts()
)

print("\n=== Non-zero features ===")
print(combined_nonzero_df)

# 저장
combined_coef_df.to_csv(
    data_dir / "processed" / "clinical_rna_coxnet_coefficients.csv",
    index=False
)

combined_nonzero_df.to_csv(
    data_dir / "processed" / "clinical_rna_coxnet_nonzero_features.csv",
    index=False
)

print("\n저장 완료")


전체 feature: 510
전체 non-zero: 24

=== Non-zero breakdown ===
feature_type
RNA         21
Clinical     3
Name: count, dtype: int64

=== Non-zero features ===
                 feature feature_type  coefficient  abs_coefficient
0    cat__stage_Stage IV     Clinical     0.615046         0.615046
1          cat__grade_G4     Clinical     0.144119         0.144119
2   cat__stage_Stage III     Clinical     0.102483         0.102483
3      ENSG00000147003.7          RNA    -0.102028         0.102028
4      ENSG00000184434.8          RNA    -0.074754         0.074754
5     ENSG00000119630.14          RNA     0.067019         0.067019
6      ENSG00000256235.2          RNA     0.064375         0.064375
7     ENSG00000074276.10          RNA    -0.063305         0.063305
8     ENSG00000119411.11          RNA    -0.053348         0.053348
9     ENSG00000103460.17          RNA    -0.045376         0.045376
10    ENSG00000162897.16          RNA     0.042413         0.042413
11    ENSG00000164309.15    

In [56]:
# =========================================
# Clinical + RNA 결과에 gene symbol 추가
# =========================================

gene_annotation = pd.read_parquet(
    data_dir / "interim" / "rna_gene_annotation.parquet"
)

# RNA feature만 gene annotation과 연결
combined_rna_nonzero = combined_nonzero_df[
    combined_nonzero_df["feature_type"] == "RNA"
].copy()

combined_rna_nonzero = combined_rna_nonzero.merge(
    gene_annotation[["gene_id", "gene_name"]],
    left_on="feature",
    right_on="gene_id",
    how="left"
)

# Clinical feature
combined_clin_nonzero = combined_nonzero_df[
    combined_nonzero_df["feature_type"] == "Clinical"
].copy()

print("=== Clinical non-zero ===")
print(
    combined_clin_nonzero[
        ["feature", "coefficient"]
    ]
)

print("\n=== RNA non-zero ===")
print(
    combined_rna_nonzero[
        ["feature", "gene_name", "coefficient"]
    ]
)

# gene symbol 포함 결과 저장
combined_rna_nonzero.to_csv(
    data_dir / "processed" / "clinical_rna_nonzero_genes_annotated.csv",
    index=False
)

print("\n저장 완료")

=== Clinical non-zero ===
                feature  coefficient
0   cat__stage_Stage IV     0.615046
1         cat__grade_G4     0.144119
2  cat__stage_Stage III     0.102483

=== RNA non-zero ===
               feature gene_name  coefficient
0    ENSG00000147003.7     CLTRN    -0.102028
1    ENSG00000184434.8    LRRC19    -0.074754
2   ENSG00000119630.14       PGF     0.067019
3    ENSG00000256235.2     SMIM3     0.064375
4   ENSG00000074276.10     CDHR2    -0.063305
5   ENSG00000119411.11     BSPRY    -0.053348
6   ENSG00000103460.17      TOX3    -0.045376
7   ENSG00000162897.16     FCAMR     0.042413
8   ENSG00000164309.15     CMYA5    -0.039674
9    ENSG00000125255.7   SLC10A2    -0.033247
10  ENSG00000131482.10      G6PC    -0.028536
11  ENSG00000179913.11    B3GNT3    -0.027739
12  ENSG00000145284.12      SCD5    -0.025438
13   ENSG00000162998.5      FRZB     0.021222
14  ENSG00000166415.15     WDR72    -0.019788
15   ENSG00000103257.9    SLC7A5     0.015134
16  ENSG00000117472.10

In [57]:
hallmark_dir = data_dir / "external" / "msigdb"
hallmark_dir.mkdir(parents=True, exist_ok=True)

In [58]:
print("Hallmark directory:", hallmark_dir)
print("Exists:", hallmark_dir.exists())

Hallmark directory: ..\data\external\msigdb
Exists: True


In [59]:
# Hallmark GMT 파일 확인
gmt_files = list(hallmark_dir.glob("h.all*.symbols.gmt"))

print("찾은 GMT 파일:", len(gmt_files))

for f in gmt_files:
    print(" -", f.name)

assert len(gmt_files) == 1, "Hallmark symbols GMT 파일이 정확히 1개 있어야 합니다."

hallmark_gmt_path = gmt_files[0]

# GMT 읽기
hallmark_sets = {}

with open(hallmark_gmt_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        pathway_name = parts[0]
        genes = parts[2:]   # 1번째=pathway명, 2번째=설명/URL, 이후=gene symbols

        hallmark_sets[pathway_name] = genes

print("\nHallmark pathway 수:", len(hallmark_sets))

print("\n처음 5개 pathway:")
for name in list(hallmark_sets.keys())[:5]:
    print(name, ":", len(hallmark_sets[name]), "genes")

찾은 GMT 파일: 1
 - h.all.v2026.1.Hs.symbols.gmt

Hallmark pathway 수: 50

처음 5개 pathway:
HALLMARK_ADIPOGENESIS : 200 genes
HALLMARK_ALLOGRAFT_REJECTION : 200 genes
HALLMARK_ANDROGEN_RESPONSE : 101 genes
HALLMARK_ANGIOGENESIS : 36 genes
HALLMARK_APICAL_JUNCTION : 200 genes


In [60]:
# =========================================
# Hallmark ↔ TCGA-KIRC gene symbol 매칭 QC
# =========================================

gene_annotation = pd.read_parquet(
    data_dir / "interim" / "rna_gene_annotation.parquet"
)

# 우리 RNA 데이터에 존재하는 gene symbol
available_symbols = set(
    gene_annotation["gene_name"]
    .dropna()
    .astype(str)
)

print("Protein-coding Ensembl genes:", len(gene_annotation))
print("Unique gene symbols:", len(available_symbols))

# gene symbol 중복 확인
symbol_counts = gene_annotation["gene_name"].value_counts()

duplicate_symbols = symbol_counts[
    symbol_counts > 1
]

print("Duplicated gene symbols:", len(duplicate_symbols))


# Hallmark별 매칭률
hallmark_coverage = []

for pathway, genes in hallmark_sets.items():

    hallmark_genes = set(genes)

    matched_genes = (
        hallmark_genes & available_symbols
    )

    hallmark_coverage.append({
        "pathway": pathway,
        "hallmark_genes": len(hallmark_genes),
        "matched_genes": len(matched_genes),
        "coverage": len(matched_genes) / len(hallmark_genes)
    })


hallmark_coverage_df = pd.DataFrame(
    hallmark_coverage
).sort_values("coverage")


print("\n=== Hallmark matching summary ===")
print(
    hallmark_coverage_df[
        ["hallmark_genes", "matched_genes", "coverage"]
    ].describe()
)

print("\n=== Coverage가 가장 낮은 10개 pathway ===")
print(
    hallmark_coverage_df.head(10).to_string(index=False)
)

Protein-coding Ensembl genes: 19962
Unique gene symbols: 19938
Duplicated gene symbols: 24

=== Hallmark matching summary ===
       hallmark_genes  matched_genes   coverage
count        50.00000      50.000000  50.000000
mean        146.44000     145.840000   0.996360
std          62.26086      61.942488   0.004748
min          32.00000      32.000000   0.982759
25%          98.00000      97.750000   0.991511
50%         180.00000     179.500000   1.000000
75%         200.00000     199.000000   1.000000
max         200.00000     200.000000   1.000000

=== Coverage가 가장 낮은 10개 pathway ===
                           pathway  hallmark_genes  matched_genes  coverage
           HALLMARK_MYC_TARGETS_V2              58             57  0.982759
    HALLMARK_FATTY_ACID_METABOLISM             158            156  0.987342
   HALLMARK_ESTROGEN_RESPONSE_LATE             200            198  0.990000
          HALLMARK_APICAL_JUNCTION             200            198  0.990000
HALLMARK_INTERFERON_GAMMA

In [61]:
# =========================================
# 중복 gene symbol 상세 확인
# =========================================

duplicate_symbol_names = (
    gene_annotation["gene_name"]
    .value_counts()
    .loc[lambda x: x > 1]
    .index
)

duplicate_annotation = (
    gene_annotation[
        gene_annotation["gene_name"].isin(
            duplicate_symbol_names
        )
    ]
    .sort_values(
        ["gene_name", "gene_id"]
    )
    .reset_index(drop=True)
)

print("Duplicated symbols:", len(duplicate_symbol_names))
print("Corresponding Ensembl rows:", len(duplicate_annotation))

print(
    duplicate_annotation[
        ["gene_id", "gene_name", "gene_type"]
    ].to_string(index=False)
)

Duplicated symbols: 24
Corresponding Ensembl rows: 48
                 gene_id gene_name      gene_type
       ENSG00000182584.6    ACTL10 protein_coding
       ENSG00000288649.1    ACTL10 protein_coding
      ENSG00000197976.12   AKAP17A protein_coding
ENSG00000197976.12_PAR_Y   AKAP17A protein_coding
      ENSG00000196433.13      ASMT protein_coding
ENSG00000196433.13_PAR_Y      ASMT protein_coding
      ENSG00000169093.16     ASMTL protein_coding
ENSG00000169093.16_PAR_Y     ASMTL protein_coding
      ENSG00000002586.20      CD99 protein_coding
ENSG00000002586.20_PAR_Y      CD99 protein_coding
      ENSG00000205755.12     CRLF2 protein_coding
ENSG00000205755.12_PAR_Y     CRLF2 protein_coding
      ENSG00000198223.17    CSF2RA protein_coding
ENSG00000198223.17_PAR_Y    CSF2RA protein_coding
      ENSG00000169084.15     DHRSX protein_coding
ENSG00000169084.15_PAR_Y     DHRSX protein_coding
      ENSG00000178605.13    GTPBP6 protein_coding
ENSG00000178605.13_PAR_Y    GTPBP6 protein_cod

In [62]:
# =========================================
# 중복 gene symbol 중 Hallmark에 실제 포함되는 것 확인
# =========================================

# Hallmark 50개에 포함된 모든 gene symbol
all_hallmark_symbols = set()

for genes in hallmark_sets.values():
    all_hallmark_symbols.update(genes)

# duplicated symbol 중 Hallmark에 포함되는 것
duplicate_in_hallmark = sorted(
    set(duplicate_symbol_names) & all_hallmark_symbols
)

print("전체 duplicated symbols:", len(duplicate_symbol_names))
print("Hallmark에 포함된 duplicated symbols:", len(duplicate_in_hallmark))

print("\n=== Hallmark에 실제 포함되는 중복 symbol ===")
for symbol in duplicate_in_hallmark:
    print(symbol)

전체 duplicated symbols: 24
Hallmark에 포함된 duplicated symbols: 7

=== Hallmark에 실제 포함되는 중복 symbol ===
CD99
CRLF2
CSF2RA
IL3RA
IL9R
SLC25A6
VAMP7


In [63]:
# Hallmark에 포함된 중복 symbol이 어떤 pathway에 들어가는지 확인

duplicate_hallmark_usage = []

for pathway, genes in hallmark_sets.items():
    for symbol in duplicate_in_hallmark:
        if symbol in genes:
            duplicate_hallmark_usage.append({
                "gene_symbol": symbol,
                "pathway": pathway
            })

duplicate_hallmark_usage_df = pd.DataFrame(
    duplicate_hallmark_usage
).sort_values(
    ["gene_symbol", "pathway"]
)

print(duplicate_hallmark_usage_df.to_string(index=False))

print(
    "\n중복 symbol별 pathway 개수:"
)

print(
    duplicate_hallmark_usage_df[
        "gene_symbol"
    ].value_counts()
)

gene_symbol                            pathway
       CD99           HALLMARK_APICAL_JUNCTION
      CRLF2   HALLMARK_IL6_JAK_STAT3_SIGNALING
     CSF2RA   HALLMARK_IL6_JAK_STAT3_SIGNALING
     CSF2RA         HALLMARK_KRAS_SIGNALING_UP
      IL3RA       HALLMARK_IL2_STAT5_SIGNALING
      IL3RA   HALLMARK_IL6_JAK_STAT3_SIGNALING
       IL9R   HALLMARK_IL6_JAK_STAT3_SIGNALING
    SLC25A6 HALLMARK_OXIDATIVE_PHOSPHORYLATION
      VAMP7         HALLMARK_PROTEIN_SECRETION

중복 symbol별 pathway 개수:
gene_symbol
CSF2RA     2
IL3RA      2
CD99       1
CRLF2      1
IL9R       1
SLC25A6    1
VAMP7      1
Name: count, dtype: int64


In [64]:
# =========================================
# Hallmark용 gene-symbol TPM matrix 생성
# =========================================

# Hallmark 50개에 포함된 모든 gene symbol
all_hallmark_symbols = set()

for genes in hallmark_sets.values():
    all_hallmark_symbols.update(genes)

# Hallmark에 포함되고 RNA matrix에도 존재하는 annotation만 사용
hallmark_annotation = gene_annotation[
    gene_annotation["gene_name"].isin(all_hallmark_symbols)
].copy()

hallmark_annotation = hallmark_annotation[
    hallmark_annotation["gene_id"].isin(rna_tpm.columns)
]

# Ensembl ID → gene symbol 매핑
gene_id_to_symbol = dict(
    zip(
        hallmark_annotation["gene_id"],
        hallmark_annotation["gene_name"]
    )
)

# Hallmark 관련 TPM만 선택
rna_hallmark_tpm = rna_tpm[
    list(gene_id_to_symbol.keys())
].copy()

# column을 Ensembl ID → gene symbol로 변경
rna_hallmark_tpm = rna_hallmark_tpm.rename(
    columns=gene_id_to_symbol
)

# 같은 gene symbol이 여러 Ensembl ID에 있으면 TPM 합산
rna_hallmark_symbol_tpm = (
    rna_hallmark_tpm
    .T
    .groupby(level=0)
    .sum()
    .T
)

print("환자 수:", rna_hallmark_symbol_tpm.shape[0])
print("Hallmark용 unique gene symbols:", rna_hallmark_symbol_tpm.shape[1])
print("중복 column:", rna_hallmark_symbol_tpm.columns.duplicated().sum())

print("\n예시 7개 중복 symbol 존재 여부:")
for symbol in duplicate_in_hallmark:
    print(symbol, symbol in rna_hallmark_symbol_tpm.columns)

환자 수: 529
Hallmark용 unique gene symbols: 4360
중복 column: 0

예시 7개 중복 symbol 존재 여부:
CD99 True
CRLF2 True
CSF2RA True
IL3RA True
IL9R True
SLC25A6 True
VAMP7 True


In [65]:
# =========================================
# Train 기준 Hallmark pathway score 생성
# =========================================

# Train / Validation 환자 분리
hallmark_train_tpm = rna_hallmark_symbol_tpm.loc[
    train_df["patient_id"]
].copy()

hallmark_val_tpm = rna_hallmark_symbol_tpm.loc[
    val_df["patient_id"]
].copy()

# 1. log2(TPM + 1) - deterministic
hallmark_train_log = np.log2(hallmark_train_tpm + 1)
hallmark_val_log = np.log2(hallmark_val_tpm + 1)

# 2. Train에서만 gene별 평균/표준편차 계산
gene_mean = hallmark_train_log.mean(axis=0)
gene_std = hallmark_train_log.std(axis=0, ddof=0)

# Train에서 분산이 0인 gene 제거
valid_genes = gene_std[gene_std > 0].index

hallmark_train_log = hallmark_train_log[valid_genes]
hallmark_val_log = hallmark_val_log[valid_genes]

gene_mean = gene_mean[valid_genes]
gene_std = gene_std[valid_genes]

# 3. Train 기준 gene z-score
hallmark_train_z = (
    hallmark_train_log - gene_mean
) / gene_std

hallmark_val_z = (
    hallmark_val_log - gene_mean
) / gene_std

# 4. Hallmark별 평균 z-score 계산
train_pathway_scores = pd.DataFrame(
    index=hallmark_train_z.index
)

val_pathway_scores = pd.DataFrame(
    index=hallmark_val_z.index
)

pathway_gene_counts = {}

for pathway, genes in hallmark_sets.items():

    matched = [
        g for g in genes
        if g in valid_genes
    ]

    pathway_gene_counts[pathway] = len(matched)

    train_pathway_scores[pathway] = (
        hallmark_train_z[matched].mean(axis=1)
    )

    val_pathway_scores[pathway] = (
        hallmark_val_z[matched].mean(axis=1)
    )

print("Train pathway scores:", train_pathway_scores.shape)
print("Validation pathway scores:", val_pathway_scores.shape)

print(
    "Pathway당 사용 gene 수 최소/최대:",
    min(pathway_gene_counts.values()),
    "/",
    max(pathway_gene_counts.values())
)

print(
    "NaN Train:",
    train_pathway_scores.isna().sum().sum()
)

print(
    "NaN Validation:",
    val_pathway_scores.isna().sum().sum()
)

Train pathway scores: (370, 50)
Validation pathway scores: (79, 50)
Pathway당 사용 gene 수 최소/최대: 32 / 200
NaN Train: 0
NaN Validation: 0


In [66]:
# =========================================
# Clinical + Hallmark Pathway CoxNet
# 5-fold CV
# =========================================

PATHWAY_L1_RATIOS = [0.1, 0.25, 0.5, 0.75, 0.9]

pathway_cv_results = []

for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(train_df, train_df["pfi_event"]),
    start=1
):

    fit_df = train_df.iloc[fit_idx].copy()
    cv_df_fold = train_df.iloc[cv_idx].copy()

    # ---------------------------------
    # 1. Clinical preprocessing
    # ---------------------------------
    fold_clin_preprocessor = ColumnTransformer(
        transformers=[
            ("age", "passthrough", ["age"]),
            (
                "cat",
                OneHotEncoder(
                    categories=[
                        ["MALE", "FEMALE"],
                        ["Stage I", "Stage II", "Stage III", "Stage IV", "Unknown"],
                        ["G1", "G2", "G3", "G4", "Unknown"]
                    ],
                    drop=["MALE", "Stage I", "G2"],
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                ["gender", "stage", "grade"]
            )
        ]
    )

    X_fit_clin = fold_clin_preprocessor.fit_transform(
        fit_df[clinical_features]
    )

    X_cv_clin = fold_clin_preprocessor.transform(
        cv_df_fold[clinical_features]
    )

    # ---------------------------------
    # 2. Hallmark RNA
    # ---------------------------------
    fold_fit_tpm = rna_hallmark_symbol_tpm.loc[
        fit_df["patient_id"]
    ]

    fold_cv_tpm = rna_hallmark_symbol_tpm.loc[
        cv_df_fold["patient_id"]
    ]

    # log2(TPM + 1)
    fold_fit_log = np.log2(fold_fit_tpm + 1)
    fold_cv_log = np.log2(fold_cv_tpm + 1)

    # Fold Fit에서만 gene 평균/표준편차 계산
    fold_gene_mean = fold_fit_log.mean(axis=0)
    fold_gene_std = fold_fit_log.std(axis=0, ddof=0)

    # Fold Fit에서 분산 0인 gene 제외
    fold_valid_genes = fold_gene_std[
        fold_gene_std > 0
    ].index

    fold_gene_mean = fold_gene_mean[fold_valid_genes]
    fold_gene_std = fold_gene_std[fold_valid_genes]

    fold_fit_z = (
        fold_fit_log[fold_valid_genes] - fold_gene_mean
    ) / fold_gene_std

    fold_cv_z = (
        fold_cv_log[fold_valid_genes] - fold_gene_mean
    ) / fold_gene_std

    # ---------------------------------
    # 3. 50개 Hallmark pathway score
    # ---------------------------------
    fit_pathway_scores = pd.DataFrame(
        index=fold_fit_z.index
    )

    cv_pathway_scores = pd.DataFrame(
        index=fold_cv_z.index
    )

    for pathway, genes in hallmark_sets.items():

        matched = [
            g for g in genes
            if g in fold_valid_genes
        ]

        fit_pathway_scores[pathway] = (
            fold_fit_z[matched].mean(axis=1)
        )

        cv_pathway_scores[pathway] = (
            fold_cv_z[matched].mean(axis=1)
        )

    # ---------------------------------
    # 4. Clinical + Pathway 결합
    # ---------------------------------
    X_fit_combined = np.hstack([
        X_fit_clin,
        fit_pathway_scores.to_numpy()
    ])

    X_cv_combined = np.hstack([
        X_cv_clin,
        cv_pathway_scores.to_numpy()
    ])

    # Fold Fit에서만 scaling
    fold_scaler = StandardScaler()

    X_fit_combined = fold_scaler.fit_transform(
        X_fit_combined
    )

    X_cv_combined = fold_scaler.transform(
        X_cv_combined
    )

    # survival target
    y_fit = Surv.from_arrays(
        event=fit_df["pfi_event"].astype(bool).to_numpy(),
        time=fit_df["pfi_time"].astype(float).to_numpy()
    )

    y_cv = Surv.from_arrays(
        event=cv_df_fold["pfi_event"].astype(bool).to_numpy(),
        time=cv_df_fold["pfi_time"].astype(float).to_numpy()
    )

    # ---------------------------------
    # 5. CoxNet tuning
    # ---------------------------------
    for l1_ratio in PATHWAY_L1_RATIOS:

        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            n_alphas=50,
            alpha_min_ratio=0.01,
            max_iter=100000
        )

        model.fit(
            X_fit_combined,
            y_fit
        )

        alpha_max = model.alphas_[0]

        for alpha_idx, alpha in enumerate(
            model.alphas_
        ):

            risk = model.predict(
                X_cv_combined,
                alpha=alpha
            )

            c_index = concordance_index_censored(
                y_cv["event"],
                y_cv["time"],
                risk
            )[0]

            pathway_cv_results.append({
                "fold": fold,
                "l1_ratio": l1_ratio,
                "alpha_idx": alpha_idx,
                "alpha": alpha,
                "alpha_ratio": alpha / alpha_max,
                "c_index": c_index,
                "n_nonzero": np.count_nonzero(
                    model.coef_[:, alpha_idx]
                )
            })

    print(f"Fold {fold} 완료")


pathway_cv_results = pd.DataFrame(
    pathway_cv_results
)

print("\n총 결과 행:", len(pathway_cv_results))
print(pathway_cv_results.head())

Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료

총 결과 행: 1250
   fold  l1_ratio  alpha_idx     alpha  alpha_ratio   c_index  n_nonzero
0     1       0.1          0  3.082137     1.000000  0.500000          0
1     1       0.1          1  2.805664     0.910298  0.671663          1
2     1       0.1          2  2.553991     0.828643  0.671663          1
3     1       0.1          3  2.324893     0.754312  0.671663          1
4     1       0.1          4  2.116346     0.686649  0.671663          1


In [67]:
pathway_cv_summary = (
    pathway_cv_results
    .groupby(
        ["l1_ratio", "alpha_idx"],
        as_index=False
    )
    .agg(
        alpha_ratio=("alpha_ratio", "mean"),
        mean_c_index=("c_index", "mean"),
        std_c_index=("c_index", "std"),
        mean_nonzero=("n_nonzero", "mean")
    )
)

pathway_best = (
    pathway_cv_summary
    .sort_values(
        ["mean_c_index", "std_c_index"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print(pathway_best.head(20))

    l1_ratio  alpha_idx  alpha_ratio  mean_c_index  std_c_index  mean_nonzero
0       0.75         31     0.054287      0.817289     0.059403          15.2
1       0.75         30     0.059636      0.816817     0.060901          14.6
2       0.90         30     0.059636      0.816261     0.062149          14.4
3       0.50         33     0.044984      0.816253     0.059913          19.6
4       0.90         31     0.054287      0.815866     0.061574          14.6
5       0.75         32     0.049417      0.815844     0.061009          16.2
6       0.50         34     0.040949      0.815823     0.060809          19.6
7       0.75         29     0.065513      0.815709     0.063355          14.0
8       0.90         29     0.065513      0.815676     0.062625          13.4
9       0.50         35     0.037276      0.815468     0.060598          20.4
10      0.75         33     0.044984      0.815242     0.060195          16.6
11      0.50         32     0.049417      0.815149     0.061067 

In [68]:
best_pathway_fold_scores = pathway_cv_results[
    (pathway_cv_results["l1_ratio"] == 0.75) &
    (pathway_cv_results["alpha_idx"] == 31)
][
    ["fold", "alpha", "alpha_ratio", "c_index", "n_nonzero"]
]

print(best_pathway_fold_scores)

print()
print(
    "Mean C-index:",
    best_pathway_fold_scores["c_index"].mean()
)

print(
    "Std C-index:",
    best_pathway_fold_scores["c_index"].std()
)


      fold     alpha  alpha_ratio   c_index  n_nonzero
181      1  0.022309     0.054287  0.768859         12
431      2  0.019895     0.054287  0.847534         18
681      3  0.022032     0.054287  0.741614         16
931      4  0.020397     0.054287  0.845207         16
1181     5  0.021327     0.054287  0.883234         14

Mean C-index: 0.8172892861700012
Std C-index: 0.05940336549496786


In [69]:
# =========================================
# Clinical + Hallmark CV final QC
# =========================================

assert len(pathway_cv_results) == 1250
assert pathway_cv_results["fold"].nunique() == 5

assert pathway_cv_results[
    ["alpha", "alpha_ratio", "c_index", "n_nonzero"]
].isna().sum().sum() == 0

assert np.isfinite(
    pathway_cv_results[
        ["alpha", "alpha_ratio", "c_index", "n_nonzero"]
    ].to_numpy()
).all()

assert train_pathway_scores.shape == (370, 50)
assert val_pathway_scores.shape == (79, 50)

assert train_pathway_scores.isna().sum().sum() == 0
assert val_pathway_scores.isna().sum().sum() == 0

print("Clinical + Hallmark CV QC: PASSED")
print("CV rows:", len(pathway_cv_results))
print("Train pathway:", train_pathway_scores.shape)
print("Validation pathway:", val_pathway_scores.shape)

Clinical + Hallmark CV QC: PASSED
CV rows: 1250
Train pathway: (370, 50)
Validation pathway: (79, 50)


In [70]:
# =========================================
# Final Clinical + Hallmark CoxNet
# Full Train fit -> Validation evaluation
# =========================================

BEST_PATHWAY_L1 = 0.75
BEST_PATHWAY_ALPHA_IDX = 31

# -----------------------------------------
# 0. 환자 순서 / pathway matrix QC
# -----------------------------------------
assert train_pathway_scores.shape == (370, 50)
assert val_pathway_scores.shape == (79, 50)

assert list(train_pathway_scores.index) == list(train_df["patient_id"])
assert list(val_pathway_scores.index) == list(val_df["patient_id"])

# -----------------------------------------
# 1. Clinical preprocessing
# -----------------------------------------
clinical_features = ["age", "gender", "stage", "grade"]

final_clin_preprocessor = ColumnTransformer(
    transformers=[
        ("age", "passthrough", ["age"]),
        (
            "cat",
            OneHotEncoder(
                categories=[
                    ["MALE", "FEMALE"],
                    ["Stage I", "Stage II", "Stage III", "Stage IV", "Unknown"],
                    ["G1", "G2", "G3", "G4", "Unknown"]
                ],
                drop=["MALE", "Stage I", "G2"],
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["gender", "stage", "grade"]
        )
    ]
)

X_train_clin = final_clin_preprocessor.fit_transform(
    train_df[clinical_features]
)

X_val_clin = final_clin_preprocessor.transform(
    val_df[clinical_features]
)

clinical_feature_names = (
    final_clin_preprocessor.get_feature_names_out()
)

print("Clinical features:", len(clinical_feature_names))

# -----------------------------------------
# 2. Clinical + Hallmark 50개 결합
# -----------------------------------------
X_train_pathway = train_pathway_scores.to_numpy()
X_val_pathway = val_pathway_scores.to_numpy()

X_train_final = np.hstack([
    X_train_clin,
    X_train_pathway
])

X_val_final = np.hstack([
    X_val_clin,
    X_val_pathway
])

feature_names = (
    list(clinical_feature_names)
    + list(train_pathway_scores.columns)
)

assert X_train_final.shape == (370, 60)
assert X_val_final.shape == (79, 60)
assert len(feature_names) == 60

print("Combined Train:", X_train_final.shape)
print("Combined Validation:", X_val_final.shape)

# -----------------------------------------
# 3. Train에서만 scaling
# -----------------------------------------
final_pathway_scaler = StandardScaler()

X_train_final_scaled = final_pathway_scaler.fit_transform(
    X_train_final
)

X_val_final_scaled = final_pathway_scaler.transform(
    X_val_final
)

# -----------------------------------------
# 4. Survival target
# -----------------------------------------
y_train_final = Surv.from_arrays(
    event=train_df["pfi_event"].astype(bool).to_numpy(),
    time=train_df["pfi_time"].astype(float).to_numpy()
)

y_val_final = Surv.from_arrays(
    event=val_df["pfi_event"].astype(bool).to_numpy(),
    time=val_df["pfi_time"].astype(float).to_numpy()
)

# -----------------------------------------
# 5. CV에서 고른 설정으로 Full Train fit
# -----------------------------------------
final_pathway_model = CoxnetSurvivalAnalysis(
    l1_ratio=BEST_PATHWAY_L1,
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

final_pathway_model.fit(
    X_train_final_scaled,
    y_train_final
)

assert len(final_pathway_model.alphas_) > BEST_PATHWAY_ALPHA_IDX

final_pathway_alpha = final_pathway_model.alphas_[
    BEST_PATHWAY_ALPHA_IDX
]

# -----------------------------------------
# 6. Validation 평가
# -----------------------------------------
val_pathway_risk = final_pathway_model.predict(
    X_val_final_scaled,
    alpha=final_pathway_alpha
)

val_pathway_cindex = concordance_index_censored(
    y_val_final["event"],
    y_val_final["time"],
    val_pathway_risk
)[0]

# -----------------------------------------
# 7. coefficient / non-zero 확인
# -----------------------------------------
final_pathway_coef = final_pathway_model.coef_[
    :, BEST_PATHWAY_ALPHA_IDX
]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": final_pathway_coef
})

coef_df["feature_type"] = (
    ["clinical"] * len(clinical_feature_names)
    + ["hallmark"] * train_pathway_scores.shape[1]
)

nonzero_df = coef_df[
    coef_df["coefficient"] != 0
].copy()

nonzero_df["abs_coefficient"] = (
    nonzero_df["coefficient"].abs()
)

nonzero_df = nonzero_df.sort_values(
    "abs_coefficient",
    ascending=False
)

# -----------------------------------------
# 8. 결과 저장
# -----------------------------------------
coef_df.to_csv(
    data_dir / "processed" / "clinical_pathway_coxnet_coefficients.csv",
    index=False
)

nonzero_df.to_csv(
    data_dir / "processed" / "clinical_pathway_coxnet_nonzero_features.csv",
    index=False
)

# -----------------------------------------
# 9. 최종 출력
# -----------------------------------------
print("\n=== Final Clinical + Hallmark ===")
print("l1_ratio:", BEST_PATHWAY_L1)
print("alpha_idx:", BEST_PATHWAY_ALPHA_IDX)
print("alpha:", final_pathway_alpha)
print("Non-zero features:", len(nonzero_df))
print("Validation Harrell C-index:", val_pathway_cindex)

print("\n=== Non-zero features ===")
print(
    nonzero_df[
        ["feature_type", "feature", "coefficient"]
    ].to_string(index=False)
)

Clinical features: 10
Combined Train: (370, 60)
Combined Validation: (79, 60)

=== Final Clinical + Hallmark ===
l1_ratio: 0.75
alpha_idx: 31
alpha: 0.021211534321841412
Non-zero features: 14
Validation Harrell C-index: 0.7733105218135158

=== Non-zero features ===
feature_type                        feature  coefficient
    clinical            cat__stage_Stage IV     0.864177
    clinical           cat__stage_Stage III     0.455075
    clinical                  cat__grade_G4     0.358504
    hallmark          HALLMARK_ANGIOGENESIS     0.293637
    hallmark HALLMARK_FATTY_ACID_METABOLISM    -0.208375
    hallmark        HALLMARK_MYC_TARGETS_V2     0.187937
    hallmark   HALLMARK_ALLOGRAFT_REJECTION    -0.185325
    clinical            cat__stage_Stage II     0.155051
    clinical                  cat__grade_G3     0.152390
    hallmark HALLMARK_INFLAMMATORY_RESPONSE    -0.119407
    clinical             cat__gender_FEMALE    -0.087482
    hallmark               HALLMARK_HYPOXIA     0.

In [71]:
# =========================================
# Validation model comparison 저장
# =========================================

validation_summary = pd.DataFrame({
    "model": [
        "RNA-only",
        "Clinical-only",
        "Clinical+RNA",
        "Clinical+Hallmark"
    ],
    "validation_harrell_c": [
        0.7262617621899059,
        0.7840034217279727,
        0.8135158254918734,
        0.7733105218135158
    ]
})

validation_summary.to_csv(
    data_dir / "processed" / "validation_model_comparison.csv",
    index=False
)

print(validation_summary.to_string(index=False))

print(
    "\nSaved:",
    data_dir / "processed" / "validation_model_comparison.csv"
)

            model  validation_harrell_c
         RNA-only              0.726262
    Clinical-only              0.784003
     Clinical+RNA              0.813516
Clinical+Hallmark              0.773311

Saved: ..\data\processed\validation_model_comparison.csv


In [72]:
# =========================================
# Hallmark integrity audit
# 현재 pathway score를 처음부터 재계산해 동일한지 검증
# =========================================

# 1. 기본 구조
assert len(hallmark_sets) == 50
assert rna_hallmark_symbol_tpm.shape == (529, 4360)
assert rna_hallmark_symbol_tpm.columns.duplicated().sum() == 0

assert len(train_df) == 370
assert len(val_df) == 79

# 2. Full Train 기준 pathway score 처음부터 재계산
audit_train_tpm = rna_hallmark_symbol_tpm.loc[
    train_df["patient_id"]
].copy()

audit_val_tpm = rna_hallmark_symbol_tpm.loc[
    val_df["patient_id"]
].copy()

audit_train_log = np.log2(audit_train_tpm + 1)
audit_val_log = np.log2(audit_val_tpm + 1)

audit_mean = audit_train_log.mean(axis=0)
audit_std = audit_train_log.std(axis=0, ddof=0)

audit_valid_genes = audit_std[audit_std > 0].index

audit_train_z = (
    audit_train_log[audit_valid_genes]
    - audit_mean[audit_valid_genes]
) / audit_std[audit_valid_genes]

audit_val_z = (
    audit_val_log[audit_valid_genes]
    - audit_mean[audit_valid_genes]
) / audit_std[audit_valid_genes]

audit_train_scores = pd.DataFrame(
    index=audit_train_z.index
)

audit_val_scores = pd.DataFrame(
    index=audit_val_z.index
)

for pathway, genes in hallmark_sets.items():

    matched = [
        g for g in genes
        if g in audit_valid_genes
    ]

    audit_train_scores[pathway] = (
        audit_train_z[matched].mean(axis=1)
    )

    audit_val_scores[pathway] = (
        audit_val_z[matched].mean(axis=1)
    )

# 3. 기존 score와 비교
assert audit_train_scores.shape == (370, 50)
assert audit_val_scores.shape == (79, 50)

assert list(audit_train_scores.columns) == list(train_pathway_scores.columns)
assert list(audit_val_scores.columns) == list(val_pathway_scores.columns)

train_diff = np.max(
    np.abs(
        audit_train_scores.to_numpy()
        - train_pathway_scores.to_numpy()
    )
)

val_diff = np.max(
    np.abs(
        audit_val_scores.to_numpy()
        - val_pathway_scores.to_numpy()
    )
)

assert np.allclose(
    audit_train_scores.to_numpy(),
    train_pathway_scores.to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

assert np.allclose(
    audit_val_scores.to_numpy(),
    val_pathway_scores.to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

assert np.isfinite(audit_train_scores.to_numpy()).all()
assert np.isfinite(audit_val_scores.to_numpy()).all()

print("Hallmark integrity audit: PASSED")
print("Train:", audit_train_scores.shape)
print("Validation:", audit_val_scores.shape)
print("Max Train difference:", train_diff)
print("Max Validation difference:", val_diff)

Hallmark integrity audit: PASSED
Train: (370, 50)
Validation: (79, 50)
Max Train difference: 0.0
Max Validation difference: 0.0


In [73]:
# ============================================================
# Clinical + Hallmark 5-fold CV leakage / reproducibility audit
# 기존 결과를 수정하지 않고 재현 검증만 수행
# ============================================================

AUDIT_L1 = 0.75
AUDIT_ALPHA_IDX = 31

audit_results = []

audit_clinical_features = [
    "age", "gender", "stage", "grade"
]

for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(train_df, train_df["pfi_event"]),
    start=1
):

    # --------------------------------------------------------
    # 1. Fold 환자 분리
    # --------------------------------------------------------
    fit_df = train_df.iloc[fit_idx].copy()
    cv_df_fold = train_df.iloc[cv_idx].copy()

    fit_ids = set(fit_df["patient_id"])
    cv_ids = set(cv_df_fold["patient_id"])

    overlap = fit_ids & cv_ids

    assert len(overlap) == 0, \
        f"Fold {fold}: Fit/CV patient overlap 발생"

    assert len(fit_df) == 296
    assert len(cv_df_fold) == 74

    # --------------------------------------------------------
    # 2. Clinical preprocessing
    # 반드시 Fold-Fit에서만 fit
    # --------------------------------------------------------
    audit_clin_preprocessor = ColumnTransformer(
        transformers=[
            (
                "age",
                "passthrough",
                ["age"]
            ),
            (
                "cat",
                OneHotEncoder(
                    categories=[
                        ["MALE", "FEMALE"],
                        [
                            "Stage I",
                            "Stage II",
                            "Stage III",
                            "Stage IV",
                            "Unknown"
                        ],
                        [
                            "G1",
                            "G2",
                            "G3",
                            "G4",
                            "Unknown"
                        ]
                    ],
                    drop=[
                        "MALE",
                        "Stage I",
                        "G2"
                    ],
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                ["gender", "stage", "grade"]
            )
        ]
    )

    X_fit_clin = (
        audit_clin_preprocessor
        .fit_transform(
            fit_df[audit_clinical_features]
        )
    )

    X_cv_clin = (
        audit_clin_preprocessor
        .transform(
            cv_df_fold[audit_clinical_features]
        )
    )

    assert X_fit_clin.shape == (296, 10)
    assert X_cv_clin.shape == (74, 10)

    # --------------------------------------------------------
    # 3. Hallmark RNA
    # raw TPM에서 다시 시작
    # --------------------------------------------------------
    fit_tpm = (
        rna_hallmark_symbol_tpm
        .loc[fit_df["patient_id"]]
        .copy()
    )

    cv_tpm = (
        rna_hallmark_symbol_tpm
        .loc[cv_df_fold["patient_id"]]
        .copy()
    )

    assert fit_tpm.shape[0] == 296
    assert cv_tpm.shape[0] == 74

    # deterministic transformation
    fit_log = np.log2(fit_tpm + 1)
    cv_log = np.log2(cv_tpm + 1)

    # --------------------------------------------------------
    # 4. gene mean/std
    # 오직 Fold-Fit에서만 계산
    # --------------------------------------------------------
    fit_gene_mean = fit_log.mean(axis=0)
    fit_gene_std = fit_log.std(
        axis=0,
        ddof=0
    )

    valid_genes = (
        fit_gene_std[
            fit_gene_std > 0
        ]
        .index
    )

    # CV 데이터는 여기서 mean/std 계산에 전혀 사용하지 않음
    fit_z = (
        fit_log[valid_genes]
        - fit_gene_mean[valid_genes]
    ) / fit_gene_std[valid_genes]

    cv_z = (
        cv_log[valid_genes]
        - fit_gene_mean[valid_genes]
    ) / fit_gene_std[valid_genes]

    assert np.isfinite(
        fit_z.to_numpy()
    ).all()

    assert np.isfinite(
        cv_z.to_numpy()
    ).all()

    # Fit에서 z-score했으므로 gene 평균은 거의 0이어야 함
    max_abs_gene_mean = float(
        fit_z.mean(axis=0).abs().max()
    )

    assert max_abs_gene_mean < 1e-10

    # --------------------------------------------------------
    # 5. 50개 Hallmark pathway score 재계산
    # --------------------------------------------------------
    fit_pathway = pd.DataFrame(
        index=fit_z.index
    )

    cv_pathway = pd.DataFrame(
        index=cv_z.index
    )

    for pathway, genes in hallmark_sets.items():

        matched = [
            gene
            for gene in genes
            if gene in valid_genes
        ]

        assert len(matched) > 0, \
            f"{fold=} {pathway}: matched gene 0개"

        fit_pathway[pathway] = (
            fit_z[matched]
            .mean(axis=1)
        )

        cv_pathway[pathway] = (
            cv_z[matched]
            .mean(axis=1)
        )

    assert fit_pathway.shape == (296, 50)
    assert cv_pathway.shape == (74, 50)

    assert np.isfinite(
        fit_pathway.to_numpy()
    ).all()

    assert np.isfinite(
        cv_pathway.to_numpy()
    ).all()

    # --------------------------------------------------------
    # 6. Clinical + Hallmark 결합
    # --------------------------------------------------------
    X_fit = np.hstack([
        X_fit_clin,
        fit_pathway.to_numpy()
    ])

    X_cv = np.hstack([
        X_cv_clin,
        cv_pathway.to_numpy()
    ])

    assert X_fit.shape == (296, 60)
    assert X_cv.shape == (74, 60)

    # --------------------------------------------------------
    # 7. Scaling
    # 반드시 Fold-Fit에서만 fit
    # --------------------------------------------------------
    audit_scaler = StandardScaler()

    X_fit_scaled = (
        audit_scaler
        .fit_transform(X_fit)
    )

    X_cv_scaled = (
        audit_scaler
        .transform(X_cv)
    )

    # --------------------------------------------------------
    # 8. survival target
    # --------------------------------------------------------
    y_fit = Surv.from_arrays(
        event=fit_df[
            "pfi_event"
        ].astype(bool).to_numpy(),

        time=fit_df[
            "pfi_time"
        ].astype(float).to_numpy()
    )

    y_cv = Surv.from_arrays(
        event=cv_df_fold[
            "pfi_event"
        ].astype(bool).to_numpy(),

        time=cv_df_fold[
            "pfi_time"
        ].astype(float).to_numpy()
    )

    # --------------------------------------------------------
    # 9. 선택된 설정만 다시 학습
    # --------------------------------------------------------
    audit_model = CoxnetSurvivalAnalysis(
        l1_ratio=AUDIT_L1,
        n_alphas=50,
        alpha_min_ratio=0.01,
        max_iter=100000
    )

    audit_model.fit(
        X_fit_scaled,
        y_fit
    )

    audit_alpha = (
        audit_model.alphas_[
            AUDIT_ALPHA_IDX
        ]
    )

    audit_risk = audit_model.predict(
        X_cv_scaled,
        alpha=audit_alpha
    )

    audit_cindex = (
        concordance_index_censored(
            y_cv["event"],
            y_cv["time"],
            audit_risk
        )[0]
    )

    audit_nonzero = int(
        np.count_nonzero(
            audit_model.coef_[
                :, AUDIT_ALPHA_IDX
            ]
        )
    )

    # --------------------------------------------------------
    # 10. 기존 CV 결과와 비교
    # --------------------------------------------------------
    original_row = (
        pathway_cv_results[
            (pathway_cv_results["fold"] == fold)
            & (
                pathway_cv_results["l1_ratio"]
                == AUDIT_L1
            )
            & (
                pathway_cv_results["alpha_idx"]
                == AUDIT_ALPHA_IDX
            )
        ]
        .iloc[0]
    )

    original_alpha = float(
        original_row["alpha"]
    )

    original_cindex = float(
        original_row["c_index"]
    )

    original_nonzero = int(
        original_row["n_nonzero"]
    )

    alpha_diff = abs(
        audit_alpha
        - original_alpha
    )

    cindex_diff = abs(
        audit_cindex
        - original_cindex
    )

    # 기존 CV가 동일한 Fit-only pipeline에서 나온 것인지 확인
    assert np.isclose(
        audit_alpha,
        original_alpha,
        rtol=1e-10,
        atol=1e-12
    ), f"Fold {fold}: alpha 불일치"

    assert np.isclose(
        audit_cindex,
        original_cindex,
        rtol=1e-10,
        atol=1e-12
    ), f"Fold {fold}: C-index 불일치"

    assert (
        audit_nonzero
        == original_nonzero
    ), f"Fold {fold}: non-zero 수 불일치"

    audit_results.append({
        "fold": fold,
        "fit_n": len(fit_df),
        "cv_n": len(cv_df_fold),
        "patient_overlap": len(overlap),
        "valid_genes": len(valid_genes),
        "max_abs_fit_gene_mean": max_abs_gene_mean,
        "original_cindex": original_cindex,
        "recomputed_cindex": audit_cindex,
        "cindex_diff": cindex_diff,
        "original_nonzero": original_nonzero,
        "recomputed_nonzero": audit_nonzero,
        "alpha_diff": alpha_diff
    })


# ------------------------------------------------------------
# 11. 최종 audit 결과
# ------------------------------------------------------------
hallmark_cv_audit = pd.DataFrame(
    audit_results
)

print(
    hallmark_cv_audit.to_string(
        index=False
    )
)

print()
print(
    "Max C-index difference:",
    hallmark_cv_audit[
        "cindex_diff"
    ].max()
)

print(
    "Max alpha difference:",
    hallmark_cv_audit[
        "alpha_diff"
    ].max()
)

print(
    "Total Fit/CV patient overlap:",
    hallmark_cv_audit[
        "patient_overlap"
    ].sum()
)

print(
    "\nClinical + Hallmark 5-fold leakage audit: PASSED"
)

AssertionError: 

In [74]:
# =========================================
# 실패한 z-score QC 원인 확인
# =========================================

gene_mean_after_z = fit_z.mean(axis=0)
abs_gene_mean_after_z = gene_mean_after_z.abs().sort_values(
    ascending=False
)

print("fit_log dtype:")
print(fit_log.dtypes.value_counts())

print("\nfit_z dtype:")
print(fit_z.dtypes.value_counts())

print(
    "\nMax absolute gene mean after z-score:",
    abs_gene_mean_after_z.iloc[0]
)

print("\n평균 0에서 가장 많이 벗어난 10개 gene:")
print(abs_gene_mean_after_z.head(10))

print("\n해당 gene들의 Fold-Fit 표준편차:")
print(
    fit_gene_std.loc[
        abs_gene_mean_after_z.head(10).index
    ]
)

fit_log dtype:
float32    4360
Name: count, dtype: int64

fit_z dtype:
float32    4356
Name: count, dtype: int64

Max absolute gene mean after z-score: 1.6334894e-06

평균 0에서 가장 많이 벗어난 10개 gene:
ARF1       0.000002
CFL1       0.000002
APH1A      0.000002
RPS2       0.000002
PPIA       0.000002
IDH2       0.000001
HNRNPC     0.000001
GLUD1      0.000001
HNRNPA1    0.000001
PSMD8      0.000001
dtype: float32

해당 gene들의 Fold-Fit 표준편차:
ARF1       0.508675
CFL1       0.518729
APH1A      0.582908
RPS2       0.667526
PPIA       0.379147
IDH2       0.710344
HNRNPC     0.541617
GLUD1      0.660197
HNRNPA1    0.625747
PSMD8      0.463666
dtype: float32


In [75]:
# ============================================================
# Clinical + Hallmark 5-fold CV
# Leakage / Reproducibility Final Audit
# 기존 결과를 수정하거나 저장하지 않고 검증만 수행
# ============================================================

AUDIT_L1 = 0.75
AUDIT_ALPHA_IDX = 31
FLOAT32_MEAN_TOL = 1e-5

audit_results = []

audit_clinical_features = [
    "age",
    "gender",
    "stage",
    "grade"
]


for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(
        train_df,
        train_df["pfi_event"]
    ),
    start=1
):

    # ========================================================
    # 1. Fold 환자 분리
    # ========================================================
    fit_df_audit = train_df.iloc[
        fit_idx
    ].copy()

    cv_df_audit = train_df.iloc[
        cv_idx
    ].copy()

    fit_ids = set(
        fit_df_audit["patient_id"]
    )

    cv_ids = set(
        cv_df_audit["patient_id"]
    )

    overlap = fit_ids & cv_ids

    # Stratified 5-fold:
    # 370명 -> 296 Fit + 74 CV
    assert len(fit_df_audit) == 296, (
        f"Fold {fold}: Fit size 오류"
    )

    assert len(cv_df_audit) == 74, (
        f"Fold {fold}: CV size 오류"
    )

    assert len(overlap) == 0, (
        f"Fold {fold}: Fit/CV 환자 overlap 발생"
    )


    # ========================================================
    # 2. Clinical preprocessing
    # Fold-Fit에서만 fit
    # ========================================================
    audit_clin_preprocessor = ColumnTransformer(
        transformers=[
            (
                "age",
                "passthrough",
                ["age"]
            ),
            (
                "cat",
                OneHotEncoder(
                    categories=[
                        [
                            "MALE",
                            "FEMALE"
                        ],
                        [
                            "Stage I",
                            "Stage II",
                            "Stage III",
                            "Stage IV",
                            "Unknown"
                        ],
                        [
                            "G1",
                            "G2",
                            "G3",
                            "G4",
                            "Unknown"
                        ]
                    ],
                    drop=[
                        "MALE",
                        "Stage I",
                        "G2"
                    ],
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                [
                    "gender",
                    "stage",
                    "grade"
                ]
            )
        ]
    )

    X_fit_clin = (
        audit_clin_preprocessor
        .fit_transform(
            fit_df_audit[
                audit_clinical_features
            ]
        )
    )

    X_cv_clin = (
        audit_clin_preprocessor
        .transform(
            cv_df_audit[
                audit_clinical_features
            ]
        )
    )

    assert X_fit_clin.shape == (296, 10)
    assert X_cv_clin.shape == (74, 10)


    # ========================================================
    # 3. Hallmark RNA
    # raw symbol-level TPM에서 다시 시작
    # ========================================================
    fit_tpm = (
        rna_hallmark_symbol_tpm
        .loc[
            fit_df_audit["patient_id"]
        ]
        .copy()
    )

    cv_tpm = (
        rna_hallmark_symbol_tpm
        .loc[
            cv_df_audit["patient_id"]
        ]
        .copy()
    )

    assert fit_tpm.shape == (296, 4360)
    assert cv_tpm.shape == (74, 4360)

    assert list(
        fit_tpm.index
    ) == list(
        fit_df_audit["patient_id"]
    )

    assert list(
        cv_tpm.index
    ) == list(
        cv_df_audit["patient_id"]
    )


    # ========================================================
    # 4. log2(TPM + 1)
    # deterministic transformation
    # ========================================================
    fit_log = np.log2(
        fit_tpm + 1
    )

    cv_log = np.log2(
        cv_tpm + 1
    )


    # ========================================================
    # 5. Gene mean / std
    # 반드시 Fold-Fit에서만 계산
    # ========================================================
    fit_gene_mean = (
        fit_log.mean(axis=0)
    )

    fit_gene_std = (
        fit_log.std(
            axis=0,
            ddof=0
        )
    )

    # Fold-Fit에서 variance=0인 gene 제외
    valid_genes = (
        fit_gene_std[
            fit_gene_std > 0
        ]
        .index
    )

    assert len(valid_genes) > 0


    # ========================================================
    # 6. Fold-Fit 기준 gene z-score
    #
    # CV에는 Fit에서 계산한 mean/std만 적용
    # ========================================================
    fit_z = (
        fit_log[valid_genes]
        - fit_gene_mean[valid_genes]
    ) / fit_gene_std[valid_genes]

    cv_z = (
        cv_log[valid_genes]
        - fit_gene_mean[valid_genes]
    ) / fit_gene_std[valid_genes]


    # NaN / Inf 검사
    assert np.isfinite(
        fit_z.to_numpy()
    ).all(), (
        f"Fold {fold}: Fit z-score에 NaN/Inf"
    )

    assert np.isfinite(
        cv_z.to_numpy()
    ).all(), (
        f"Fold {fold}: CV z-score에 NaN/Inf"
    )


    # float32 numerical precision QC
    max_abs_gene_mean = float(
        fit_z
        .mean(axis=0)
        .abs()
        .max()
    )

    assert (
        max_abs_gene_mean
        < FLOAT32_MEAN_TOL
    ), (
        f"Fold {fold}: "
        f"z-score mean residual too large: "
        f"{max_abs_gene_mean}"
    )


    # ========================================================
    # 7. 50 Hallmark pathway score 재계산
    # mean standardized-expression score
    # ========================================================
    fit_pathway = pd.DataFrame(
        index=fit_z.index
    )

    cv_pathway = pd.DataFrame(
        index=cv_z.index
    )

    pathway_gene_counts = {}

    for pathway, genes in hallmark_sets.items():

        matched = [
            gene
            for gene in genes
            if gene in valid_genes
        ]

        pathway_gene_counts[
            pathway
        ] = len(matched)

        assert len(matched) > 0, (
            f"Fold {fold} / {pathway}: "
            "matched gene가 0개"
        )

        fit_pathway[
            pathway
        ] = (
            fit_z[
                matched
            ]
            .mean(axis=1)
        )

        cv_pathway[
            pathway
        ] = (
            cv_z[
                matched
            ]
            .mean(axis=1)
        )


    assert fit_pathway.shape == (
        296,
        50
    )

    assert cv_pathway.shape == (
        74,
        50
    )

    assert np.isfinite(
        fit_pathway.to_numpy()
    ).all()

    assert np.isfinite(
        cv_pathway.to_numpy()
    ).all()


    # ========================================================
    # 8. Clinical + Hallmark 결합
    # 10 clinical + 50 pathways = 60
    # ========================================================
    X_fit = np.hstack([
        X_fit_clin,
        fit_pathway.to_numpy()
    ])

    X_cv = np.hstack([
        X_cv_clin,
        cv_pathway.to_numpy()
    ])

    assert X_fit.shape == (
        296,
        60
    )

    assert X_cv.shape == (
        74,
        60
    )


    # ========================================================
    # 9. StandardScaler
    # 반드시 Fold-Fit에서만 fit
    # ========================================================
    audit_scaler = StandardScaler()

    X_fit_scaled = (
        audit_scaler
        .fit_transform(
            X_fit
        )
    )

    X_cv_scaled = (
        audit_scaler
        .transform(
            X_cv
        )
    )

    assert np.isfinite(
        X_fit_scaled
    ).all()

    assert np.isfinite(
        X_cv_scaled
    ).all()


    # ========================================================
    # 10. Survival target
    # ========================================================
    y_fit = Surv.from_arrays(
        event=(
            fit_df_audit[
                "pfi_event"
            ]
            .astype(bool)
            .to_numpy()
        ),
        time=(
            fit_df_audit[
                "pfi_time"
            ]
            .astype(float)
            .to_numpy()
        )
    )

    y_cv = Surv.from_arrays(
        event=(
            cv_df_audit[
                "pfi_event"
            ]
            .astype(bool)
            .to_numpy()
        ),
        time=(
            cv_df_audit[
                "pfi_time"
            ]
            .astype(float)
            .to_numpy()
        )
    )


    # ========================================================
    # 11. 기존 선택 설정으로 CoxNet 재학습
    # ========================================================
    audit_model = CoxnetSurvivalAnalysis(
        l1_ratio=AUDIT_L1,
        n_alphas=50,
        alpha_min_ratio=0.01,
        max_iter=100000
    )

    audit_model.fit(
        X_fit_scaled,
        y_fit
    )

    assert (
        len(audit_model.alphas_)
        > AUDIT_ALPHA_IDX
    )

    audit_alpha = (
        audit_model.alphas_[
            AUDIT_ALPHA_IDX
        ]
    )


    # ========================================================
    # 12. Fold-CV risk prediction
    # ========================================================
    audit_risk = (
        audit_model.predict(
            X_cv_scaled,
            alpha=audit_alpha
        )
    )

    audit_cindex = (
        concordance_index_censored(
            y_cv["event"],
            y_cv["time"],
            audit_risk
        )[0]
    )

    audit_nonzero = int(
        np.count_nonzero(
            audit_model.coef_[
                :,
                AUDIT_ALPHA_IDX
            ]
        )
    )


    # ========================================================
    # 13. 기존 pathway_cv_results와 비교
    # ========================================================
    original_rows = (
        pathway_cv_results[
            (
                pathway_cv_results[
                    "fold"
                ] == fold
            )
            &
            (
                pathway_cv_results[
                    "l1_ratio"
                ] == AUDIT_L1
            )
            &
            (
                pathway_cv_results[
                    "alpha_idx"
                ] == AUDIT_ALPHA_IDX
            )
        ]
    )

    assert len(original_rows) == 1, (
        f"Fold {fold}: "
        "기존 CV 결과 행이 정확히 1개가 아님"
    )

    original_row = (
        original_rows.iloc[0]
    )

    original_alpha = float(
        original_row["alpha"]
    )

    original_cindex = float(
        original_row["c_index"]
    )

    original_nonzero = int(
        original_row["n_nonzero"]
    )


    # ========================================================
    # 14. 재현성 비교
    # ========================================================
    alpha_diff = abs(
        float(audit_alpha)
        - original_alpha
    )

    cindex_diff = abs(
        float(audit_cindex)
        - original_cindex
    )


    assert np.isclose(
        audit_alpha,
        original_alpha,
        rtol=1e-10,
        atol=1e-12
    ), (
        f"Fold {fold}: alpha 불일치 "
        f"{audit_alpha} vs {original_alpha}"
    )

    assert np.isclose(
        audit_cindex,
        original_cindex,
        rtol=1e-10,
        atol=1e-12
    ), (
        f"Fold {fold}: C-index 불일치 "
        f"{audit_cindex} vs {original_cindex}"
    )

    assert (
        audit_nonzero
        == original_nonzero
    ), (
        f"Fold {fold}: non-zero feature 수 불일치"
    )


    # ========================================================
    # 15. Fold 결과 기록
    # ========================================================
    audit_results.append({
        "fold": fold,

        "fit_n": len(
            fit_df_audit
        ),

        "cv_n": len(
            cv_df_audit
        ),

        "patient_overlap": len(
            overlap
        ),

        "valid_genes": len(
            valid_genes
        ),

        "min_pathway_genes": min(
            pathway_gene_counts.values()
        ),

        "max_pathway_genes": max(
            pathway_gene_counts.values()
        ),

        "max_abs_fit_gene_mean": (
            max_abs_gene_mean
        ),

        "original_alpha": (
            original_alpha
        ),

        "recomputed_alpha": float(
            audit_alpha
        ),

        "alpha_diff": (
            alpha_diff
        ),

        "original_cindex": (
            original_cindex
        ),

        "recomputed_cindex": float(
            audit_cindex
        ),

        "cindex_diff": (
            cindex_diff
        ),

        "original_nonzero": (
            original_nonzero
        ),

        "recomputed_nonzero": (
            audit_nonzero
        )
    })


# ============================================================
# 16. 최종 Audit 결과
# ============================================================

hallmark_cv_audit = pd.DataFrame(
    audit_results
)

print(
    hallmark_cv_audit.to_string(
        index=False
    )
)

print()

print(
    "Max z-score mean residual:",
    hallmark_cv_audit[
        "max_abs_fit_gene_mean"
    ].max()
)

print(
    "Max C-index difference:",
    hallmark_cv_audit[
        "cindex_diff"
    ].max()
)

print(
    "Max alpha difference:",
    hallmark_cv_audit[
        "alpha_diff"
    ].max()
)

print(
    "Total Fit/CV patient overlap:",
    hallmark_cv_audit[
        "patient_overlap"
    ].sum()
)

print(
    "\nClinical + Hallmark 5-fold leakage / "
    "reproducibility audit: PASSED"
)

 fold  fit_n  cv_n  patient_overlap  valid_genes  min_pathway_genes  max_pathway_genes  max_abs_fit_gene_mean  original_alpha  recomputed_alpha  alpha_diff  original_cindex  recomputed_cindex  cindex_diff  original_nonzero  recomputed_nonzero
    1    296    74                0         4356                 32                200               0.000002        0.022309          0.022309         0.0         0.768859           0.768859          0.0                12                  12
    2    296    74                0         4356                 32                200               0.000002        0.019895          0.019895         0.0         0.847534           0.847534          0.0                18                  18
    3    296    74                0         4356                 32                200               0.000002        0.022032          0.022032         0.0         0.741614           0.741614          0.0                16                  16
    4    296    74          

In [76]:
# ============================================================
# Clinical + Hallmark 5-fold CV
# Leakage / Reproducibility Final Audit
# 기존 결과를 수정하거나 저장하지 않고 검증만 수행
# ============================================================

AUDIT_L1 = 0.75
AUDIT_ALPHA_IDX = 31
FLOAT32_MEAN_TOL = 1e-5

audit_results = []

audit_clinical_features = [
    "age",
    "gender",
    "stage",
    "grade"
]


for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(
        train_df,
        train_df["pfi_event"]
    ),
    start=1
):

    # ========================================================
    # 1. Fold 환자 분리
    # ========================================================
    fit_df_audit = train_df.iloc[
        fit_idx
    ].copy()

    cv_df_audit = train_df.iloc[
        cv_idx
    ].copy()

    fit_ids = set(
        fit_df_audit["patient_id"]
    )

    cv_ids = set(
        cv_df_audit["patient_id"]
    )

    overlap = fit_ids & cv_ids

    # Stratified 5-fold:
    # 370명 -> 296 Fit + 74 CV
    assert len(fit_df_audit) == 296, (
        f"Fold {fold}: Fit size 오류"
    )

    assert len(cv_df_audit) == 74, (
        f"Fold {fold}: CV size 오류"
    )

    assert len(overlap) == 0, (
        f"Fold {fold}: Fit/CV 환자 overlap 발생"
    )


    # ========================================================
    # 2. Clinical preprocessing
    # Fold-Fit에서만 fit
    # ========================================================
    audit_clin_preprocessor = ColumnTransformer(
        transformers=[
            (
                "age",
                "passthrough",
                ["age"]
            ),
            (
                "cat",
                OneHotEncoder(
                    categories=[
                        [
                            "MALE",
                            "FEMALE"
                        ],
                        [
                            "Stage I",
                            "Stage II",
                            "Stage III",
                            "Stage IV",
                            "Unknown"
                        ],
                        [
                            "G1",
                            "G2",
                            "G3",
                            "G4",
                            "Unknown"
                        ]
                    ],
                    drop=[
                        "MALE",
                        "Stage I",
                        "G2"
                    ],
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                [
                    "gender",
                    "stage",
                    "grade"
                ]
            )
        ]
    )

    X_fit_clin = (
        audit_clin_preprocessor
        .fit_transform(
            fit_df_audit[
                audit_clinical_features
            ]
        )
    )

    X_cv_clin = (
        audit_clin_preprocessor
        .transform(
            cv_df_audit[
                audit_clinical_features
            ]
        )
    )

    assert X_fit_clin.shape == (296, 10)
    assert X_cv_clin.shape == (74, 10)


    # ========================================================
    # 3. Hallmark RNA
    # raw symbol-level TPM에서 다시 시작
    # ========================================================
    fit_tpm = (
        rna_hallmark_symbol_tpm
        .loc[
            fit_df_audit["patient_id"]
        ]
        .copy()
    )

    cv_tpm = (
        rna_hallmark_symbol_tpm
        .loc[
            cv_df_audit["patient_id"]
        ]
        .copy()
    )

    assert fit_tpm.shape == (296, 4360)
    assert cv_tpm.shape == (74, 4360)

    assert list(
        fit_tpm.index
    ) == list(
        fit_df_audit["patient_id"]
    )

    assert list(
        cv_tpm.index
    ) == list(
        cv_df_audit["patient_id"]
    )


    # ========================================================
    # 4. log2(TPM + 1)
    # deterministic transformation
    # ========================================================
    fit_log = np.log2(
        fit_tpm + 1
    )

    cv_log = np.log2(
        cv_tpm + 1
    )


    # ========================================================
    # 5. Gene mean / std
    # 반드시 Fold-Fit에서만 계산
    # ========================================================
    fit_gene_mean = (
        fit_log.mean(axis=0)
    )

    fit_gene_std = (
        fit_log.std(
            axis=0,
            ddof=0
        )
    )

    # Fold-Fit에서 variance=0인 gene 제외
    valid_genes = (
        fit_gene_std[
            fit_gene_std > 0
        ]
        .index
    )

    assert len(valid_genes) > 0


    # ========================================================
    # 6. Fold-Fit 기준 gene z-score
    #
    # CV에는 Fit에서 계산한 mean/std만 적용
    # ========================================================
    fit_z = (
        fit_log[valid_genes]
        - fit_gene_mean[valid_genes]
    ) / fit_gene_std[valid_genes]

    cv_z = (
        cv_log[valid_genes]
        - fit_gene_mean[valid_genes]
    ) / fit_gene_std[valid_genes]


    # NaN / Inf 검사
    assert np.isfinite(
        fit_z.to_numpy()
    ).all(), (
        f"Fold {fold}: Fit z-score에 NaN/Inf"
    )

    assert np.isfinite(
        cv_z.to_numpy()
    ).all(), (
        f"Fold {fold}: CV z-score에 NaN/Inf"
    )


    # float32 numerical precision QC
    max_abs_gene_mean = float(
        fit_z
        .mean(axis=0)
        .abs()
        .max()
    )

    assert (
        max_abs_gene_mean
        < FLOAT32_MEAN_TOL
    ), (
        f"Fold {fold}: "
        f"z-score mean residual too large: "
        f"{max_abs_gene_mean}"
    )


    # ========================================================
    # 7. 50 Hallmark pathway score 재계산
    # mean standardized-expression score
    # ========================================================
    fit_pathway = pd.DataFrame(
        index=fit_z.index
    )

    cv_pathway = pd.DataFrame(
        index=cv_z.index
    )

    pathway_gene_counts = {}

    for pathway, genes in hallmark_sets.items():

        matched = [
            gene
            for gene in genes
            if gene in valid_genes
        ]

        pathway_gene_counts[
            pathway
        ] = len(matched)

        assert len(matched) > 0, (
            f"Fold {fold} / {pathway}: "
            "matched gene가 0개"
        )

        fit_pathway[
            pathway
        ] = (
            fit_z[
                matched
            ]
            .mean(axis=1)
        )

        cv_pathway[
            pathway
        ] = (
            cv_z[
                matched
            ]
            .mean(axis=1)
        )


    assert fit_pathway.shape == (
        296,
        50
    )

    assert cv_pathway.shape == (
        74,
        50
    )

    assert np.isfinite(
        fit_pathway.to_numpy()
    ).all()

    assert np.isfinite(
        cv_pathway.to_numpy()
    ).all()


    # ========================================================
    # 8. Clinical + Hallmark 결합
    # 10 clinical + 50 pathways = 60
    # ========================================================
    X_fit = np.hstack([
        X_fit_clin,
        fit_pathway.to_numpy()
    ])

    X_cv = np.hstack([
        X_cv_clin,
        cv_pathway.to_numpy()
    ])

    assert X_fit.shape == (
        296,
        60
    )

    assert X_cv.shape == (
        74,
        60
    )


    # ========================================================
    # 9. StandardScaler
    # 반드시 Fold-Fit에서만 fit
    # ========================================================
    audit_scaler = StandardScaler()

    X_fit_scaled = (
        audit_scaler
        .fit_transform(
            X_fit
        )
    )

    X_cv_scaled = (
        audit_scaler
        .transform(
            X_cv
        )
    )

    assert np.isfinite(
        X_fit_scaled
    ).all()

    assert np.isfinite(
        X_cv_scaled
    ).all()


    # ========================================================
    # 10. Survival target
    # ========================================================
    y_fit = Surv.from_arrays(
        event=(
            fit_df_audit[
                "pfi_event"
            ]
            .astype(bool)
            .to_numpy()
        ),
        time=(
            fit_df_audit[
                "pfi_time"
            ]
            .astype(float)
            .to_numpy()
        )
    )

    y_cv = Surv.from_arrays(
        event=(
            cv_df_audit[
                "pfi_event"
            ]
            .astype(bool)
            .to_numpy()
        ),
        time=(
            cv_df_audit[
                "pfi_time"
            ]
            .astype(float)
            .to_numpy()
        )
    )


    # ========================================================
    # 11. 기존 선택 설정으로 CoxNet 재학습
    # ========================================================
    audit_model = CoxnetSurvivalAnalysis(
        l1_ratio=AUDIT_L1,
        n_alphas=50,
        alpha_min_ratio=0.01,
        max_iter=100000
    )

    audit_model.fit(
        X_fit_scaled,
        y_fit
    )

    assert (
        len(audit_model.alphas_)
        > AUDIT_ALPHA_IDX
    )

    audit_alpha = (
        audit_model.alphas_[
            AUDIT_ALPHA_IDX
        ]
    )


    # ========================================================
    # 12. Fold-CV risk prediction
    # ========================================================
    audit_risk = (
        audit_model.predict(
            X_cv_scaled,
            alpha=audit_alpha
        )
    )

    audit_cindex = (
        concordance_index_censored(
            y_cv["event"],
            y_cv["time"],
            audit_risk
        )[0]
    )

    audit_nonzero = int(
        np.count_nonzero(
            audit_model.coef_[
                :,
                AUDIT_ALPHA_IDX
            ]
        )
    )


    # ========================================================
    # 13. 기존 pathway_cv_results와 비교
    # ========================================================
    original_rows = (
        pathway_cv_results[
            (
                pathway_cv_results[
                    "fold"
                ] == fold
            )
            &
            (
                pathway_cv_results[
                    "l1_ratio"
                ] == AUDIT_L1
            )
            &
            (
                pathway_cv_results[
                    "alpha_idx"
                ] == AUDIT_ALPHA_IDX
            )
        ]
    )

    assert len(original_rows) == 1, (
        f"Fold {fold}: "
        "기존 CV 결과 행이 정확히 1개가 아님"
    )

    original_row = (
        original_rows.iloc[0]
    )

    original_alpha = float(
        original_row["alpha"]
    )

    original_cindex = float(
        original_row["c_index"]
    )

    original_nonzero = int(
        original_row["n_nonzero"]
    )


    # ========================================================
    # 14. 재현성 비교
    # ========================================================
    alpha_diff = abs(
        float(audit_alpha)
        - original_alpha
    )

    cindex_diff = abs(
        float(audit_cindex)
        - original_cindex
    )


    assert np.isclose(
        audit_alpha,
        original_alpha,
        rtol=1e-10,
        atol=1e-12
    ), (
        f"Fold {fold}: alpha 불일치 "
        f"{audit_alpha} vs {original_alpha}"
    )

    assert np.isclose(
        audit_cindex,
        original_cindex,
        rtol=1e-10,
        atol=1e-12
    ), (
        f"Fold {fold}: C-index 불일치 "
        f"{audit_cindex} vs {original_cindex}"
    )

    assert (
        audit_nonzero
        == original_nonzero
    ), (
        f"Fold {fold}: non-zero feature 수 불일치"
    )


    # ========================================================
    # 15. Fold 결과 기록
    # ========================================================
    audit_results.append({
        "fold": fold,

        "fit_n": len(
            fit_df_audit
        ),

        "cv_n": len(
            cv_df_audit
        ),

        "patient_overlap": len(
            overlap
        ),

        "valid_genes": len(
            valid_genes
        ),

        "min_pathway_genes": min(
            pathway_gene_counts.values()
        ),

        "max_pathway_genes": max(
            pathway_gene_counts.values()
        ),

        "max_abs_fit_gene_mean": (
            max_abs_gene_mean
        ),

        "original_alpha": (
            original_alpha
        ),

        "recomputed_alpha": float(
            audit_alpha
        ),

        "alpha_diff": (
            alpha_diff
        ),

        "original_cindex": (
            original_cindex
        ),

        "recomputed_cindex": float(
            audit_cindex
        ),

        "cindex_diff": (
            cindex_diff
        ),

        "original_nonzero": (
            original_nonzero
        ),

        "recomputed_nonzero": (
            audit_nonzero
        )
    })


# ============================================================
# 16. 최종 Audit 결과
# ============================================================

hallmark_cv_audit = pd.DataFrame(
    audit_results
)

print(
    hallmark_cv_audit.to_string(
        index=False
    )
)

print()

print(
    "Max z-score mean residual:",
    hallmark_cv_audit[
        "max_abs_fit_gene_mean"
    ].max()
)

print(
    "Max C-index difference:",
    hallmark_cv_audit[
        "cindex_diff"
    ].max()
)

print(
    "Max alpha difference:",
    hallmark_cv_audit[
        "alpha_diff"
    ].max()
)

print(
    "Total Fit/CV patient overlap:",
    hallmark_cv_audit[
        "patient_overlap"
    ].sum()
)

print(
    "\nClinical + Hallmark 5-fold leakage / "
    "reproducibility audit: PASSED"
)

 fold  fit_n  cv_n  patient_overlap  valid_genes  min_pathway_genes  max_pathway_genes  max_abs_fit_gene_mean  original_alpha  recomputed_alpha  alpha_diff  original_cindex  recomputed_cindex  cindex_diff  original_nonzero  recomputed_nonzero
    1    296    74                0         4356                 32                200               0.000002        0.022309          0.022309         0.0         0.768859           0.768859          0.0                12                  12
    2    296    74                0         4356                 32                200               0.000002        0.019895          0.019895         0.0         0.847534           0.847534          0.0                18                  18
    3    296    74                0         4356                 32                200               0.000002        0.022032          0.022032         0.0         0.741614           0.741614          0.0                16                  16
    4    296    74          

In [77]:
hallmark_cv_audit.to_csv(
    data_dir / "processed" / "clinical_pathway_cv_audit.csv",
    index=False
)

print(
    "Saved:",
    data_dir / "processed" / "clinical_pathway_cv_audit.csv"
)

Saved: ..\data\processed\clinical_pathway_cv_audit.csv


In [78]:
# =========================================
# PRE-TEST persistence + split audit
# =========================================

processed_dir = data_dir / "processed"

required_files = [
    # cohort / split
    "kirc_modeling_cohort.parquet",
    "rna_tpm_modeling_raw.parquet",
    "splits.csv",

    # RNA-only
    "rna_coxnet_coefficients.csv",
    "rna_coxnet_nonzero_genes.csv",

    # Clinical-only
    "clinical_coxnet_coefficients.csv",
    "clinical_coxnet_nonzero_features.csv",

    # Clinical + RNA
    "clinical_rna_coxnet_coefficients.csv",
    "clinical_rna_coxnet_nonzero_features.csv",
    "clinical_rna_nonzero_genes_annotated.csv",

    # Clinical + Hallmark
    "clinical_pathway_coxnet_coefficients.csv",
    "clinical_pathway_coxnet_nonzero_features.csv",
    "clinical_pathway_cv_audit.csv",

    # Validation comparison
    "validation_model_comparison.csv",
]

# -----------------------------
# 1. 파일 존재 여부
# -----------------------------
audit_rows = []

for filename in required_files:
    path = processed_dir / filename

    audit_rows.append({
        "file": filename,
        "exists": path.exists(),
        "size_kb": (
            round(path.stat().st_size / 1024, 2)
            if path.exists()
            else None
        )
    })

file_audit = pd.DataFrame(audit_rows)

print("=== Required files ===")
print(file_audit.to_string(index=False))

assert file_audit["exists"].all(), \
    "필수 파일 중 누락된 파일이 있습니다."

# -----------------------------
# 2. split 재확인
# -----------------------------
splits_check = pd.read_csv(
    processed_dir / "splits.csv"
)

print("\n=== Split counts ===")
print(
    splits_check["split"].value_counts()
)

assert len(splits_check) == 529
assert splits_check["patient_id"].is_unique

train_ids = set(
    splits_check.loc[
        splits_check["split"] == "train",
        "patient_id"
    ]
)

val_ids = set(
    splits_check.loc[
        splits_check["split"] == "validation",
        "patient_id"
    ]
)

test_ids = set(
    splits_check.loc[
        splits_check["split"] == "test",
        "patient_id"
    ]
)

assert len(train_ids) == 370
assert len(val_ids) == 79
assert len(test_ids) == 80

assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0

assert len(
    train_ids | val_ids | test_ids
) == 529

print("\nAll required files: PASSED")
print("Train / Validation / Test overlap: 0")
print("PRE-TEST audit: PASSED")

=== Required files ===
                                        file  exists  size_kb
                kirc_modeling_cohort.parquet    True    56.38
                rna_tpm_modeling_raw.parquet    True 59117.07
                                  splits.csv    True    10.66
                 rna_coxnet_coefficients.csv    True    31.70
                rna_coxnet_nonzero_genes.csv    True     9.42
            clinical_coxnet_coefficients.csv    True     0.54
        clinical_coxnet_nonzero_features.csv    True     0.48
        clinical_rna_coxnet_coefficients.csv    True    16.65
    clinical_rna_coxnet_nonzero_features.csv    True     1.58
    clinical_rna_nonzero_genes_annotated.csv    True     1.91
    clinical_pathway_coxnet_coefficients.csv    True     2.58
clinical_pathway_coxnet_nonzero_features.csv    True     1.02
               clinical_pathway_cv_audit.csv    True     0.90
             validation_model_comparison.csv    True     0.16

=== Split counts ===
split
train         370
t

In [79]:
# =========================================
# Final model specification freeze
# Test 평가 전에 모델 설정 고정
# =========================================

model_freeze = pd.DataFrame([
    {
        "model": "RNA-only",
        "input": "RNA Top-1000 MAD genes",
        "n_input_features": 1000,
        "rna_top_k": 1000,
        "l1_ratio": 0.75,
        "alpha_idx": 27,
        "cv_alpha_ratio": 0.079060,
        "full_train_alpha": 0.019114577707188597,
        "validation_harrell_c": 0.7262617621899059
    },

    {
        "model": "Clinical-only",
        "input": "Age + Sex + Stage + Grade (10 encoded features)",
        "n_input_features": 10,
        "rna_top_k": None,
        "l1_ratio": 0.90,
        "alpha_idx": 37,
        "cv_alpha_ratio": 0.030888,
        "full_train_alpha": 0.010057565713433423,
        "validation_harrell_c": 0.7840034217279727
    },

    {
        "model": "Clinical+RNA",
        "input": "Clinical 10 + RNA Top-500 MAD genes",
        "n_input_features": 510,
        "rna_top_k": 500,
        "l1_ratio": 0.50,
        "alpha_idx": 18,
        "cv_alpha_ratio": 0.184207,
        "full_train_alpha": 0.1079631601142658,
        "validation_harrell_c": 0.8135158254918734
    },

    {
        "model": "Clinical+Hallmark",
        "input": "Clinical 10 + 50 Hallmark pathway scores",
        "n_input_features": 60,
        "rna_top_k": None,
        "l1_ratio": 0.75,
        "alpha_idx": 31,
        "cv_alpha_ratio": 0.054287,
        "full_train_alpha": 0.021211534321841412,
        "validation_harrell_c": 0.7733105218135158
    }
])

freeze_path = (
    data_dir
    / "processed"
    / "final_model_specifications.csv"
)

model_freeze.to_csv(
    freeze_path,
    index=False
)

print(model_freeze.to_string(index=False))

print("\nSaved:", freeze_path)

            model                                           input  n_input_features  rna_top_k  l1_ratio  alpha_idx  cv_alpha_ratio  full_train_alpha  validation_harrell_c
         RNA-only                          RNA Top-1000 MAD genes              1000     1000.0      0.75         27        0.079060          0.019115              0.726262
    Clinical-only Age + Sex + Stage + Grade (10 encoded features)                10        NaN      0.90         37        0.030888          0.010058              0.784003
     Clinical+RNA             Clinical 10 + RNA Top-500 MAD genes               510      500.0      0.50         18        0.184207          0.107963              0.813516
Clinical+Hallmark        Clinical 10 + 50 Hallmark pathway scores                60        NaN      0.75         31        0.054287          0.021212              0.773311

Saved: ..\data\processed\final_model_specifications.csv


In [80]:
# ============================================================
# FINAL TEST input QC
# 아직 모델 예측 / C-index 계산하지 않음
# ============================================================

# ------------------------------------------------------------
# 1. frozen split에서 Test 환자 다시 확인
# ------------------------------------------------------------
splits_check = pd.read_csv(
    data_dir / "processed" / "splits.csv"
)

test_ids_frozen = (
    splits_check.loc[
        splits_check["split"] == "test",
        "patient_id"
    ]
    .tolist()
)

assert len(test_ids_frozen) == 80
assert len(set(test_ids_frozen)) == 80


# ------------------------------------------------------------
# 2. modeling cohort에서 Test clinical data 재구성
# ------------------------------------------------------------
cohort_check = pd.read_parquet(
    data_dir / "processed" / "kirc_modeling_cohort.parquet"
)

test_df_check = (
    cohort_check
    .set_index("patient_id")
    .loc[test_ids_frozen]
    .reset_index()
)

assert test_df_check.shape[0] == 80
assert test_df_check["patient_id"].is_unique

assert test_df_check[
    ["pfi_event", "pfi_time"]
].isna().sum().sum() == 0

assert (
    test_df_check["pfi_time"] > 0
).all()


# ------------------------------------------------------------
# 3. Test RNA matrix 정렬 확인
# ------------------------------------------------------------
rna_check = pd.read_parquet(
    data_dir / "processed" / "rna_tpm_modeling_raw.parquet"
)

test_rna_check = rna_check.loc[
    test_ids_frozen
].copy()

assert test_rna_check.shape == (
    80,
    19962
)

assert test_rna_check.index.is_unique

assert (
    list(test_rna_check.index)
    == test_ids_frozen
)


# ------------------------------------------------------------
# 4. Train / Validation / Test overlap 재확인
# ------------------------------------------------------------
train_ids_frozen = set(
    splits_check.loc[
        splits_check["split"] == "train",
        "patient_id"
    ]
)

val_ids_frozen = set(
    splits_check.loc[
        splits_check["split"] == "validation",
        "patient_id"
    ]
)

test_ids_set = set(
    test_ids_frozen
)

assert len(
    train_ids_frozen & test_ids_set
) == 0

assert len(
    val_ids_frozen & test_ids_set
) == 0


# ------------------------------------------------------------
# 5. Feature availability QC
# ------------------------------------------------------------
required_clinical = [
    "age",
    "gender",
    "stage",
    "grade"
]

assert test_df_check[
    required_clinical
].isna().sum().sum() == 0

assert np.isfinite(
    test_rna_check.to_numpy()
).all()


# ------------------------------------------------------------
# 6. 결과 출력
# ------------------------------------------------------------
print("FINAL TEST INPUT QC: PASSED")
print("Test patients:", len(test_df_check))
print("Test RNA:", test_rna_check.shape)
print(
    "Clinical missing:",
    test_df_check[
        required_clinical
    ].isna().sum().sum()
)
print(
    "RNA NaN/Inf:",
    np.size(test_rna_check.to_numpy())
    - np.isfinite(test_rna_check.to_numpy()).sum()
)
print("Train/Test overlap: 0")
print("Validation/Test overlap: 0")
print("No Test prediction has been calculated yet.")

FINAL TEST INPUT QC: PASSED
Test patients: 80
Test RNA: (80, 19962)
Clinical missing: 0
RNA NaN/Inf: 0
Train/Test overlap: 0
Validation/Test overlap: 0
No Test prediction has been calculated yet.


In [81]:
# ============================================================
# FINAL TEST 1/4 — RNA-only
# Frozen Train pipeline 재구성 → Test 80명 1회 평가
# ============================================================

RNA_TOP_K = 1000
RNA_L1_RATIO = 0.75
RNA_ALPHA_IDX = 27
RNA_FROZEN_ALPHA = 0.019114577707188597

# ------------------------------------------------------------
# 1. Train / Test raw RNA
# ------------------------------------------------------------
rna_raw = pd.read_parquet(
    data_dir / "processed" / "rna_tpm_modeling_raw.parquet"
)

rna_train_raw = rna_raw.loc[
    train_df["patient_id"]
].copy()

rna_test_raw = rna_raw.loc[
    test_df_check["patient_id"]
].copy()

assert rna_train_raw.shape == (370, 19962)
assert rna_test_raw.shape == (80, 19962)

# ------------------------------------------------------------
# 2. Train-only low-expression filtering
# TPM >= 1 in >= 20% of Train
# ------------------------------------------------------------
min_samples = int(np.ceil(370 * 0.20))

expr_counts = (
    rna_train_raw >= 1.0
).sum(axis=0)

kept_genes = expr_counts[
    expr_counts >= min_samples
].index

assert len(kept_genes) == 14220

rna_train_filtered = rna_train_raw[kept_genes]
rna_test_filtered = rna_test_raw[kept_genes]

# ------------------------------------------------------------
# 3. log2(TPM + 1)
# ------------------------------------------------------------
rna_train_log = np.log2(
    rna_train_filtered + 1
)

rna_test_log = np.log2(
    rna_test_filtered + 1
)

# ------------------------------------------------------------
# 4. Train-only MAD → Top 1000
# ------------------------------------------------------------
train_median = rna_train_log.median(axis=0)

train_mad = (
    rna_train_log
    .sub(train_median, axis=1)
    .abs()
    .median(axis=0)
)

top1000_genes = (
    train_mad
    .sort_values(ascending=False)
    .head(RNA_TOP_K)
    .index
)

assert len(top1000_genes) == 1000

X_train_rna = rna_train_log[
    top1000_genes
].to_numpy()

X_test_rna = rna_test_log[
    top1000_genes
].to_numpy()

# ------------------------------------------------------------
# 5. StandardScaler — Train에서만 fit
# ------------------------------------------------------------
rna_test_scaler = StandardScaler()

X_train_rna_scaled = (
    rna_test_scaler.fit_transform(
        X_train_rna
    )
)

X_test_rna_scaled = (
    rna_test_scaler.transform(
        X_test_rna
    )
)

# ------------------------------------------------------------
# 6. Survival targets
# ------------------------------------------------------------
y_train_rna = Surv.from_arrays(
    event=train_df["pfi_event"].astype(bool).to_numpy(),
    time=train_df["pfi_time"].astype(float).to_numpy()
)

y_test = Surv.from_arrays(
    event=test_df_check["pfi_event"].astype(bool).to_numpy(),
    time=test_df_check["pfi_time"].astype(float).to_numpy()
)

# ------------------------------------------------------------
# 7. Frozen CoxNet 재학습
# ------------------------------------------------------------
rna_test_model = CoxnetSurvivalAnalysis(
    l1_ratio=RNA_L1_RATIO,
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

rna_test_model.fit(
    X_train_rna_scaled,
    y_train_rna
)

rna_test_alpha = (
    rna_test_model.alphas_[
        RNA_ALPHA_IDX
    ]
)

# frozen alpha와 재현되는지 확인
assert np.isclose(
    rna_test_alpha,
    RNA_FROZEN_ALPHA,
    rtol=1e-8,
    atol=1e-10
), (
    f"RNA alpha mismatch: "
    f"{rna_test_alpha} vs {RNA_FROZEN_ALPHA}"
)

# ------------------------------------------------------------
# 8. 저장된 coefficient와 재현성 확인
# ------------------------------------------------------------
saved_rna_coef = pd.read_csv(
    data_dir
    / "processed"
    / "rna_coxnet_coefficients.csv"
)

assert "gene_id" in saved_rna_coef.columns
assert "coefficient" in saved_rna_coef.columns

assert set(saved_rna_coef["gene_id"]) == set(top1000_genes)

saved_coef_ordered = (
    saved_rna_coef
    .set_index("gene_id")
    .loc[top1000_genes, "coefficient"]
    .to_numpy()
)

recomputed_coef = (
    rna_test_model.coef_[
        :, RNA_ALPHA_IDX
    ]
)

coef_max_diff = np.max(
    np.abs(
        saved_coef_ordered
        - recomputed_coef
    )
)

assert np.allclose(
    saved_coef_ordered,
    recomputed_coef,
    rtol=1e-8,
    atol=1e-10
)

# ------------------------------------------------------------
# 9. Test risk / Harrell C-index
# ------------------------------------------------------------
rna_test_risk = rna_test_model.predict(
    X_test_rna_scaled,
    alpha=rna_test_alpha
)

rna_test_cindex = concordance_index_censored(
    y_test["event"],
    y_test["time"],
    rna_test_risk
)[0]

# ------------------------------------------------------------
# 10. Test prediction table 시작
# ------------------------------------------------------------
test_predictions = pd.DataFrame({
    "patient_id": test_df_check["patient_id"].to_numpy(),
    "pfi_event": test_df_check["pfi_event"].to_numpy(),
    "pfi_time": test_df_check["pfi_time"].to_numpy(),
    "rna_only_risk": rna_test_risk
})

# ------------------------------------------------------------
# 결과
# ------------------------------------------------------------
print("RNA-only frozen-model reproducibility: PASSED")
print("Train genes after low-expression filter:", len(kept_genes))
print("RNA input features:", len(top1000_genes))
print("Frozen alpha:", rna_test_alpha)
print("Max coefficient difference:", coef_max_diff)
print("Test patients:", len(test_predictions))
print("Test events:", int(test_df_check["pfi_event"].sum()))
print("RNA-only TEST Harrell C-index:", rna_test_cindex)

RNA-only frozen-model reproducibility: PASSED
Train genes after low-expression filter: 14220
RNA input features: 1000
Frozen alpha: 0.019114577707188597
Max coefficient difference: 9.71445146547012e-17
Test patients: 80
Test events: 24
RNA-only TEST Harrell C-index: 0.7145110410094637


In [82]:
# ============================================================
# FINAL TEST 2/4 — Clinical-only
# Frozen Train pipeline 재구성 → Test 80명 1회 평가
# ============================================================

CLIN_L1_RATIO = 0.90
CLIN_ALPHA_IDX = 37
CLIN_FROZEN_ALPHA = 0.010057565713433423

clinical_features = [
    "age",
    "gender",
    "stage",
    "grade"
]

# ------------------------------------------------------------
# 1. Clinical preprocessing
# Train에서만 fit
# ------------------------------------------------------------
clinical_test_preprocessor = ColumnTransformer(
    transformers=[
        (
            "age",
            "passthrough",
            ["age"]
        ),
        (
            "cat",
            OneHotEncoder(
                categories=[
                    ["MALE", "FEMALE"],
                    [
                        "Stage I",
                        "Stage II",
                        "Stage III",
                        "Stage IV",
                        "Unknown"
                    ],
                    [
                        "G1",
                        "G2",
                        "G3",
                        "G4",
                        "Unknown"
                    ]
                ],
                drop=[
                    "MALE",
                    "Stage I",
                    "G2"
                ],
                handle_unknown="ignore",
                sparse_output=False
            ),
            [
                "gender",
                "stage",
                "grade"
            ]
        )
    ]
)

X_train_clin = clinical_test_preprocessor.fit_transform(
    train_df[clinical_features]
)

X_test_clin = clinical_test_preprocessor.transform(
    test_df_check[clinical_features]
)

clinical_feature_names = (
    clinical_test_preprocessor
    .get_feature_names_out()
)

assert X_train_clin.shape == (370, 10)
assert X_test_clin.shape == (80, 10)
assert len(clinical_feature_names) == 10


# ------------------------------------------------------------
# 2. StandardScaler
# Train에서만 fit
# ------------------------------------------------------------
clinical_test_scaler = StandardScaler()

X_train_clin_scaled = (
    clinical_test_scaler.fit_transform(
        X_train_clin
    )
)

X_test_clin_scaled = (
    clinical_test_scaler.transform(
        X_test_clin
    )
)


# ------------------------------------------------------------
# 3. Survival target
# ------------------------------------------------------------
y_train_clin = Surv.from_arrays(
    event=train_df[
        "pfi_event"
    ].astype(bool).to_numpy(),

    time=train_df[
        "pfi_time"
    ].astype(float).to_numpy()
)

# y_test는 RNA-only 단계에서 이미 생성됨
assert len(y_test) == 80


# ------------------------------------------------------------
# 4. Frozen Clinical CoxNet 재학습
# ------------------------------------------------------------
clinical_test_model = CoxnetSurvivalAnalysis(
    l1_ratio=CLIN_L1_RATIO,
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

clinical_test_model.fit(
    X_train_clin_scaled,
    y_train_clin
)

clinical_test_alpha = (
    clinical_test_model.alphas_[
        CLIN_ALPHA_IDX
    ]
)

# frozen alpha 재현 확인
assert np.isclose(
    clinical_test_alpha,
    CLIN_FROZEN_ALPHA,
    rtol=1e-8,
    atol=1e-10
), (
    f"Clinical alpha mismatch: "
    f"{clinical_test_alpha} vs "
    f"{CLIN_FROZEN_ALPHA}"
)


# ------------------------------------------------------------
# 5. 저장된 coefficient와 재현성 확인
# ------------------------------------------------------------
saved_clin_coef = pd.read_csv(
    data_dir
    / "processed"
    / "clinical_coxnet_coefficients.csv"
)

assert "feature" in saved_clin_coef.columns
assert "coefficient" in saved_clin_coef.columns

assert set(
    saved_clin_coef["feature"]
) == set(
    clinical_feature_names
)

saved_clin_coef_ordered = (
    saved_clin_coef
    .set_index("feature")
    .loc[
        clinical_feature_names,
        "coefficient"
    ]
    .to_numpy()
)

recomputed_clin_coef = (
    clinical_test_model.coef_[
        :,
        CLIN_ALPHA_IDX
    ]
)

clin_coef_max_diff = np.max(
    np.abs(
        saved_clin_coef_ordered
        - recomputed_clin_coef
    )
)

assert np.allclose(
    saved_clin_coef_ordered,
    recomputed_clin_coef,
    rtol=1e-8,
    atol=1e-10
)


# ------------------------------------------------------------
# 6. Test risk / Harrell C-index
# ------------------------------------------------------------
clinical_test_risk = (
    clinical_test_model.predict(
        X_test_clin_scaled,
        alpha=clinical_test_alpha
    )
)

clinical_test_cindex = (
    concordance_index_censored(
        y_test["event"],
        y_test["time"],
        clinical_test_risk
    )[0]
)


# ------------------------------------------------------------
# 7. Test prediction table에 추가
# ------------------------------------------------------------
assert list(
    test_predictions["patient_id"]
) == list(
    test_df_check["patient_id"]
)

test_predictions[
    "clinical_only_risk"
] = clinical_test_risk


# ------------------------------------------------------------
# 8. 출력
# ------------------------------------------------------------
print(
    "Clinical-only frozen-model "
    "reproducibility: PASSED"
)

print(
    "Clinical input features:",
    len(clinical_feature_names)
)

print(
    "Frozen alpha:",
    clinical_test_alpha
)

print(
    "Max coefficient difference:",
    clin_coef_max_diff
)

print(
    "Test patients:",
    len(test_predictions)
)

print(
    "Test events:",
    int(
        test_df_check[
            "pfi_event"
        ].sum()
    )
)

print(
    "Clinical-only TEST Harrell C-index:",
    clinical_test_cindex
)

Clinical-only frozen-model reproducibility: PASSED
Clinical input features: 10
Frozen alpha: 0.010057565713433423
Max coefficient difference: 6.591949208711867e-17
Test patients: 80
Test events: 24
Clinical-only TEST Harrell C-index: 0.8868296529968455


In [83]:
# ============================================================
# FINAL TEST 3/4 — Clinical + RNA
# Frozen Train pipeline 재구성 → Test 80명 1회 평가
# ============================================================

COMB_RNA_TOP_K = 500
COMB_L1_RATIO = 0.50
COMB_ALPHA_IDX = 18
COMB_FROZEN_ALPHA = 0.1079631601142658

clinical_features = [
    "age",
    "gender",
    "stage",
    "grade"
]

# ------------------------------------------------------------
# 1. Raw Train / Test RNA
# ------------------------------------------------------------
rna_raw_comb = pd.read_parquet(
    data_dir / "processed" / "rna_tpm_modeling_raw.parquet"
)

rna_train_comb_raw = rna_raw_comb.loc[
    train_df["patient_id"]
].copy()

rna_test_comb_raw = rna_raw_comb.loc[
    test_df_check["patient_id"]
].copy()

assert rna_train_comb_raw.shape == (370, 19962)
assert rna_test_comb_raw.shape == (80, 19962)


# ------------------------------------------------------------
# 2. Train-only low-expression filter
# TPM >= 1 in >= 20% of Train
# ------------------------------------------------------------
comb_min_samples = int(
    np.ceil(len(rna_train_comb_raw) * 0.20)
)

comb_expr_counts = (
    rna_train_comb_raw >= 1.0
).sum(axis=0)

comb_kept_genes = comb_expr_counts[
    comb_expr_counts >= comb_min_samples
].index

assert comb_min_samples == 74
assert len(comb_kept_genes) == 14220


# ------------------------------------------------------------
# 3. log2(TPM + 1)
# ------------------------------------------------------------
rna_train_comb_log = np.log2(
    rna_train_comb_raw[comb_kept_genes] + 1
)

rna_test_comb_log = np.log2(
    rna_test_comb_raw[comb_kept_genes] + 1
)


# ------------------------------------------------------------
# 4. Train-only MAD → Top 500
# ------------------------------------------------------------
comb_train_median = (
    rna_train_comb_log.median(axis=0)
)

comb_train_mad = (
    rna_train_comb_log
    .sub(comb_train_median, axis=1)
    .abs()
    .median(axis=0)
)

comb_top500_genes = (
    comb_train_mad
    .sort_values(ascending=False)
    .head(COMB_RNA_TOP_K)
    .index
)

assert len(comb_top500_genes) == 500


# ------------------------------------------------------------
# 5. Clinical preprocessing
# Train에서만 fit
# ------------------------------------------------------------
comb_clin_preprocessor = ColumnTransformer(
    transformers=[
        (
            "age",
            "passthrough",
            ["age"]
        ),
        (
            "cat",
            OneHotEncoder(
                categories=[
                    ["MALE", "FEMALE"],
                    [
                        "Stage I",
                        "Stage II",
                        "Stage III",
                        "Stage IV",
                        "Unknown"
                    ],
                    [
                        "G1",
                        "G2",
                        "G3",
                        "G4",
                        "Unknown"
                    ]
                ],
                drop=[
                    "MALE",
                    "Stage I",
                    "G2"
                ],
                handle_unknown="ignore",
                sparse_output=False
            ),
            [
                "gender",
                "stage",
                "grade"
            ]
        )
    ]
)

X_train_comb_clin = (
    comb_clin_preprocessor.fit_transform(
        train_df[clinical_features]
    )
)

X_test_comb_clin = (
    comb_clin_preprocessor.transform(
        test_df_check[clinical_features]
    )
)

comb_clin_feature_names = (
    comb_clin_preprocessor
    .get_feature_names_out()
)

assert X_train_comb_clin.shape == (370, 10)
assert X_test_comb_clin.shape == (80, 10)
assert len(comb_clin_feature_names) == 10


# ------------------------------------------------------------
# 6. RNA Top500 matrix
# ------------------------------------------------------------
X_train_comb_rna = (
    rna_train_comb_log[
        comb_top500_genes
    ]
    .to_numpy()
)

X_test_comb_rna = (
    rna_test_comb_log[
        comb_top500_genes
    ]
    .to_numpy()
)

assert X_train_comb_rna.shape == (370, 500)
assert X_test_comb_rna.shape == (80, 500)


# ------------------------------------------------------------
# 7. Clinical + RNA 결합
# 10 + 500 = 510
# ------------------------------------------------------------
X_train_comb = np.hstack([
    X_train_comb_clin,
    X_train_comb_rna
])

X_test_comb = np.hstack([
    X_test_comb_clin,
    X_test_comb_rna
])

comb_feature_names = (
    list(comb_clin_feature_names)
    + list(comb_top500_genes)
)

assert X_train_comb.shape == (370, 510)
assert X_test_comb.shape == (80, 510)
assert len(comb_feature_names) == 510


# ------------------------------------------------------------
# 8. StandardScaler
# Train에서만 fit
# ------------------------------------------------------------
comb_test_scaler = StandardScaler()

X_train_comb_scaled = (
    comb_test_scaler.fit_transform(
        X_train_comb
    )
)

X_test_comb_scaled = (
    comb_test_scaler.transform(
        X_test_comb
    )
)

assert np.isfinite(
    X_train_comb_scaled
).all()

assert np.isfinite(
    X_test_comb_scaled
).all()


# ------------------------------------------------------------
# 9. Survival target
# ------------------------------------------------------------
y_train_comb = Surv.from_arrays(
    event=train_df[
        "pfi_event"
    ].astype(bool).to_numpy(),

    time=train_df[
        "pfi_time"
    ].astype(float).to_numpy()
)

assert len(y_test) == 80


# ------------------------------------------------------------
# 10. Frozen Clinical + RNA CoxNet 재학습
# ------------------------------------------------------------
comb_test_model = CoxnetSurvivalAnalysis(
    l1_ratio=COMB_L1_RATIO,
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

comb_test_model.fit(
    X_train_comb_scaled,
    y_train_comb
)

comb_test_alpha = (
    comb_test_model.alphas_[
        COMB_ALPHA_IDX
    ]
)

assert np.isclose(
    comb_test_alpha,
    COMB_FROZEN_ALPHA,
    rtol=1e-8,
    atol=1e-10
), (
    f"Clinical+RNA alpha mismatch: "
    f"{comb_test_alpha} vs "
    f"{COMB_FROZEN_ALPHA}"
)


# ------------------------------------------------------------
# 11. 저장 coefficient와 완전 재현되는지 확인
# ------------------------------------------------------------
saved_comb_coef = pd.read_csv(
    data_dir
    / "processed"
    / "clinical_rna_coxnet_coefficients.csv"
)

assert "feature" in saved_comb_coef.columns
assert "coefficient" in saved_comb_coef.columns
assert len(saved_comb_coef) == 510

# feature 집합도 동일해야 함
assert set(
    saved_comb_coef["feature"]
) == set(
    comb_feature_names
), "저장된 Clinical+RNA feature와 재구성 feature가 다릅니다."

saved_comb_coef_ordered = (
    saved_comb_coef
    .set_index("feature")
    .loc[
        comb_feature_names,
        "coefficient"
    ]
    .to_numpy()
)

recomputed_comb_coef = (
    comb_test_model.coef_[
        :,
        COMB_ALPHA_IDX
    ]
)

comb_coef_max_diff = np.max(
    np.abs(
        saved_comb_coef_ordered
        - recomputed_comb_coef
    )
)

assert np.allclose(
    saved_comb_coef_ordered,
    recomputed_comb_coef,
    rtol=1e-8,
    atol=1e-10
)


# ------------------------------------------------------------
# 12. Test risk / Harrell C-index
# ------------------------------------------------------------
clinical_rna_test_risk = (
    comb_test_model.predict(
        X_test_comb_scaled,
        alpha=comb_test_alpha
    )
)

clinical_rna_test_cindex = (
    concordance_index_censored(
        y_test["event"],
        y_test["time"],
        clinical_rna_test_risk
    )[0]
)


# ------------------------------------------------------------
# 13. Test prediction table에 추가
# ------------------------------------------------------------
assert list(
    test_predictions["patient_id"]
) == list(
    test_df_check["patient_id"]
)

test_predictions[
    "clinical_rna_risk"
] = clinical_rna_test_risk


# ------------------------------------------------------------
# 14. 출력
# ------------------------------------------------------------
print(
    "Clinical+RNA frozen-model reproducibility: PASSED"
)

print(
    "Train genes after low-expression filter:",
    len(comb_kept_genes)
)

print(
    "RNA Top-K:",
    len(comb_top500_genes)
)

print(
    "Combined input features:",
    len(comb_feature_names)
)

print(
    "Frozen alpha:",
    comb_test_alpha
)

print(
    "Max coefficient difference:",
    comb_coef_max_diff
)

print(
    "Test patients:",
    len(test_predictions)
)

print(
    "Test events:",
    int(
        test_df_check[
            "pfi_event"
        ].sum()
    )
)

print(
    "Clinical+RNA TEST Harrell C-index:",
    clinical_rna_test_cindex
)

Clinical+RNA frozen-model reproducibility: PASSED
Train genes after low-expression filter: 14220
RNA Top-K: 500
Combined input features: 510
Frozen alpha: 0.1079631601142658
Max coefficient difference: 9.71445146547012e-17
Test patients: 80
Test events: 24
Clinical+RNA TEST Harrell C-index: 0.8919558359621451


In [84]:
# ============================================================
# FINAL TEST 4/4 — Clinical + Hallmark
# Frozen Train pipeline 재구성 → Test 80명 1회 평가
# ============================================================

PATH_L1_RATIO = 0.75
PATH_ALPHA_IDX = 31
PATH_FROZEN_ALPHA = 0.021211534321841412

clinical_features = [
    "age",
    "gender",
    "stage",
    "grade"
]

# ------------------------------------------------------------
# 1. Train / Test Hallmark symbol-level TPM
# ------------------------------------------------------------
path_train_tpm = (
    rna_hallmark_symbol_tpm
    .loc[train_df["patient_id"]]
    .copy()
)

path_test_tpm = (
    rna_hallmark_symbol_tpm
    .loc[test_df_check["patient_id"]]
    .copy()
)

assert path_train_tpm.shape == (370, 4360)
assert path_test_tpm.shape == (80, 4360)

assert list(
    path_train_tpm.index
) == list(
    train_df["patient_id"]
)

assert list(
    path_test_tpm.index
) == list(
    test_df_check["patient_id"]
)


# ------------------------------------------------------------
# 2. log2(TPM + 1)
# ------------------------------------------------------------
path_train_log = np.log2(
    path_train_tpm + 1
)

path_test_log = np.log2(
    path_test_tpm + 1
)


# ------------------------------------------------------------
# 3. Gene mean / std
# 반드시 Full Train 370명에서만 계산
# ------------------------------------------------------------
path_gene_mean = (
    path_train_log.mean(axis=0)
)

path_gene_std = (
    path_train_log.std(
        axis=0,
        ddof=0
    )
)

path_valid_genes = (
    path_gene_std[
        path_gene_std > 0
    ]
    .index
)

print(
    "Hallmark valid genes:",
    len(path_valid_genes)
)


# ------------------------------------------------------------
# 4. Train 기준 gene z-score
# Test에는 Train mean/std만 적용
# ------------------------------------------------------------
path_train_z = (
    path_train_log[path_valid_genes]
    - path_gene_mean[path_valid_genes]
) / path_gene_std[path_valid_genes]

path_test_z = (
    path_test_log[path_valid_genes]
    - path_gene_mean[path_valid_genes]
) / path_gene_std[path_valid_genes]

assert np.isfinite(
    path_train_z.to_numpy()
).all()

assert np.isfinite(
    path_test_z.to_numpy()
).all()


# ------------------------------------------------------------
# 5. 50 Hallmark pathway scores
# ------------------------------------------------------------
recomputed_train_pathway = pd.DataFrame(
    index=path_train_z.index
)

test_pathway_scores = pd.DataFrame(
    index=path_test_z.index
)

test_pathway_gene_counts = {}

for pathway, genes in hallmark_sets.items():

    matched = [
        gene
        for gene in genes
        if gene in path_valid_genes
    ]

    assert len(matched) > 0, (
        f"{pathway}: matched gene 0개"
    )

    test_pathway_gene_counts[
        pathway
    ] = len(matched)

    recomputed_train_pathway[
        pathway
    ] = (
        path_train_z[
            matched
        ]
        .mean(axis=1)
    )

    test_pathway_scores[
        pathway
    ] = (
        path_test_z[
            matched
        ]
        .mean(axis=1)
    )


assert recomputed_train_pathway.shape == (370, 50)
assert test_pathway_scores.shape == (80, 50)

assert np.isfinite(
    test_pathway_scores.to_numpy()
).all()


# ------------------------------------------------------------
# 6. 기존 Full-Train pathway score 재현 확인
# ------------------------------------------------------------
assert list(
    recomputed_train_pathway.columns
) == list(
    train_pathway_scores.columns
)

train_pathway_max_diff = np.max(
    np.abs(
        recomputed_train_pathway.to_numpy()
        - train_pathway_scores.to_numpy()
    )
)

assert np.allclose(
    recomputed_train_pathway.to_numpy(),
    train_pathway_scores.to_numpy(),
    rtol=1e-10,
    atol=1e-12
)


# ------------------------------------------------------------
# 7. Clinical preprocessing
# Train에서만 fit
# ------------------------------------------------------------
path_clin_preprocessor = ColumnTransformer(
    transformers=[
        (
            "age",
            "passthrough",
            ["age"]
        ),
        (
            "cat",
            OneHotEncoder(
                categories=[
                    [
                        "MALE",
                        "FEMALE"
                    ],
                    [
                        "Stage I",
                        "Stage II",
                        "Stage III",
                        "Stage IV",
                        "Unknown"
                    ],
                    [
                        "G1",
                        "G2",
                        "G3",
                        "G4",
                        "Unknown"
                    ]
                ],
                drop=[
                    "MALE",
                    "Stage I",
                    "G2"
                ],
                handle_unknown="ignore",
                sparse_output=False
            ),
            [
                "gender",
                "stage",
                "grade"
            ]
        )
    ]
)

X_train_path_clin = (
    path_clin_preprocessor.fit_transform(
        train_df[
            clinical_features
        ]
    )
)

X_test_path_clin = (
    path_clin_preprocessor.transform(
        test_df_check[
            clinical_features
        ]
    )
)

path_clin_feature_names = (
    path_clin_preprocessor
    .get_feature_names_out()
)

assert X_train_path_clin.shape == (370, 10)
assert X_test_path_clin.shape == (80, 10)


# ------------------------------------------------------------
# 8. Clinical 10 + Hallmark 50
# ------------------------------------------------------------
X_train_path = np.hstack([
    X_train_path_clin,
    recomputed_train_pathway.to_numpy()
])

X_test_path = np.hstack([
    X_test_path_clin,
    test_pathway_scores.to_numpy()
])

path_feature_names = (
    list(path_clin_feature_names)
    + list(
        recomputed_train_pathway.columns
    )
)

assert X_train_path.shape == (370, 60)
assert X_test_path.shape == (80, 60)
assert len(path_feature_names) == 60


# ------------------------------------------------------------
# 9. StandardScaler
# Train에서만 fit
# ------------------------------------------------------------
path_test_scaler = StandardScaler()

X_train_path_scaled = (
    path_test_scaler.fit_transform(
        X_train_path
    )
)

X_test_path_scaled = (
    path_test_scaler.transform(
        X_test_path
    )
)

assert np.isfinite(
    X_train_path_scaled
).all()

assert np.isfinite(
    X_test_path_scaled
).all()


# ------------------------------------------------------------
# 10. Survival target
# ------------------------------------------------------------
y_train_path = Surv.from_arrays(
    event=train_df[
        "pfi_event"
    ].astype(bool).to_numpy(),

    time=train_df[
        "pfi_time"
    ].astype(float).to_numpy()
)

assert len(y_test) == 80


# ------------------------------------------------------------
# 11. Frozen Clinical + Hallmark CoxNet 재학습
# ------------------------------------------------------------
path_test_model = CoxnetSurvivalAnalysis(
    l1_ratio=PATH_L1_RATIO,
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

path_test_model.fit(
    X_train_path_scaled,
    y_train_path
)

path_test_alpha = (
    path_test_model.alphas_[
        PATH_ALPHA_IDX
    ]
)

assert np.isclose(
    path_test_alpha,
    PATH_FROZEN_ALPHA,
    rtol=1e-8,
    atol=1e-10
), (
    f"Clinical+Hallmark alpha mismatch: "
    f"{path_test_alpha} vs "
    f"{PATH_FROZEN_ALPHA}"
)


# ------------------------------------------------------------
# 12. 저장된 coefficient와 재현성 확인
# ------------------------------------------------------------
saved_path_coef = pd.read_csv(
    data_dir
    / "processed"
    / "clinical_pathway_coxnet_coefficients.csv"
)

assert "feature" in saved_path_coef.columns
assert "coefficient" in saved_path_coef.columns
assert len(saved_path_coef) == 60

assert set(
    saved_path_coef["feature"]
) == set(
    path_feature_names
)

saved_path_coef_ordered = (
    saved_path_coef
    .set_index("feature")
    .loc[
        path_feature_names,
        "coefficient"
    ]
    .to_numpy()
)

recomputed_path_coef = (
    path_test_model.coef_[
        :,
        PATH_ALPHA_IDX
    ]
)

path_coef_max_diff = np.max(
    np.abs(
        saved_path_coef_ordered
        - recomputed_path_coef
    )
)

assert np.allclose(
    saved_path_coef_ordered,
    recomputed_path_coef,
    rtol=1e-8,
    atol=1e-10
)


# ------------------------------------------------------------
# 13. Test risk / Harrell C-index
# ------------------------------------------------------------
clinical_pathway_test_risk = (
    path_test_model.predict(
        X_test_path_scaled,
        alpha=path_test_alpha
    )
)

clinical_pathway_test_cindex = (
    concordance_index_censored(
        y_test["event"],
        y_test["time"],
        clinical_pathway_test_risk
    )[0]
)


# ------------------------------------------------------------
# 14. Test prediction table에 추가
# ------------------------------------------------------------
assert list(
    test_predictions["patient_id"]
) == list(
    test_df_check["patient_id"]
)

test_predictions[
    "clinical_pathway_risk"
] = clinical_pathway_test_risk


# ------------------------------------------------------------
# 15. 출력
# ------------------------------------------------------------
print(
    "Clinical+Hallmark frozen-model reproducibility: PASSED"
)

print(
    "Hallmark valid genes:",
    len(path_valid_genes)
)

print(
    "Pathway gene count min/max:",
    min(test_pathway_gene_counts.values()),
    "/",
    max(test_pathway_gene_counts.values())
)

print(
    "Train pathway max difference:",
    train_pathway_max_diff
)

print(
    "Combined input features:",
    len(path_feature_names)
)

print(
    "Frozen alpha:",
    path_test_alpha
)

print(
    "Max coefficient difference:",
    path_coef_max_diff
)

print(
    "Test patients:",
    len(test_predictions)
)

print(
    "Test events:",
    int(
        test_df_check[
            "pfi_event"
        ].sum()
    )
)

print(
    "Clinical+Hallmark TEST Harrell C-index:",
    clinical_pathway_test_cindex
)

Hallmark valid genes: 4356
Clinical+Hallmark frozen-model reproducibility: PASSED
Hallmark valid genes: 4356
Pathway gene count min/max: 32 / 200
Train pathway max difference: 0.0
Combined input features: 60
Frozen alpha: 0.021211534321841412
Max coefficient difference: 8.326672684688674e-17
Test patients: 80
Test events: 24
Clinical+Hallmark TEST Harrell C-index: 0.8856466876971609


In [85]:
# ============================================================
# Final Test results + patient-level predictions 저장
# ============================================================

# 1. Test prediction table QC
expected_prediction_cols = [
    "patient_id",
    "pfi_event",
    "pfi_time",
    "rna_only_risk",
    "clinical_only_risk",
    "clinical_rna_risk",
    "clinical_pathway_risk",
]

assert list(test_predictions.columns) == expected_prediction_cols
assert len(test_predictions) == 80
assert test_predictions["patient_id"].is_unique

assert test_predictions[
    [
        "rna_only_risk",
        "clinical_only_risk",
        "clinical_rna_risk",
        "clinical_pathway_risk",
    ]
].isna().sum().sum() == 0

assert np.isfinite(
    test_predictions[
        [
            "rna_only_risk",
            "clinical_only_risk",
            "clinical_rna_risk",
            "clinical_pathway_risk",
        ]
    ].to_numpy()
).all()


# 2. Final model comparison
final_model_comparison = pd.DataFrame({
    "model": [
        "RNA-only",
        "Clinical-only",
        "Clinical+RNA",
        "Clinical+Hallmark",
    ],
    "validation_harrell_c": [
        0.7262617621899059,
        0.7840034217279727,
        0.8135158254918734,
        0.7733105218135158,
    ],
    "test_harrell_c": [
        rna_test_cindex,
        clinical_test_cindex,
        clinical_rna_test_cindex,
        clinical_pathway_test_cindex,
    ],
})


# 3. 저장
prediction_path = (
    data_dir
    / "processed"
    / "test_predictions.csv"
)

comparison_path = (
    data_dir
    / "processed"
    / "final_model_comparison.csv"
)

test_predictions.to_csv(
    prediction_path,
    index=False
)

final_model_comparison.to_csv(
    comparison_path,
    index=False
)


# 4. 출력
print(final_model_comparison.to_string(index=False))

print("\nSaved:")
print(prediction_path)
print(comparison_path)

            model  validation_harrell_c  test_harrell_c
         RNA-only              0.726262        0.714511
    Clinical-only              0.784003        0.886830
     Clinical+RNA              0.813516        0.891956
Clinical+Hallmark              0.773311        0.885647

Saved:
..\data\processed\test_predictions.csv
..\data\processed\final_model_comparison.csv


In [86]:
# ============================================================
# FINAL TEST — Paired bootstrap 95% CI
# Fixed frozen-model predictions only
# ============================================================

from sksurv.metrics import concordance_index_censored

BOOTSTRAP_N = 5000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

# ------------------------------------------------------------
# 1. Test outcome / prediction 준비
# ------------------------------------------------------------
event = (
    test_predictions["pfi_event"]
    .astype(bool)
    .to_numpy()
)

time = (
    test_predictions["pfi_time"]
    .astype(float)
    .to_numpy()
)

risk_columns = {
    "RNA-only": "rna_only_risk",
    "Clinical-only": "clinical_only_risk",
    "Clinical+RNA": "clinical_rna_risk",
    "Clinical+Hallmark": "clinical_pathway_risk",
}

risks = {
    model: test_predictions[col].to_numpy()
    for model, col in risk_columns.items()
}

n_test = len(test_predictions)

assert n_test == 80
assert event.sum() == 24


# ------------------------------------------------------------
# 2. 원래 Test C-index 다시 계산
# ------------------------------------------------------------
observed_c = {}

for model, risk in risks.items():

    observed_c[model] = (
        concordance_index_censored(
            event,
            time,
            risk
        )[0]
    )

print("=== Observed Test C-index ===")

for model, value in observed_c.items():
    print(
        f"{model:20s}: {value:.6f}"
    )


# ------------------------------------------------------------
# 3. Paired patient-level bootstrap
# ------------------------------------------------------------
bootstrap_rows = []

invalid_bootstrap = 0

for b in range(BOOTSTRAP_N):

    # 같은 환자가 여러 번 뽑힐 수 있는
    # ordinary nonparametric bootstrap
    idx = rng.integers(
        0,
        n_test,
        size=n_test
    )

    event_b = event[idx]
    time_b = time[idx]

    row = {
        "bootstrap": b
    }

    try:

        for model, risk in risks.items():

            risk_b = risk[idx]

            c_b = (
                concordance_index_censored(
                    event_b,
                    time_b,
                    risk_b
                )[0]
            )

            row[model] = c_b

        bootstrap_rows.append(
            row
        )

    except Exception:
        # 비교 가능한 pair가 없는 극단적인 bootstrap sample
        invalid_bootstrap += 1


bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

print(
    "\nValid bootstrap samples:",
    len(bootstrap_df)
)

print(
    "Invalid bootstrap samples:",
    invalid_bootstrap
)

assert len(bootstrap_df) >= 0.95 * BOOTSTRAP_N, (
    "유효 bootstrap 반복이 너무 적습니다."
)


# ------------------------------------------------------------
# 4. 각 모델 C-index 95% percentile CI
# ------------------------------------------------------------
ci_rows = []

for model in risk_columns.keys():

    lower = bootstrap_df[
        model
    ].quantile(0.025)

    upper = bootstrap_df[
        model
    ].quantile(0.975)

    ci_rows.append({
        "model": model,
        "test_harrell_c": observed_c[model],
        "ci_95_lower": lower,
        "ci_95_upper": upper
    })


test_ci_df = pd.DataFrame(
    ci_rows
)


# ------------------------------------------------------------
# 5. Paired C-index difference
#
# 핵심 질문:
# RNA를 Clinical에 추가했을 때
# discrimination이 얼마나 달라졌는가?
# ------------------------------------------------------------
bootstrap_df[
    "delta_clinical_rna_vs_clinical"
] = (
    bootstrap_df["Clinical+RNA"]
    - bootstrap_df["Clinical-only"]
)

bootstrap_df[
    "delta_pathway_vs_clinical"
] = (
    bootstrap_df["Clinical+Hallmark"]
    - bootstrap_df["Clinical-only"]
)

bootstrap_df[
    "delta_clinical_rna_vs_rna"
] = (
    bootstrap_df["Clinical+RNA"]
    - bootstrap_df["RNA-only"]
)


difference_rows = [
    {
        "comparison": "Clinical+RNA - Clinical-only",
        "observed_difference": (
            observed_c["Clinical+RNA"]
            - observed_c["Clinical-only"]
        ),
        "ci_95_lower": bootstrap_df[
            "delta_clinical_rna_vs_clinical"
        ].quantile(0.025),
        "ci_95_upper": bootstrap_df[
            "delta_clinical_rna_vs_clinical"
        ].quantile(0.975),
    },

    {
        "comparison": "Clinical+Hallmark - Clinical-only",
        "observed_difference": (
            observed_c["Clinical+Hallmark"]
            - observed_c["Clinical-only"]
        ),
        "ci_95_lower": bootstrap_df[
            "delta_pathway_vs_clinical"
        ].quantile(0.025),
        "ci_95_upper": bootstrap_df[
            "delta_pathway_vs_clinical"
        ].quantile(0.975),
    },

    {
        "comparison": "Clinical+RNA - RNA-only",
        "observed_difference": (
            observed_c["Clinical+RNA"]
            - observed_c["RNA-only"]
        ),
        "ci_95_lower": bootstrap_df[
            "delta_clinical_rna_vs_rna"
        ].quantile(0.025),
        "ci_95_upper": bootstrap_df[
            "delta_clinical_rna_vs_rna"
        ].quantile(0.975),
    }
]

difference_ci_df = pd.DataFrame(
    difference_rows
)


# ------------------------------------------------------------
# 6. 저장
# ------------------------------------------------------------
test_ci_df.to_csv(
    data_dir
    / "processed"
    / "test_harrell_c_bootstrap_ci.csv",
    index=False
)

difference_ci_df.to_csv(
    data_dir
    / "processed"
    / "test_harrell_c_difference_ci.csv",
    index=False
)


# ------------------------------------------------------------
# 7. 출력
# ------------------------------------------------------------
print(
    "\n=== Test Harrell C-index 95% Bootstrap CI ==="
)

print(
    test_ci_df.to_string(
        index=False
    )
)

print(
    "\n=== Paired C-index Difference 95% Bootstrap CI ==="
)

print(
    difference_ci_df.to_string(
        index=False
    )
)

=== Observed Test C-index ===
RNA-only            : 0.714511
Clinical-only       : 0.886830
Clinical+RNA        : 0.891956
Clinical+Hallmark   : 0.885647

Valid bootstrap samples: 5000
Invalid bootstrap samples: 0

=== Test Harrell C-index 95% Bootstrap CI ===
            model  test_harrell_c  ci_95_lower  ci_95_upper
         RNA-only        0.714511     0.589566     0.829764
    Clinical-only        0.886830     0.826026     0.940299
     Clinical+RNA        0.891956     0.844103     0.938447
Clinical+Hallmark        0.885647     0.817031     0.944232

=== Paired C-index Difference 95% Bootstrap CI ===
                       comparison  observed_difference  ci_95_lower  ci_95_upper
     Clinical+RNA - Clinical-only             0.005126    -0.024295     0.044045
Clinical+Hallmark - Clinical-only            -0.001183    -0.040630     0.029281
          Clinical+RNA - RNA-only             0.177445     0.067657     0.306509


In [87]:
# ============================================================
# Survival evaluation horizon QC
# 1y / 3y / 5y 평가가 실제 데이터에서 가능한지 확인
# ============================================================

HORIZONS = {
    "1_year": 365,
    "3_year": 1095,
    "5_year": 1825,
}

def horizon_summary(df, name):
    rows = []

    for label, t in HORIZONS.items():

        at_risk = (df["pfi_time"] >= t).sum()

        events_before = (
            (df["pfi_event"] == 1)
            & (df["pfi_time"] <= t)
        ).sum()

        censored_before = (
            (df["pfi_event"] == 0)
            & (df["pfi_time"] < t)
        ).sum()

        rows.append({
            "dataset": name,
            "horizon": label,
            "days": t,
            "n": len(df),
            "at_risk_at_horizon": int(at_risk),
            "events_by_horizon": int(events_before),
            "censored_before_horizon": int(censored_before),
        })

    return pd.DataFrame(rows)


horizon_qc = pd.concat([
    horizon_summary(train_df, "Train"),
    horizon_summary(val_df, "Validation"),
    horizon_summary(test_df_check, "Test"),
], ignore_index=True)

print(horizon_qc.to_string(index=False))

print("\nFollow-up distribution:")
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df_check),
]:
    print(
        name,
        df["pfi_time"].describe(
            percentiles=[0.25, 0.5, 0.75, 0.9]
        ).to_dict()
    )

   dataset horizon  days   n  at_risk_at_horizon  events_by_horizon  censored_before_horizon
     Train  1_year   365 370                 275                 56                       39
     Train  3_year  1095 370                 182                 85                      103
     Train  5_year  1825 370                  84                101                      185
Validation  1_year   365  79                  58                 10                       11
Validation  3_year  1095  79                  35                 18                       26
Validation  5_year  1825  79                  14                 23                       42
      Test  1_year   365  80                  63                 11                        6
      Test  3_year  1095  80                  37                 19                       24
      Test  5_year  1825  80                  24                 21                       35

Follow-up distribution:
Train {'count': 370.0, 'mean': 1167.205405405

In [88]:
# ============================================================
# IPCW support QC
# Train censoring distribution G(t) 확인
# ============================================================

from sksurv.nonparametric import kaplan_meier_estimator

train_event = (
    train_df["pfi_event"]
    .astype(bool)
    .to_numpy()
)

train_time = (
    train_df["pfi_time"]
    .astype(float)
    .to_numpy()
)

# reverse=True → censoring distribution G(t)
censor_time, censor_surv = kaplan_meier_estimator(
    train_event,
    train_time,
    reverse=True
)


def step_value_at(t, x, y):
    """
    Kaplan-Meier step function의 t 시점 값.
    t 이전 마지막 step 값을 반환.
    """
    idx = np.searchsorted(
        x,
        t,
        side="right"
    ) - 1

    if idx < 0:
        return 1.0

    return float(y[idx])


horizon_ipcw_qc = []

for label, t in [
    ("1_year", 365),
    ("3_year", 1095),
    ("5_year", 1825),
]:

    g_t = step_value_at(
        t,
        censor_time,
        censor_surv
    )

    horizon_ipcw_qc.append({
        "horizon": label,
        "days": t,
        "G_hat_train": g_t,
        "approx_IPCW_at_horizon": (
            np.inf if g_t == 0
            else 1.0 / g_t
        ),
        "test_at_risk": int(
            (test_df_check["pfi_time"] >= t).sum()
        ),
        "test_events_by_t": int(
            (
                (test_df_check["pfi_event"] == 1)
                & (test_df_check["pfi_time"] <= t)
            ).sum()
        ),
    })


ipcw_support_qc = pd.DataFrame(
    horizon_ipcw_qc
)

print(
    ipcw_support_qc.to_string(
        index=False
    )
)

print(
    "\nTrain max follow-up:",
    train_df["pfi_time"].max()
)

print(
    "Test max follow-up:",
    test_df_check["pfi_time"].max()
)

print(
    "Test max EVENT time:",
    test_df_check.loc[
        test_df_check["pfi_event"] == 1,
        "pfi_time"
    ].max()
)

horizon  days  G_hat_train  approx_IPCW_at_horizon  test_at_risk  test_events_by_t
 1_year   365     0.880069                1.136274            63                11
 3_year  1095     0.665627                1.502343            37                19
 5_year  1825     0.349239                2.863367            24                21

Train max follow-up: 3987.0
Test max follow-up: 4074.0
Test max EVENT time: 3766.0


In [89]:
# ============================================================
# FINAL TEST — IPCW discrimination metrics
# 1-year / 3-year cumulative dynamic AUC
# 3-year Uno IPCW C-index
#
# 모델 재학습 없음
# frozen Test risk scores만 사용
# ============================================================

from sksurv.metrics import (
    cumulative_dynamic_auc,
    concordance_index_ipcw,
)

# ------------------------------------------------------------
# 1. Survival outcome
# ------------------------------------------------------------
y_train_eval = Surv.from_arrays(
    event=train_df["pfi_event"].astype(bool).to_numpy(),
    time=train_df["pfi_time"].astype(float).to_numpy()
)

y_test_eval = Surv.from_arrays(
    event=test_predictions["pfi_event"].astype(bool).to_numpy(),
    time=test_predictions["pfi_time"].astype(float).to_numpy()
)

assert len(y_train_eval) == 370
assert len(y_test_eval) == 80


# ------------------------------------------------------------
# 2. Frozen risk scores
# ------------------------------------------------------------
risk_map = {
    "RNA-only":
        test_predictions["rna_only_risk"].to_numpy(),

    "Clinical-only":
        test_predictions["clinical_only_risk"].to_numpy(),

    "Clinical+RNA":
        test_predictions["clinical_rna_risk"].to_numpy(),

    "Clinical+Hallmark":
        test_predictions["clinical_pathway_risk"].to_numpy(),
}

for model, risk in risk_map.items():
    assert len(risk) == 80
    assert np.isfinite(risk).all()


# ------------------------------------------------------------
# 3. Evaluation horizons
# ------------------------------------------------------------
eval_times = np.array(
    [365.0, 1095.0]
)

UNO_TAU = 1095.0


# ------------------------------------------------------------
# 4. Cumulative / dynamic AUC
# ------------------------------------------------------------
auc_rows = []

for model, risk in risk_map.items():

    auc_values, mean_auc = cumulative_dynamic_auc(
        y_train_eval,
        y_test_eval,
        risk,
        eval_times
    )

    auc_rows.append({
        "model": model,
        "auc_1year": float(auc_values[0]),
        "auc_3year": float(auc_values[1]),
        "mean_auc_1to3year_points": float(mean_auc),
    })


time_auc_df = pd.DataFrame(
    auc_rows
)


# ------------------------------------------------------------
# 5. Uno IPCW C-index truncated at 3 years
# ------------------------------------------------------------
uno_rows = []

for model, risk in risk_map.items():

    uno_result = concordance_index_ipcw(
        y_train_eval,
        y_test_eval,
        risk,
        tau=UNO_TAU
    )

    uno_rows.append({
        "model": model,
        "uno_c_3year": float(uno_result[0])
    })


uno_df = pd.DataFrame(
    uno_rows
)


# ------------------------------------------------------------
# 6. Harrell + Uno + AUC 하나로 합치기
# ------------------------------------------------------------
discrimination_summary = (
    final_model_comparison[
        [
            "model",
            "test_harrell_c"
        ]
    ]
    .merge(
        uno_df,
        on="model"
    )
    .merge(
        time_auc_df,
        on="model"
    )
)


# ------------------------------------------------------------
# 7. 저장
# ------------------------------------------------------------
discrimination_path = (
    data_dir
    / "processed"
    / "test_discrimination_metrics.csv"
)

discrimination_summary.to_csv(
    discrimination_path,
    index=False
)


# ------------------------------------------------------------
# 8. 출력
# ------------------------------------------------------------
print(
    "=== FINAL TEST DISCRIMINATION METRICS ==="
)

print(
    discrimination_summary.to_string(
        index=False
    )
)

print(
    "\nSaved:",
    discrimination_path
)

=== FINAL TEST DISCRIMINATION METRICS ===
            model  test_harrell_c  uno_c_3year  auc_1year  auc_3year  mean_auc_1to3year_points
         RNA-only        0.714511     0.703233   0.707653   0.723406                  0.715037
    Clinical-only        0.886830     0.879822   0.888022   0.910865                  0.898729
     Clinical+RNA        0.891956     0.896988   0.879702   0.943951                  0.909816
Clinical+Hallmark        0.885647     0.868238   0.900662   0.895253                  0.898127

Saved: ..\data\processed\test_discrimination_metrics.csv


In [90]:
# ============================================================
# FINAL TEST — Brier score / IBS
# 동일 frozen CoxNet + baseline survival estimation
# ============================================================

from sksurv.metrics import brier_score, integrated_brier_score

# ------------------------------------------------------------
# 1. 평가 시간
# ------------------------------------------------------------
BRIER_TIMES = np.array([365.0, 1095.0])

# IBS는 1년~3년 구간을 촘촘하게 평가
IBS_TIMES = np.linspace(
    365.0,
    1095.0,
    100
)


# ------------------------------------------------------------
# 2. Baseline survival을 포함한 frozen 모델 재적합
# ------------------------------------------------------------

baseline_models = {
    "RNA-only": (
        CoxnetSurvivalAnalysis(
            l1_ratio=0.75,
            n_alphas=50,
            alpha_min_ratio=0.01,
            max_iter=100000,
            fit_baseline_model=True
        ),
        X_train_rna_scaled,
        X_test_rna_scaled,
        27,
        0.019114577707188597,
        test_predictions["rna_only_risk"].to_numpy()
    ),

    "Clinical-only": (
        CoxnetSurvivalAnalysis(
            l1_ratio=0.90,
            n_alphas=50,
            alpha_min_ratio=0.01,
            max_iter=100000,
            fit_baseline_model=True
        ),
        X_train_clin_scaled,
        X_test_clin_scaled,
        37,
        0.010057565713433423,
        test_predictions["clinical_only_risk"].to_numpy()
    ),

    "Clinical+RNA": (
        CoxnetSurvivalAnalysis(
            l1_ratio=0.50,
            n_alphas=50,
            alpha_min_ratio=0.01,
            max_iter=100000,
            fit_baseline_model=True
        ),
        X_train_comb_scaled,
        X_test_comb_scaled,
        18,
        0.1079631601142658,
        test_predictions["clinical_rna_risk"].to_numpy()
    ),

    "Clinical+Hallmark": (
        CoxnetSurvivalAnalysis(
            l1_ratio=0.75,
            n_alphas=50,
            alpha_min_ratio=0.01,
            max_iter=100000,
            fit_baseline_model=True
        ),
        X_train_path_scaled,
        X_test_path_scaled,
        31,
        0.021211534321841412,
        test_predictions["clinical_pathway_risk"].to_numpy()
    ),
}


# ------------------------------------------------------------
# 3. Survival outcome
# ------------------------------------------------------------
y_train_brier = Surv.from_arrays(
    event=train_df["pfi_event"].astype(bool).to_numpy(),
    time=train_df["pfi_time"].astype(float).to_numpy()
)

y_test_brier = Surv.from_arrays(
    event=test_predictions["pfi_event"].astype(bool).to_numpy(),
    time=test_predictions["pfi_time"].astype(float).to_numpy()
)


# ------------------------------------------------------------
# 4. 각 모델 계산
# ------------------------------------------------------------
brier_rows = []

fitted_baseline_models = {}

for model_name, (
    model,
    X_train,
    X_test,
    alpha_idx,
    frozen_alpha,
    frozen_risk
) in baseline_models.items():

    # Train only
    model.fit(
        X_train,
        y_train_brier
    )

    alpha = model.alphas_[alpha_idx]

    # alpha 재현 확인
    assert np.isclose(
        alpha,
        frozen_alpha,
        rtol=1e-8,
        atol=1e-10
    ), (
        f"{model_name}: alpha mismatch "
        f"{alpha} vs {frozen_alpha}"
    )

    # --------------------------------------------------------
    # 기존 risk score와 동일한지 확인
    # --------------------------------------------------------
    new_risk = model.predict(
        X_test,
        alpha=alpha
    )

    risk_max_diff = np.max(
        np.abs(
            new_risk
            - frozen_risk
        )
    )

    assert np.allclose(
        new_risk,
        frozen_risk,
        rtol=1e-8,
        atol=1e-10
    )

    # --------------------------------------------------------
    # Survival functions
    # --------------------------------------------------------
    surv_funcs = model.predict_survival_function(
        X_test,
        alpha=alpha
    )

    # 1y / 3y survival probabilities
    surv_brier = np.asarray([
        [
            fn(t)
            for t in BRIER_TIMES
        ]
        for fn in surv_funcs
    ])

    # IBS용 survival probabilities
    surv_ibs = np.asarray([
        [
            fn(t)
            for t in IBS_TIMES
        ]
        for fn in surv_funcs
    ])

    assert surv_brier.shape == (80, 2)
    assert surv_ibs.shape == (80, 100)

    assert np.isfinite(surv_brier).all()
    assert np.isfinite(surv_ibs).all()

    # survival probability 범위 확인
    assert (
        (surv_brier >= 0)
        & (surv_brier <= 1)
    ).all()

    # --------------------------------------------------------
    # Brier score
    # --------------------------------------------------------
    _, bs = brier_score(
        y_train_brier,
        y_test_brier,
        surv_brier,
        BRIER_TIMES
    )

    # --------------------------------------------------------
    # Integrated Brier Score
    # --------------------------------------------------------
    ibs = integrated_brier_score(
        y_train_brier,
        y_test_brier,
        surv_ibs,
        IBS_TIMES
    )

    brier_rows.append({
        "model": model_name,
        "brier_1year": float(bs[0]),
        "brier_3year": float(bs[1]),
        "ibs_1to3year": float(ibs),
        "risk_reproduction_max_diff": float(risk_max_diff),
    })

    fitted_baseline_models[
        model_name
    ] = model


# ------------------------------------------------------------
# 5. 결과
# ------------------------------------------------------------
brier_summary = pd.DataFrame(
    brier_rows
)

print(
    "=== TEST BRIER / IBS ==="
)

print(
    brier_summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 6. 저장
# ------------------------------------------------------------
brier_path = (
    data_dir
    / "processed"
    / "test_brier_ibs_metrics.csv"
)

brier_summary.to_csv(
    brier_path,
    index=False
)

print(
    "\nSaved:",
    brier_path
)

=== TEST BRIER / IBS ===
            model  brier_1year  brier_3year  ibs_1to3year  risk_reproduction_max_diff
         RNA-only     0.118574     0.165247      0.148245                         0.0
    Clinical-only     0.095692     0.097831      0.091632                         0.0
     Clinical+RNA     0.113825     0.096787      0.099160                         0.0
Clinical+Hallmark     0.093559     0.098681      0.093250                         0.0

Saved: ..\data\processed\test_brier_ibs_metrics.csv


In [91]:
# ============================================================
# FINAL TEST — Grouped calibration diagnostic
# Risk tertiles × 1-year / 3-year
# ============================================================

from sksurv.nonparametric import kaplan_meier_estimator

CAL_TIMES = [365.0, 1095.0]

# ------------------------------------------------------------
# 1. 각 모델의 Test survival probability 계산
# ------------------------------------------------------------
calibration_inputs = {
    "RNA-only": (
        fitted_baseline_models["RNA-only"],
        X_test_rna_scaled,
        0.019114577707188597
    ),

    "Clinical-only": (
        fitted_baseline_models["Clinical-only"],
        X_test_clin_scaled,
        0.010057565713433423
    ),

    "Clinical+RNA": (
        fitted_baseline_models["Clinical+RNA"],
        X_test_comb_scaled,
        0.1079631601142658
    ),

    "Clinical+Hallmark": (
        fitted_baseline_models["Clinical+Hallmark"],
        X_test_path_scaled,
        0.021211534321841412
    ),
}

predicted_risk_probs = {}

for model_name, (
    model,
    X_test,
    alpha
) in calibration_inputs.items():

    surv_funcs = model.predict_survival_function(
        X_test,
        alpha=alpha
    )

    # event probability = 1 - survival probability
    pred_event = np.asarray([
        [
            1.0 - fn(t)
            for t in CAL_TIMES
        ]
        for fn in surv_funcs
    ])

    assert pred_event.shape == (80, 2)
    assert np.isfinite(pred_event).all()
    assert (
        (pred_event >= 0)
        & (pred_event <= 1)
    ).all()

    predicted_risk_probs[
        model_name
    ] = pred_event


# ------------------------------------------------------------
# 2. KM observed event probability helper
# ------------------------------------------------------------
def km_event_probability(
    event,
    time,
    horizon
):
    km_time, km_surv = kaplan_meier_estimator(
        event,
        time
    )

    idx = np.searchsorted(
        km_time,
        horizon,
        side="right"
    ) - 1

    if idx < 0:
        survival = 1.0
    else:
        survival = float(
            km_surv[idx]
        )

    return 1.0 - survival


# ------------------------------------------------------------
# 3. 모델별 predicted 3-year risk로 tertile 생성
#
# 같은 risk grouping으로 1y / 3y를 비교해서
# group interpretation을 일관되게 유지
# ------------------------------------------------------------
cal_rows = []

event_test = (
    test_predictions["pfi_event"]
    .astype(bool)
    .to_numpy()
)

time_test = (
    test_predictions["pfi_time"]
    .astype(float)
    .to_numpy()
)

for model_name, pred_event in predicted_risk_probs.items():

    pred_1y = pred_event[:, 0]
    pred_3y = pred_event[:, 1]

    # 3-year predicted risk 기준 tertile
    tertile = pd.qcut(
        pred_3y,
        q=3,
        labels=[
            "Low",
            "Middle",
            "High"
        ]
    )

    for group in [
        "Low",
        "Middle",
        "High"
    ]:

        mask = (
            np.asarray(tertile)
            == group
        )

        n_group = int(
            mask.sum()
        )

        assert n_group > 0

        for horizon_label, horizon, pred in [
            (
                "1_year",
                365.0,
                pred_1y
            ),
            (
                "3_year",
                1095.0,
                pred_3y
            ),
        ]:

            observed_event_prob = (
                km_event_probability(
                    event_test[mask],
                    time_test[mask],
                    horizon
                )
            )

            mean_predicted_prob = float(
                np.mean(
                    pred[mask]
                )
            )

            cal_rows.append({
                "model": model_name,
                "risk_group": group,
                "horizon": horizon_label,
                "n": n_group,
                "mean_predicted_event_prob":
                    mean_predicted_prob,
                "km_observed_event_prob":
                    observed_event_prob,
                "calibration_difference":
                    mean_predicted_prob
                    - observed_event_prob
            })


calibration_df = pd.DataFrame(
    cal_rows
)

# ------------------------------------------------------------
# 4. 출력
# ------------------------------------------------------------
print(
    "=== GROUPED CALIBRATION DIAGNOSTIC ==="
)

print(
    calibration_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 5. 저장
# ------------------------------------------------------------
calibration_path = (
    data_dir
    / "processed"
    / "test_grouped_calibration.csv"
)

calibration_df.to_csv(
    calibration_path,
    index=False
)

print(
    "\nSaved:",
    calibration_path
)

=== GROUPED CALIBRATION DIAGNOSTIC ===
            model risk_group horizon  n  mean_predicted_event_prob  km_observed_event_prob  calibration_difference
         RNA-only        Low  1_year 27                   0.012231                0.078526               -0.066295
         RNA-only        Low  3_year 27                   0.030253                0.177255               -0.147002
         RNA-only     Middle  1_year 26                   0.050504                0.076923               -0.026419
         RNA-only     Middle  3_year 26                   0.121316                0.118881                0.002435
         RNA-only       High  1_year 27                   0.274175                0.277328               -0.003153
         RNA-only       High  3_year 27                   0.500670                0.503163               -0.002493
    Clinical-only        Low  1_year 27                   0.038305                0.000000                0.038305
    Clinical-only        Low  3_year 27  

In [92]:
# ============================================================
# FINAL PERFORMANCE SUMMARY
# 발표 / README용 최종 성능표
# ============================================================

# ------------------------------------------------------------
# 1. Bootstrap CI
# ------------------------------------------------------------
harrell_ci = pd.read_csv(
    data_dir
    / "processed"
    / "test_harrell_c_bootstrap_ci.csv"
)

# ------------------------------------------------------------
# 2. Discrimination metrics
# ------------------------------------------------------------
disc = pd.read_csv(
    data_dir
    / "processed"
    / "test_discrimination_metrics.csv"
)

# ------------------------------------------------------------
# 3. Brier / IBS
# ------------------------------------------------------------
brier = pd.read_csv(
    data_dir
    / "processed"
    / "test_brier_ibs_metrics.csv"
)

# ------------------------------------------------------------
# 4. 하나로 합치기
# ------------------------------------------------------------
final_performance_summary = (
    disc
    .merge(
        harrell_ci[
            [
                "model",
                "ci_95_lower",
                "ci_95_upper"
            ]
        ],
        on="model",
        how="left"
    )
    .merge(
        brier[
            [
                "model",
                "brier_1year",
                "brier_3year",
                "ibs_1to3year"
            ]
        ],
        on="model",
        how="left"
    )
)

# 컬럼 순서 정리
final_performance_summary = (
    final_performance_summary[
        [
            "model",
            "test_harrell_c",
            "ci_95_lower",
            "ci_95_upper",
            "uno_c_3year",
            "auc_1year",
            "auc_3year",
            "brier_1year",
            "brier_3year",
            "ibs_1to3year",
        ]
    ]
)


# ------------------------------------------------------------
# 5. 저장
# ------------------------------------------------------------
summary_path = (
    data_dir
    / "processed"
    / "final_performance_summary.csv"
)

final_performance_summary.to_csv(
    summary_path,
    index=False
)


# ------------------------------------------------------------
# 6. 출력
# ------------------------------------------------------------
print(
    "=== FINAL PERFORMANCE SUMMARY ==="
)

print(
    final_performance_summary.to_string(
        index=False
    )
)

print(
    "\nSaved:",
    summary_path
)

=== FINAL PERFORMANCE SUMMARY ===
            model  test_harrell_c  ci_95_lower  ci_95_upper  uno_c_3year  auc_1year  auc_3year  brier_1year  brier_3year  ibs_1to3year
         RNA-only        0.714511     0.589566     0.829764     0.703233   0.707653   0.723406     0.118574     0.165247      0.148245
    Clinical-only        0.886830     0.826026     0.940299     0.879822   0.888022   0.910865     0.095692     0.097831      0.091632
     Clinical+RNA        0.891956     0.844103     0.938447     0.896988   0.879702   0.943951     0.113825     0.096787      0.099160
Clinical+Hallmark        0.885647     0.817031     0.944232     0.868238   0.900662   0.895253     0.093559     0.098681      0.093250

Saved: ..\data\processed\final_performance_summary.csv
